# AutoResearch v2 â€” Momentum Alpha Discovery (T4Ã—2 Fixed)

This guarded stage evaluates deterministic/classical momentum baselines plus
program-evolution candidates on 100 US equities. If an LLM stage is enabled,
the report records loaded model id, execution state, and call count explicitly.

**Key differences from v1:**
- Model: **Qwen2.5-Coder-14B-Instruct (4-bit, default)** â€” fits comfortably on T4Ã—2 (~8 GB).
  Optionally switch to 32B-Instruct (needs both T4s fully; riskier on free tier).
- Universe: ~100 liquid US equities (vs 20)
- Reflection: reported only when recorded LLM/reflection calls actually execute
- Scoring: annual consistency (not just aggregate Sharpe)

**Kaggle setup:** Accelerator = **GPU T4 Ã—2**, Internet = **On**

> **OOM fix:** default model changed to 14B; device_map set to `balanced`;
> `max_memory` caps per-GPU usage to leave room for activations/KV cache.


## Cell 1 â€” Install dependencies

In [ ]:
!pip install -q yfinance wandb bitsandbytes "transformers>=4.44" "accelerate>=0.33"


## Cell 2 â€” Imports, configuration, paths

In [ ]:
import os, sys, json, re, time, random, gc, ast, traceback, hashlib
import threading as _th
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import yfinance as yf
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
try:
    import wandb
except Exception:
    wandb = None

# ?? Reproducibility ??
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ?? Runtime env ??
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

HF_TOKEN = None
WANDB_API_KEY = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    WANDB_API_KEY = user_secrets.get_secret("WANDB_API_KEY")
except Exception:
    HF_TOKEN = None
    WANDB_API_KEY = None
if not HF_TOKEN:
    HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
if not WANDB_API_KEY:
    WANDB_API_KEY = os.getenv("WANDB_API_KEY")
if HF_TOKEN:
    print("HF_TOKEN detected: authenticated HF Hub downloads enabled")
else:
    print("HF_TOKEN not set: using anonymous HF Hub access (lower rate limits)")
if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    os.environ.setdefault("WANDB_SILENT", "true")
    print("WANDB_API_KEY detected: W&B logging enabled")
else:
    print("WANDB_API_KEY not set: W&B logging disabled")

# ?? Paths ??
OUT = Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
RESEARCH_LOG = OUT / "research_log.jsonl"
PRICES_CACHE = OUT / "prices.parquet"
BEST_CODE    = OUT / "best_signal.py"
SHARPE_PLOT  = OUT / "sharpe_progress.png"
EQUITY_PLOT  = OUT / "equity_curves.png"
SUMMARY_MD   = OUT / "experiment_summary.md"
MEMO_FILE    = OUT / "research_memo.txt"
DETERMINISTIC_FILE = OUT / "deterministic_results.json"
PARAM_SEARCH_FILE = OUT / "parameter_search_results.json"

# ?? Model selection ??
MODEL_ID = "Qwen/Qwen2.5-Coder-14B-Instruct"
FALLBACK_MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"
BENCHMARK_MODE = True
BENCHMARK_OBJECTIVE = "best_verified_momentum_2xt4"
BENCHMARK_METHOD_FAMILIES = ["deterministic", "classical_risk_managed", "autoresearch_evolution", "lstm_sharpe", "tabular_ml", "gbt"]
APPROVED_WINNER_EDGE_OVER_DETERMINISTIC = 0.10
CHRONOLOGICAL_HOLDOUT_SEGMENTS = 3
ROLLING_HOLDOUT_SEGMENTS = 3
ROLLING_HOLDOUT_MIN_POINTS = 126
CODE_MODEL_CANDIDATES = [MODEL_ID, FALLBACK_MODEL_ID]
RUN_MODEL_AB = False
MODEL_AB_REPORT_FILE = OUT / "model_ab_report.json"

# ?? Loop config ??
N_BATCHES       = 6
REFLECT_EVERY   = 5
SANDBOX_TIMEOUT = 30
TRAIN_END       = "2022-12-31"

def _env_truthy(value, default=False):
    if value is None:
        return default
    return str(value).strip().lower() in {"1", "true", "yes", "on"}


# ?? Execution flags ??
RUN_BASELINE_SWEEP       = True
RUN_DETERMINISTIC_SEARCH = True
RUN_LLM_STAGE            = _env_truthy(os.getenv("AUTORESEARCH_RUN_LLM_STAGE"), default=False)
RUN_MOE_STAGE            = False
RUN_BNB_MODEL_LOAD       = _env_truthy(os.getenv("AUTORESEARCH_ENABLE_BNB_LOAD"), default=False)
RUN_LLM_SMOKE            = _env_truthy(os.getenv("AUTORESEARCH_RUN_LLM_SMOKE"), default=False)
RUN_PARAM_SEARCH         = True
RUN_HELDOUT_EVAL         = True
RUN_REPORTS              = True
# ?? Structured parameter search ??
PARAM_SEARCH_TRIALS = 200
PARAM_SEARCH_SEED = SEED
PARAM_SEARCH_MODE = "random"
PARAM_SEARCH_SHORT_SPANS = [36, 39, 42, 45, 48, 51, 54, 57, 60, 63, 66]
PARAM_SEARCH_LONG_SPANS = [90, 95, 100, 105, 110, 120, 130, 140, 150]
PARAM_SEARCH_REGIME_THRESHOLDS = [18.0, 20.0, 22.0, 24.0, 28.0]
PARAM_SEARCH_VOL_WINDOWS = [10, 20, 30]
PARAM_SEARCH_VOL_GATE_WINDOWS = [10, 20, 40]

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {name}  {mem:.1f} GB")
print(f"flags: baseline={RUN_BASELINE_SWEEP} deterministic={RUN_DETERMINISTIC_SEARCH} param_search={RUN_PARAM_SEARCH} heldout={RUN_HELDOUT_EVAL} reports={RUN_REPORTS}")


# Optional W&B telemetry. This must never be required for the research loop to run.
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "momentum trading")
WANDB_ENTITY = os.getenv("WANDB_ENTITY", "srikantsubham10000-na")
WANDB_RUN = None
WANDB_DISABLED_REASON = None


def _wandb_scalar(v):
    if isinstance(v, (np.integer, np.floating)):
        return float(v)
    if isinstance(v, (int, float, bool, str)) or v is None:
        return v
    return None


def wandb_init_once():
    global WANDB_RUN, WANDB_DISABLED_REASON
    if WANDB_RUN is not None:
        return WANDB_RUN
    if not WANDB_API_KEY:
        WANDB_DISABLED_REASON = "missing_api_key"
        return None
    if wandb is None:
        WANDB_DISABLED_REASON = "wandb_import_failed"
        return None
    config = {
        "model_id": MODEL_ID,
        "fallback_model_id": FALLBACK_MODEL_ID,
        "seed": SEED,
        "selection_objective": globals().get("SELECTION_OBJECTIVE", "market_neutral_net_sharpe"),
        "run_baseline_sweep": RUN_BASELINE_SWEEP,
        "run_deterministic_search": RUN_DETERMINISTIC_SEARCH,
        "run_param_search": RUN_PARAM_SEARCH,
        "run_heldout_eval": RUN_HELDOUT_EVAL,
        "run_llm_smoke": RUN_LLM_SMOKE,
        "param_search_trials": PARAM_SEARCH_TRIALS,
        "train_end": TRAIN_END,
    }
    try:
        WANDB_RUN = wandb.init(
            entity=WANDB_ENTITY,
            project=WANDB_PROJECT,
            name=f"autoresearch-v2-{time.strftime('%Y%m%d-%H%M%S')}",
            config=config,
            reinit=True,
        )
    except Exception as e:
        if " " in WANDB_PROJECT:
            try:
                WANDB_RUN = wandb.init(
                    entity=WANDB_ENTITY,
                    project=WANDB_PROJECT.replace(" ", "-"),
                    name=f"autoresearch-v2-{time.strftime('%Y%m%d-%H%M%S')}",
                    config=config,
                    reinit=True,
                )
            except Exception as e2:
                WANDB_DISABLED_REASON = f"init_failed:{type(e2).__name__}"
                print(f"W&B disabled: {WANDB_DISABLED_REASON}")
                return None
        else:
            WANDB_DISABLED_REASON = f"init_failed:{type(e).__name__}"
            print(f"W&B disabled: {WANDB_DISABLED_REASON}")
            return None
    return WANDB_RUN


def wandb_log(metrics, step=None):
    run = wandb_init_once()
    if run is None:
        return
    clean = {}
    for k, v in dict(metrics).items():
        sv = _wandb_scalar(v)
        if sv is not None:
            clean[str(k)] = sv
    if clean:
        wandb.log(clean, step=step)


def wandb_log_candidate(entry):
    if not isinstance(entry, dict):
        return
    step = entry.get("iter") if isinstance(entry.get("iter"), int) else None
    metrics = {"candidate/error": 1 if entry.get("error") else 0}
    for key in (
        "score", "train_sharpe", "raw_sharpe", "inv_sharpe", "beta", "turnover",
        "consistency", "ann_return", "dd", "wf_median", "wf_min",
        "raw_cs_std", "raw_long_frac", "raw_short_frac", "signal_activity",
    ):
        if isinstance(entry.get(key), (int, float, np.integer, np.floating)):
            metrics[f"candidate/{key}"] = float(entry[key])
    wandb_log(metrics, step=step)


def wandb_log_artifacts(paths):
    run = wandb_init_once()
    if run is None:
        return
    for p in paths:
        try:
            p = Path(p)
            if p.exists():
                wandb.save(str(p))
        except Exception:
            pass


wandb_init_once()


## Cell 3 â€” Download and cache price data

~100 liquid US equities, 2015â€“2024 via yfinance (free).
Tickers with <70 % data coverage are dropped automatically.
Cached to parquet after first download.
Split: 2015â€“2022 = train, 2023â€“2024 = held-out test.


In [ ]:
TICKERS = [
    # â”€â”€ Tech / Semis â”€â”€
    "AAPL","MSFT","GOOGL","AMZN","META","NVDA","TSLA","AVGO",
    "ORCL","CRM","ADBE","CSCO","ACN","INTC","AMD","QCOM",
    "TXN","AMAT","MU","LRCX","ADI","IBM","INTU",
    # â”€â”€ Financials â”€â”€
    "JPM","V","MA","BAC","WFC","GS","MS","BLK","AXP","C",
    "SPGI","CME","SCHW",
    # â”€â”€ Healthcare â”€â”€
    "UNH","JNJ","LLY","PFE","ABT","TMO","MRK","ABBV",
    "DHR","AMGN","MDT","BMY","GILD","ISRG","VRTX","REGN",
    # â”€â”€ Consumer â”€â”€
    "WMT","PG","KO","PEP","COST","MCD","NKE","SBUX",
    "TGT","CL","MDLZ","EL","PM","MO",
    # â”€â”€ Industrials â”€â”€
    "CAT","HON","UPS","BA","GE","RTX","LMT","DE",
    "MMM","UNP","FDX","EMR","ETN",
    # â”€â”€ Energy â”€â”€
    "XOM","CVX","COP","SLB","EOG","PSX",
    # â”€â”€ Communication â”€â”€
    "DIS","NFLX","CMCSA","T","VZ",
    # â”€â”€ Materials â”€â”€
    "LIN","APD","ECL","SHW","FCX","NEM",
    # â”€â”€ Utilities â”€â”€
    "NEE","DUK","SO","D",
    # â”€â”€ Real Estate â”€â”€
    "PLD","AMT","CCI","SPG","EQIX",
    # â”€â”€ Retail / Other â”€â”€
    "HD","LOW","TJX","ROST","PYPL",
]

START = "2015-01-01"
END   = "2024-12-31"

def load_prices():
    if PRICES_CACHE.exists():
        df = pd.read_parquet(PRICES_CACHE)
        cols = df.columns.get_level_values(1).unique()
        print(f"cached: {len(cols)} tickers, "
              f"{df.index.min().date()} â†’ {df.index.max().date()}")
        return df
    print(f"downloading {len(TICKERS)} tickers from yfinance ...")
    last_err = None
    for attempt in range(3):
        try:
            raw = yf.download(TICKERS, start=START, end=END,
                              auto_adjust=True, progress=True, threads=True)
            if raw.empty:
                raise ValueError("empty dataframe")
            break
        except Exception as e:
            last_err = e
            print(f"  attempt {attempt+1} failed: {e}")
            time.sleep(5)
    else:
        raise RuntimeError(f"yfinance download failed after 3 attempts: {last_err}")

    close = raw["Close"].dropna(how="all")

    # keep tickers with >70 % data coverage
    coverage = close.notna().mean()
    good = sorted(coverage[coverage > 0.7].index.tolist())
    close = close[good].ffill().bfill()
    # drop any remaining all-NaN columns
    close = close.dropna(axis=1, how="any")
    volume = raw["Volume"].reindex(columns=close.columns,
                                    index=close.index).fillna(0)
    df = pd.concat({"close": close, "volume": volume}, axis=1)
    df.to_parquet(PRICES_CACHE)
    print(f"saved: {close.shape[1]} tickers, {close.shape[0]} days")
    return df

prices   = load_prices()
close_all  = prices["close"].astype(float)
volume_all = prices["volume"].astype(float)

mask = close_all.index <= pd.Timestamp(TRAIN_END)
close_train,  close_test  = close_all[mask], close_all[~mask]
volume_train, volume_test = volume_all[mask], volume_all[~mask]

print(f"train : {close_train.shape}  ({close_train.index.min().date()} â†’ "
      f"{close_train.index.max().date()})")
print(f"test  : {close_test.shape}   ({close_test.index.min().date()} â†’ "
      f"{close_test.index.max().date()})")
print(f"universe: {len(close_all.columns)} tickers")

# â”€â”€ Regime data: VIX + 10Y yield (free via yfinance) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
REGIME_CACHE = OUT / "regime.parquet"

def load_regime_data(index):
    """Download ^VIX and ^TNX, align to equity trading calendar."""
    if REGIME_CACHE.exists():
        df = pd.read_parquet(REGIME_CACHE)
        print(f"regime cached: {df.index.min().date()} â†’ {df.index.max().date()}")
        return df
    print("downloading ^VIX and ^TNX ...")
    frames = {}
    for sym, col in [("^VIX", "VIX"), ("^TNX", "TNX")]:
        raw = yf.download(sym, start=START, end=END,
                          auto_adjust=True, progress=False)
        # yfinance >= 0.2 may return MultiIndex for single tickers
        close_col = raw["Close"] if "Close" in raw.columns else raw.iloc[:, 0]
        frames[col] = close_col.squeeze()
    df = pd.DataFrame(frames).reindex(index).ffill().bfill()
    df.to_parquet(REGIME_CACHE)
    print(f"saved regime: {df.shape}")
    return df

regime_all   = load_regime_data(close_all.index)
vix_all      = regime_all["VIX"]
tnx_all      = regime_all["TNX"]

vix_train, vix_test = vix_all[mask], vix_all[~mask]
tnx_train, tnx_test = tnx_all[mask], tnx_all[~mask]
print(f"VIX  train mean={vix_train.mean():.1f}  test mean={vix_test.mean():.1f}")
print(f"TNX  train mean={tnx_train.mean():.2f}%  test mean={tnx_test.mean():.2f}%")


## Cell 4 â€” Backtester with annual consistency

Vectorised pandas. Positions = `signal.shift(1)` (no lookahead).
Equal-weight across assets. 5 bps per-turn transaction cost.
Reports both aggregate Sharpe and **annual consistency** (fraction of
calendar years with positive active Sharpe).


In [ ]:
BETA_NEUTRALIZE_POSITIONS = True
BETA_NEUTRAL_LOOKBACK = 126
BETA_NEUTRAL_MIN_PERIODS = 40
BETA_NEUTRAL_EPS = 1e-10


def _coerce_signal(signal_df, close_df):
    """Coerce candidate output into a numeric frame aligned to prices."""
    sig = signal_df.copy()
    sig = sig.reindex(index=close_df.index, columns=close_df.columns)
    sig = sig.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return sig.clip(-1.0, 1.0)


def signal_quality(signal_df, close_df):
    """Diagnostics on the raw signal before market-neutral normalization."""
    raw = _coerce_signal(signal_df, close_df)
    demeaned = raw.sub(raw.mean(axis=1), axis=0)
    gross = demeaned.abs().sum(axis=1)

    return {
        "raw_cs_std": float(raw.std(axis=1).mean()),
        "raw_abs_row_mean": float(raw.mean(axis=1).abs().mean()),
        "raw_long_frac": float((raw > 0).mean().mean()),
        "raw_short_frac": float((raw < 0).mean().mean()),
        "signal_activity": float((gross > 1e-8).mean()),
        "gross_exposure": float(gross.mean()),
    }


def _unit_gross_positions(centered):
    gross = centered.abs().sum(axis=1).replace(0.0, np.nan)
    return centered.div(gross, axis=0).fillna(0.0)


def _rolling_asset_betas(close_df, lookback=BETA_NEUTRAL_LOOKBACK):
    """Estimate per-asset beta using only returns available before each signal date."""
    ret = close_df.pct_change().fillna(0.0)
    bench = ret.mean(axis=1)
    min_periods = min(BETA_NEUTRAL_MIN_PERIODS, int(lookback))
    asset_mean = ret.rolling(lookback, min_periods=min_periods).mean()
    bench_mean = bench.rolling(lookback, min_periods=min_periods).mean()
    cov = ret.mul(bench, axis=0).rolling(lookback, min_periods=min_periods).mean()
    cov = cov - asset_mean.mul(bench_mean, axis=0)
    var = bench.pow(2).rolling(lookback, min_periods=min_periods).mean() - bench_mean.pow(2)
    betas = cov.div(var.replace(0.0, np.nan), axis=0)
    return betas.shift(1).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(-5.0, 5.0)


def _beta_neutralize_positions(pos, close_df):
    """Project daily weights away from lagged equal-weight market beta."""
    centered = pos.sub(pos.mean(axis=1), axis=0)
    betas = _rolling_asset_betas(close_df).reindex_like(centered).fillna(0.0)
    beta_exposure = (centered * betas).sum(axis=1)
    beta_norm = betas.pow(2).sum(axis=1).replace(0.0, np.nan)
    hedge = betas.mul(beta_exposure.div(beta_norm).fillna(0.0), axis=0)
    adjusted = centered - hedge
    adjusted = adjusted.sub(adjusted.mean(axis=1), axis=0)
    return _unit_gross_positions(adjusted)


def _market_neutral_positions(signal_df, close_df, beta_neutralize=BETA_NEUTRALIZE_POSITIONS):
    """Convert raw scores to dollar-neutral unit-gross weights per date."""
    raw = _coerce_signal(signal_df, close_df)
    centered = raw.sub(raw.mean(axis=1), axis=0)
    pos = _unit_gross_positions(centered)
    if beta_neutralize:
        return _beta_neutralize_positions(pos, close_df)
    return pos


def _series_metrics(series):
    series = series.dropna()
    ann_ret = float(series.mean() * 252)
    ann_vol = float(series.std() * np.sqrt(252))
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0.0
    equity = (1 + series).cumprod()
    max_dd = float((equity / equity.cummax() - 1).min()) if len(equity) else 0.0
    hit = float((series > 0).mean()) if len(series) else 0.0
    return ann_ret, ann_vol, sharpe, max_dd, hit, equity


def _annual_consistency(series):
    series = series.dropna()
    annual_sharpes = []
    if len(series):
        for yr in sorted(series.index.year.unique()):
            part = series[series.index.year == yr]
            if len(part) < 50:
                continue
            vol = float(part.std() * np.sqrt(252))
            annual_sharpes.append(float(part.mean() * 252 / vol) if vol > 0 else 0.0)
    consistency = float(sum(1 for s in annual_sharpes if s > 0) / max(len(annual_sharpes), 1))
    min_annual = float(min(annual_sharpes)) if annual_sharpes else 0.0
    return consistency, annual_sharpes, min_annual


def backtest(signal_df, close_df, cost_bps=1.0):
    # Enforce the competition objective in the harness itself: active,
    # cross-sectional, market-neutral predictions. This closes the loophole
    # where all-long signals scored as equity beta.
    quality = signal_quality(signal_df, close_df)
    raw_sig = _market_neutral_positions(signal_df, close_df, beta_neutralize=False)
    sig = _market_neutral_positions(signal_df, close_df)
    pos = sig.shift(1).fillna(0.0)
    raw_pos = raw_sig.shift(1).fillna(0.0)
    ret = close_df.pct_change().fillna(0.0)

    raw_gross = (raw_pos * ret).sum(axis=1)
    gross = (pos * ret).sum(axis=1)
    turnover = 0.5 * pos.diff().abs().sum(axis=1).fillna(0.0)
    cost = turnover * (cost_bps / 10000.0)
    net = gross - cost

    ann_ret, ann_vol, sharpe, max_dd, hit, equity = _series_metrics(net)
    consistency, annual_sharpes, min_annual = _annual_consistency(net)
    gross_ann_ret, gross_ann_vol, gross_sharpe, gross_dd, gross_hit, _ = _series_metrics(gross)

    inv_gross = -gross
    inv_net = inv_gross - cost
    inv_ann_ret, inv_ann_vol, inv_sharpe, inv_dd, inv_hit, inv_equity = _series_metrics(inv_net)
    inv_consistency, inv_annual_sharpes, inv_min_annual = _annual_consistency(inv_net)

    bench = ret.mean(axis=1)
    spread = net - bench
    spread_ann_ret, spread_ann_vol, spread_sharpe, spread_dd, spread_hit, spread_equity = _series_metrics(spread)

    if bench.std() > 0 and net.std() > 0:
        beta = float(np.cov(net, bench)[0, 1] / np.var(bench))
    else:
        beta = 0.0
    if bench.std() > 0 and raw_gross.std() > 0:
        beta_raw = float(np.cov(raw_gross, bench)[0, 1] / np.var(bench))
    else:
        beta_raw = 0.0
    if bench.std() > 0 and inv_net.std() > 0:
        beta_inv = float(np.cov(inv_net, bench)[0, 1] / np.var(bench))
    else:
        beta_inv = 0.0

    out = {
        "ann_return": ann_ret,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "sharpe_raw": gross_sharpe,
        "max_dd": max_dd,
        "hit_rate": hit,
        "avg_turnover": float(turnover.mean()),
        "beta": beta,
        "beta_neutralized": bool(BETA_NEUTRALIZE_POSITIONS),
        "beta_raw": beta_raw,
        "beta_neutralized_value": beta,
        "beta_reduction": float(abs(beta_raw) - abs(beta)),
        "consistency": consistency,
        "annual_sharpes": annual_sharpes,
        "min_annual": min_annual,
        "equity": equity,
        "gross_ann_return": gross_ann_ret,
        "gross_ann_vol": gross_ann_vol,
        "gross_max_dd": gross_dd,
        "gross_hit_rate": gross_hit,
        "inverted_ann_return": inv_ann_ret,
        "inverted_ann_vol": inv_ann_vol,
        "inverted_sharpe": inv_sharpe,
        "inverted_max_dd": inv_dd,
        "inverted_hit_rate": inv_hit,
        "inverted_equity": inv_equity,
        "inverted_beta": beta_inv,
        "inverted_consistency": inv_consistency,
        "inverted_annual_sharpes": inv_annual_sharpes,
        "inverted_min_annual": inv_min_annual,
        "benchmark_spread_ann_return": spread_ann_ret,
        "benchmark_spread_ann_vol": spread_ann_vol,
        "benchmark_spread_sharpe": spread_sharpe,
        "benchmark_spread_max_dd": spread_dd,
        "benchmark_spread_hit_rate": spread_hit,
        "benchmark_spread_equity": spread_equity,
        "prefer_inverted": bool(inv_sharpe > sharpe + 0.10),
    }
    out.update(quality)

    # Backward-compatible aliases used by later notebook cells.
    out["sharpe_inverted"] = inv_sharpe
    out["ann_return_inverted"] = inv_ann_ret
    out["ann_vol_inverted"] = inv_ann_vol
    out["max_dd_inverted"] = inv_dd
    out["hit_rate_inverted"] = inv_hit
    out["equity_inverted"] = inv_equity
    out["beta_inverted"] = beta_inv
    out["consistency_inverted"] = inv_consistency
    out["annual_sharpes_inverted"] = inv_annual_sharpes
    out["min_annual_inverted"] = inv_min_annual
    out["excess_ann_return"] = spread_ann_ret
    out["excess_ann_vol"] = spread_ann_vol
    out["excess_sharpe"] = spread_sharpe
    out["excess_max_dd"] = spread_dd
    out["excess_hit_rate"] = spread_hit
    out["excess_equity"] = spread_equity
    return out

## Cell 5 â€” Sandbox executor

AST-validated, restricted builtins, thread-based timeout.
Works from any thread (main or ThreadPoolExecutor worker).


In [ ]:
import inspect as _inspect

FORBIDDEN = [
    "import os", "import sys", "import subprocess", "import socket",
    "import shutil", "import requests", "import urllib",
    "open(", "__import__", "eval(", "exec(",
    "compile(", "globals(", "locals(", "getattr(", "setattr(", "delattr(",
    "input(", "quit(", "exit(", "__builtins__", "__class__", "__bases__",
    "__subclasses__", "pathlib", "Path(",
]
ALLOWED_IMPORTS = {"numpy", "np", "pandas", "pd", "math"}
_ALLOWED_IMPORT_LINE = re.compile(r"^\s*(?:from\s+(numpy|pandas|math)\s+import\s+.+|import\s+(.+))\s*$")

def validate_code(code_str):
    low = code_str.lower()
    for bad in FORBIDDEN:
        if bad.lower() in low:
            return False, f"forbidden: {bad!r}"
    try:
        tree = ast.parse(code_str)
    except SyntaxError as e:
        return False, f"syntax: {e}"
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                if alias.name.split(".")[0] not in ALLOWED_IMPORTS:
                    return False, f"disallowed import: {alias.name}"
        elif isinstance(node, ast.ImportFrom):
            if (node.module or "").split(".")[0] not in ALLOWED_IMPORTS:
                return False, f"disallowed from-import: {node.module}"
    if "def signal" not in code_str:
        return False, "no `def signal` found"
    return True, "ok"

SAFE_BUILTINS = {
    k: (__builtins__[k] if isinstance(__builtins__, dict)
        else getattr(__builtins__, k))
    for k in ["len","range","min","max","abs","sum","int","float","bool",
              "list","dict","tuple","set","str","True","False","None",
              "enumerate","zip","sorted","reversed","map","filter","round",
              "isinstance","type","any","all","print","slice"]
    if (k in __builtins__ if isinstance(__builtins__, dict)
        else hasattr(__builtins__, k))
}

def run_signal_code(code_str, close_df, volume_df,
                    vix_s=None, tnx_s=None, timeout=SANDBOX_TIMEOUT):
    """Sandboxed execution. Passes vix/tnx only if signal declares them."""
    ok, msg = validate_code(code_str)
    if not ok:
        return None, f"VALIDATION: {msg}"

    cleaned_lines = []
    for line in code_str.splitlines():
        m = _ALLOWED_IMPORT_LINE.match(line)
        if not m:
            cleaned_lines.append(line)
            continue
        raw_mods = []
        if m.group(1):
            raw_mods.append(m.group(1))
        if m.group(2):
            raw_mods.extend(part.strip().split(" as ")[0].strip() for part in m.group(2).split(","))
        roots = {mod.split(".")[0] for mod in raw_mods if mod}
        if not roots.issubset(ALLOWED_IMPORTS):
            cleaned_lines.append(line)
    cleaned_code = "\n".join(cleaned_lines)

    ns = {"np": np, "pd": pd, "math": __import__("math"), "__builtins__": SAFE_BUILTINS}
    result = [None, None]

    def _run():
        try:
            exec(cleaned_code, ns)
            if "signal" not in ns or not callable(ns["signal"]):
                result[1] = "no callable `signal` after exec"
                return
            try:
                params = set(_inspect.signature(ns["signal"]).parameters)
            except (ValueError, TypeError):
                params = set()
            kwargs = {}
            if "vix" in params and vix_s is not None:
                kwargs["vix"] = vix_s.copy()
            if "tnx" in params and tnx_s is not None:
                kwargs["tnx"] = tnx_s.copy()
            result[0] = ns["signal"](close_df.copy(), volume_df.copy(), **kwargs)
        except Exception as e:
            result[1] = f"{type(e).__name__}: {e}"

    t = _th.Thread(target=_run, daemon=True)
    t.start()
    t.join(timeout=timeout)
    if t.is_alive():
        return None, f"TIMEOUT (>{timeout}s)"
    if result[1] is not None:
        return None, result[1]
    out = result[0]
    if not isinstance(out, pd.DataFrame):
        return None, f"returned {type(out).__name__}, need DataFrame"
    if out.shape != close_df.shape:
        return None, f"shape {out.shape} != expected {close_df.shape}"
    ok_signal, sig_msg = validate_signal(out, expected_shape=close_df.shape)
    if not ok_signal:
        return None, f"SIGNAL_VALIDATION: {sig_msg}"
    out = out.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(-1, 1)
    return out, None


def validate_signal(sig_df, expected_shape=None):
    s = sig_df.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(-1, 1)
    if expected_shape is not None and s.shape != expected_shape:
        return False, f"shape {s.shape} != expected {expected_shape}"

    row_std = float(s.std(axis=1).mean())
    long_frac = float((s > 0.05).mean().mean())
    short_frac = float((s < -0.05).mean().mean())
    abs_row_mean = float(s.mean(axis=1).abs().mean())
    demeaned = s.sub(s.mean(axis=1), axis=0)
    active_frac = float((demeaned.abs().sum(axis=1) > 1e-8).mean())

    if row_std < 0.02:
        return False, f"near-constant cross-section (row_std={row_std:.4f})"
    if min(long_frac, short_frac) < 0.08:
        return False, f"not genuinely long-short (L={long_frac:.2f} S={short_frac:.2f})"
    if abs_row_mean > 0.50:
        return False, f"directionally biased before neutralization (abs_row_mean={abs_row_mean:.2f})"
    if active_frac < 0.50:
        return False, f"dead after market-neutral normalization (active_frac={active_frac:.2f})"
    return True, "ok"


def detect_lookahead(code_str, close_df, volume_df,
                     vix_s=None, tnx_s=None, split_frac=0.6, seed=0):
    sig_real, err = run_signal_code(code_str, close_df, volume_df,
                                    vix_s, tnx_s, timeout=20)
    if err is not None:
        return False, err
    T   = int(len(close_df) * split_frac)
    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(close_df) - T)
    c2, v2 = close_df.copy(), volume_df.copy()
    c2.iloc[T:] = close_df.iloc[T:].values[perm]
    v2.iloc[T:] = volume_df.iloc[T:].values[perm]
    vix2 = tnx2 = None
    if vix_s is not None:
        vix2 = vix_s.copy(); vix2.iloc[T:] = vix_s.iloc[T:].values[perm]
    if tnx_s is not None:
        tnx2 = tnx_s.copy(); tnx2.iloc[T:] = tnx_s.iloc[T:].values[perm]
    sig_perm, err = run_signal_code(code_str, c2, v2, vix2, tnx2, timeout=20)
    if err is not None:
        return False, f"shuffle_run: {err}"
    a = sig_real.iloc[:T].fillna(0).values
    b = sig_perm.iloc[:T].fillna(0).values
    diff = float(np.abs(a - b).mean())
    if diff > 1e-6:
        return False, f"LOOKAHEAD detected (diff={diff:.6f})"
    return True, "ok"


# â”€â”€ self-tests â”€â”€
_test = """
def signal(close, volume):
    r = close.pct_change(20).rank(axis=1, pct=True)
    out = (r - 0.5) * 2
    return out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
"""
_s, _e = run_signal_code(_test, close_train, volume_train)
print("sandbox self-test:", "OK" if _e is None else f"FAIL: {_e}")

_test_regime = """
def signal(close, volume, vix=None, tnx=None):
    r = close.pct_change(20).rank(axis=1, pct=True)
    out = (r - 0.5) * 2
    if vix is not None:
        high_vol = (vix > 20).astype(float).values[:, None]
        out = out * (1 + 0.3 * high_vol)
    return out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
"""
_s2, _e2 = run_signal_code(_test_regime, close_train, volume_train,
                            vix_s=vix_train, tnx_s=tnx_train)
print("regime self-test:", "OK" if _e2 is None else f"FAIL: {_e2}")


## Cell 6 â€” Load model (4-bit quantised)

Qwen2.5-Coder-14B-Instruct at NF4 via bitsandbytes â†’ ~8 GB, fits on one T4
and leaves the second GPU free for KV-cache / activations.
`device_map='balanced'` splits layers evenly across both GPUs.
`max_memory` hard-caps per-GPU usage so activations never OOM.


In [ ]:
# Clear GPU memory before optional model loading.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

REQUESTED_LLM_MODEL = bool(RUN_LLM_STAGE or RUN_MOE_STAGE or RUN_LLM_SMOKE)
SHOULD_LOAD_LLM_MODEL = bool(REQUESTED_LLM_MODEL and RUN_BNB_MODEL_LOAD)
tok = None
model = None
ACTIVE_MODEL_ID = None
LLM_LOAD_ERROR = None

hub_kwargs = {"token": HF_TOKEN} if HF_TOKEN else {}


def _load_tokenizer(model_id):
    t = AutoTokenizer.from_pretrained(model_id, **hub_kwargs)
    if t.pad_token_id is None:
        t.pad_token_id = t.eos_token_id
    return t


n_gpus = torch.cuda.device_count()
per_gpu_cap = "11GiB" if n_gpus >= 2 else "13GiB"
max_memory = {i: per_gpu_cap for i in range(n_gpus)}
max_memory["cpu"] = "48GiB"
offload_dir = OUT / "hf_offload"
offload_dir.mkdir(exist_ok=True)


def _bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )


def _load_model(model_id):
    return AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=_bnb_config(),
        device_map="auto",
        max_memory=max_memory,
        low_cpu_mem_usage=True,
        offload_state_dict=True,
        offload_folder=str(offload_dir),
        dtype=torch.float16,
        **hub_kwargs,
    )


def refresh_llm_runtime_status():
    globals()["LLM_RUNTIME_STATUS"] = {
        "configured_model_id": globals().get("MODEL_ID"),
        "fallback_model_id": globals().get("FALLBACK_MODEL_ID"),
        "llm_stage_enabled": bool(globals().get("RUN_LLM_STAGE", False)),
        "moe_stage_enabled": bool(globals().get("RUN_MOE_STAGE", False)),
        "llm_smoke_enabled": bool(globals().get("RUN_LLM_SMOKE", False)),
        "bnb_model_load_enabled": bool(globals().get("RUN_BNB_MODEL_LOAD", False)),
        "requested_model": bool(globals().get("REQUESTED_LLM_MODEL", False)),
        "should_load": bool(globals().get("SHOULD_LOAD_LLM_MODEL", False)),
        "loaded": bool(globals().get("tok") is not None and globals().get("model") is not None),
        "active_model_id": globals().get("ACTIVE_MODEL_ID"),
        "load_error": globals().get("LLM_LOAD_ERROR"),
        "benchmark_mode": globals().get("BENCHMARK_MODE", False),
    }
    return globals()["LLM_RUNTIME_STATUS"]


def ensure_model_loaded():
    global tok, model, MODEL_ID, ACTIVE_MODEL_ID, LLM_LOAD_ERROR
    if tok is not None and model is not None:
        refresh_llm_runtime_status()
        return tok, model
    if not RUN_BNB_MODEL_LOAD:
        LLM_LOAD_ERROR = "bnb_model_load_disabled"
        refresh_llm_runtime_status()
        raise RuntimeError(
            "LLM model loading is disabled for this Kaggle run because the current "
            "CUDA/bitsandbytes stack can hard-crash the kernel. Set "
            "AUTORESEARCH_ENABLE_BNB_LOAD=1 only for a dedicated model-load test."
        )
    if torch.cuda.device_count() == 0:
        LLM_LOAD_ERROR = "no_cuda_gpu"
        refresh_llm_runtime_status()
        raise RuntimeError("No CUDA GPU detected. This notebook is configured for T4 sessions.")

    active_model_id = MODEL_ID
    tok = _load_tokenizer(active_model_id)
    try:
        model = _load_model(active_model_id)
    except Exception as e:
        msg = str(e).lower()
        is_oom = ("outofmemory" in msg) or ("out of memory" in msg) or isinstance(e, torch.OutOfMemoryError)
        if (not is_oom) or active_model_id == FALLBACK_MODEL_ID:
            LLM_LOAD_ERROR = f"{type(e).__name__}: {e}"
            refresh_llm_runtime_status()
            raise
        print(f"Primary model OOM: {MODEL_ID}")
        print("Falling back to 7B model for stability on current T4 session...")
        gc.collect(); torch.cuda.empty_cache()
        active_model_id = FALLBACK_MODEL_ID
        tok = _load_tokenizer(active_model_id)
        model = _load_model(active_model_id)

    model.eval()
    MODEL_ID = active_model_id
    ACTIVE_MODEL_ID = active_model_id
    LLM_LOAD_ERROR = None
    print(f"model loaded: {MODEL_ID}")
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / 1e9
        total = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {alloc:.1f} GB allocated / {total:.1f} GB total")
    refresh_llm_runtime_status()
    return tok, model


if SHOULD_LOAD_LLM_MODEL:
    ensure_model_loaded()
    if RUN_LLM_SMOKE:
        try:
            smoke_ids = tok("LLM smoke test", return_tensors="pt").input_ids.to(model.device)
            with torch.no_grad():
                _ = model.generate(smoke_ids, max_new_tokens=1, do_sample=False, pad_token_id=tok.pad_token_id)
            globals()["LLM_SMOKE_STATUS"] = {"executed": True, "ok": True}
            print("LLM smoke test: OK")
        except Exception as e:
            globals()["LLM_SMOKE_STATUS"] = {"executed": True, "ok": False, "error": f"{type(e).__name__}: {e}"}
            print(f"LLM smoke test failed: {type(e).__name__}: {e}")
else:
    if REQUESTED_LLM_MODEL:
        LLM_LOAD_ERROR = "bnb_model_load_disabled"
        print("LLM stage requested, but bitsandbytes model load is disabled for kernel stability.")
        print("Set AUTORESEARCH_ENABLE_BNB_LOAD=1 only for a dedicated model-load smoke test.")
    else:
        print("LLM stages disabled; skipping model load.")
    refresh_llm_runtime_status()


# Model A/B helpers. Safe to call when model loading is explicitly enabled.
def unload_llm_model():
    global tok, model, ACTIVE_MODEL_ID
    globals().pop("model", None)
    globals().pop("tok", None)
    model = None
    tok = None
    ACTIVE_MODEL_ID = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
    refresh_llm_runtime_status()
    return True


def switch_llm_model(model_id):
    global MODEL_ID, ACTIVE_MODEL_ID, model, tok
    target = str(model_id)
    if not target:
        raise ValueError("model_id is required")
    if not RUN_BNB_MODEL_LOAD:
        raise RuntimeError("Model switching requires AUTORESEARCH_ENABLE_BNB_LOAD=1.")

    current = globals().get("ACTIVE_MODEL_ID", globals().get("MODEL_ID"))
    if target == current and model is not None and tok is not None:
        return model

    previous = current
    unload_llm_model()
    try:
        ACTIVE_MODEL_ID = target
        tok = _load_tokenizer(ACTIVE_MODEL_ID)
        model = _load_model(ACTIVE_MODEL_ID)
        model.eval()
        MODEL_ID = ACTIVE_MODEL_ID
        print(f"model switched: {MODEL_ID}")
        refresh_llm_runtime_status()
        return model
    except Exception:
        unload_llm_model()
        if previous and previous != target:
            try:
                ACTIVE_MODEL_ID = previous
                tok = _load_tokenizer(ACTIVE_MODEL_ID)
                model = _load_model(ACTIVE_MODEL_ID)
                model.eval()
                MODEL_ID = ACTIVE_MODEL_ID
                print(f"model switch failed; restored: {MODEL_ID}")
                refresh_llm_runtime_status()
            except Exception:
                unload_llm_model()
        raise


refresh_llm_runtime_status()


## Cell 7 â€” LLM generation helper (cached system prefix, 800 tokens)


In [ ]:
# ?? Lazy cache for system-prompt token IDs (order-safe across cells) ???????????
# This avoids NameError when this cell runs before SYSTEM_PROMPT is defined.
_SYS_IDS = None

def _ensure_sys_ids():
    global _SYS_IDS
    if _SYS_IDS is not None:
        return _SYS_IDS
    if "SYSTEM_PROMPT" not in globals():
        raise RuntimeError("SYSTEM_PROMPT is not defined yet. Run the prompt cell before generating.")
    _sys_text = tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT}],
        tokenize=False, add_generation_prompt=False,
    )
    _SYS_IDS = tok(_sys_text, return_tensors="pt",
                   add_special_tokens=False)["input_ids"].to("cuda:0")
    print(f"system prefix cached: {_SYS_IDS.shape[1]} tokens")
    return _SYS_IDS


@torch.inference_mode()
def llm(user_content, max_new_tokens=800, temperature=0.8, top_p=0.95):
    """user_content: plain string for the user turn.
    System prompt is prepended automatically from cached token IDs.
    """
    ensure_model_loaded()
    sys_ids = _ensure_sys_ids()
    user_text = tok.apply_chat_template(
        [{"role": "user", "content": user_content}],
        tokenize=False, add_generation_prompt=True,
    )
    user_ids = tok(user_text, return_tensors="pt",
                   add_special_tokens=False)["input_ids"].to("cuda:0")

    input_ids = torch.cat([sys_ids, user_ids], dim=1)
    # Preserve system prefix; trim user content from the left if over budget
    if input_ids.shape[1] > 16_000:
        keep_user = 16_000 - sys_ids.shape[1]
        user_ids  = user_ids[:, -keep_user:]
        input_ids = torch.cat([sys_ids, user_ids], dim=1)

    attn = torch.ones_like(input_ids)
    out  = model.generate(
        input_ids=input_ids, attention_mask=attn,
        max_new_tokens=max_new_tokens,       # was 1200 ? signals are short
        do_sample=temperature > 0,
        temperature=max(temperature, 1e-5),
        top_p=top_p,
        pad_token_id=tok.pad_token_id,
    )
    gen = out[0, input_ids.shape[1]:]
    return tok.decode(gen, skip_special_tokens=True).strip()

print("llm() ready (lazy system-prefix cache)")


## Cell 8 â€” Prompts + candidate extraction

The system prompt includes a **feature catalog** so the model knows what
building blocks are available. The user prompt includes the research log,
the latest reflection memo, and recent failure examples.


In [ ]:
SYSTEM_PROMPT = """You are a quantitative researcher discovering alpha signals for a long-short equity strategy on ~100 US stocks.

Given:
- `close`  : pd.DataFrame of daily adjusted close prices (rows=dates, cols=tickers)
- `volume` : pd.DataFrame of daily volumes (same shape)
- `vix`    (optional): pd.Series of daily VIX levels, aligned to close.index
- `tnx`    (optional): pd.Series of daily 10Y Treasury yield (%), aligned to close.index

Write: def signal(close, volume, vix=None, tnx=None) -> pd.DataFrame
Return values in [-1, +1]. +1 = strongest long, -1 = strongest short.
Positions are lagged by 1 day - no lookahead needed in your code.

Hard rules:
- Use ONLY numpy (np) and pandas (pd). Do NOT write any import statements.
- Return shape MUST equal close.shape.
- Keep functions under 40 lines.
- End EVERY signal with:
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
- Do NOT use np.sign(rank - 0.5), np.sign(rank), or any one-sided threshold that can collapse to all-long/all-short.
- Prefer percentile ranks: r = feature.rank(axis=1, pct=True); out = (r - 0.5) * 2; then row-demean as above.

CRITICAL - lookahead bias auto-rejects your signal:
- NEVER use full-series aggregations: close.mean(), volume.std(), close.quantile()
- ONLY use trailing windows: .pct_change(N), .rolling(N).mean(), .ewm(span=N), .shift(N>0)
- Cross-sectional ops on a single date are fine: .rank(axis=1), .mean(axis=1)

CRITICAL - signal family discipline:
- Allowed mutation types in this pass: span_tweak, rank_normalization/rank_norm, vol_scaling/vol_scale, volume_gate, regime_gate, ts_momentum, short_reversal, vol_adjusted, volume_confirm, regime_momentum, multi_factor.
- Allowed factor families: ewm, momentum, mean_reversion, volume, volatility, regime, multi_factor.
- Preserve the parent signal's core idea when mutating a winner anchor.
- Use cross-sectional percentile ranks and trailing windows; no unsupported data sources.

We optimize on market-neutral train Sharpe with walk-forward robustness, penalising beta and turnover.
If a hypothesis is contrarian, encode the negative sign directly in the signal formula.

Regime conditioning with vix / tnx (declare as optional keyword args):
  vix and tnx are pd.Series - use .values[:,None] to broadcast over tickers.
"""

USER_PROMPT_TMPL = """## Research log (sorted by selection score)
{log_summary}

## Deterministic baselines (sorted by selection score)
{baseline_summary}

## Winner anchor
{winner_anchor}

## Parent pool for this batch
{parent_summary}

## Research memo
{memo}
{failures}

---
Batch {iter_n} of {n_batches}.

Propose 3 MUTATED signal functions.
At least TWO candidates MUST mutate the winner anchor cluster above.
Each candidate MUST preserve the selected parent family's core structure; EWM parents should keep their EWM backbone.
Use only these mutation types: span_tweak, rank_normalization/rank_norm, vol_scaling/vol_scale, volume_gate, regime_gate, ts_momentum, short_reversal, vol_adjusted, volume_confirm, regime_momentum, multi_factor.
Volume, volatility, regime, reversal, trend, and multi-factor variants are allowed when they use trailing data and remain market-neutral.
Do NOT abandon the parent cluster without declaring and justifying the mutation family.
Do NOT repeat signals already in the research log or deterministic baselines.
Do NOT include import lines.
Aim to improve selection score while keeping beta and turnover low.

Output format - follow EXACTLY:

=== CANDIDATE 1 ===
PARENT_ID: <parent id copied exactly from the pool>
MUTATION_TYPE: <span_tweak|rank_normalization|rank_norm|vol_scaling|vol_scale|volume_gate|regime_gate|ts_momentum|short_reversal|vol_adjusted|volume_confirm|regime_momentum|multi_factor>
HYPOTHESIS: <one sentence>
FAMILY: <ewm|momentum|mean_reversion|volume|volatility|regime|multi_factor>
```python
def signal(close, volume, vix=None, tnx=None):
    ...
    return out
```

=== CANDIDATE 2 ===
PARENT_ID: <...>
MUTATION_TYPE: <...>
HYPOTHESIS: <one sentence>
FAMILY: <...>
```python
def signal(close, volume, vix=None, tnx=None):
    ...
    return out
```

=== CANDIDATE 3 ===
PARENT_ID: <...>
MUTATION_TYPE: <...>
HYPOTHESIS: <one sentence>
FAMILY: <...>
```python
def signal(close, volume, vix=None, tnx=None):
    ...
    return out
```
"""

MOE_EXPERT_FOCUS = {
    "span_expert": """Focus on span_tweak and rank_normalization around strong EWM clusters.
Propose span moves that are non-trivial and not near-duplicates of current winners.""",
    "regime_expert": """Focus on regime_gate and robust regime logic using vix/tnx as light modifiers.
Avoid only repeating low-VIX gating; include diverse regime hypotheses while keeping EWM backbone.""",
    "risk_expert": """Focus on robustness: improve wf_min and reduce instability.
Prioritize mutations likely to keep beta near zero and avoid fragile, high-variance transforms.""",
    "turnover_expert": """Focus on turnover control and smoother signals.
Prefer transformations that reduce churn while preserving train Sharpe and walk-forward stability.""",
}

def build_expert_prompt(base_prompt, expert_id, mutation_priors_text="", expert_budget_text=""):
    focus = MOE_EXPERT_FOCUS.get(expert_id, "Focus on robust EWM mutations.")
    extra = [
        f"## Expert role\n{expert_id}",
        f"## Expert focus\n{focus}",
    ]
    if mutation_priors_text:
        extra.append(f"## Mutation priors\n{mutation_priors_text}")
    if expert_budget_text:
        extra.append(f"## Expert budget\n{expert_budget_text}")
    extra.append(
        "Output exactly 3 candidates. Keep diversity across clusters and mutation types. "
        "Do not repeat signatures or near-duplicate span pairs."
    )
    return base_prompt + "\n\n" + "\n\n".join(extra)

REFLECTION_PROMPT = """Review this experiment log and write a research memo.

## Successful signals (sorted by selection score)
{successes}

## Cluster summary
{cluster_summary}

## Failures
{failure_summary}

## Current best
{best_info}

Write a concise RESEARCH MEMO (5-7 bullet points):
1. Which clusters, factor families, and mutation types show the most promise? Cite score and train Sharpe values.
2. What short/long span ranges work best?
3. Which mutation types are wasting batches?
4. What specific next mutations should be tried?
5. Which light modifiers help without hurting turnover or beta?
Be specific and quantitative.
"""

SIGNAL_FAMILIES = {
    "momentum": ["pct_change", "momentum", "12_1", "12-1", "trend", "ts_momentum", "time_series"],
    "mean_reversion": ["reversion", "reversal", "short_reversal", "bollinger", "zscore", "z_score", "mean_rev", "contrarian", "overbought"],
    "volume": ["volume", "vol_ratio", "turnover", "amihud", "volume_confirm", "liquidity"],
    "volatility": ["volatility", ".std()", "variance", "atr", "realised", "realized", "vol_adjusted", "vol_scale"],
    "ewm": ["ewm(", "ema", "exponential", "crossover", "fast / slow", "fast/slow"],
    "regime": ["vix", "tnx", "regime", "conditional", "high_vol", "rising", "state", "regime_momentum"],
    "multi_factor": ["combine", "average", "blend", "weight", "composite", "multi_factor", "multi factor", "ensemble"],
}
VALID_FAMILIES = list(SIGNAL_FAMILIES) + ["other"]
VALID_MUTATIONS = {'span_tweak', 'rank_normalization', 'rank_norm', 'vol_scaling', 'vol_scale', 'volume_gate', 'regime_gate', 'ts_momentum', 'short_reversal', 'vol_adjusted', 'volume_confirm', 'regime_momentum', 'multi_factor'}
_NUM_RE = re.compile(r"\b\d+\b")
_EWM_SPAN_RE = re.compile(r"ewm\(span\s*=\s*(\d+)\)", re.IGNORECASE)

def normalize_family(raw):
    if not raw:
        return None
    fam = raw.lower().strip()
    fam = re.sub(r"[`*_#>:\-.]+", " ", fam)
    fam = re.sub(r"\s+", "_", fam).strip("_")
    aliases = {
        "meanreversion": "mean_reversion",
        "mean_revert": "mean_reversion",
        "meanrev": "mean_reversion",
        "multifactor": "multi_factor",
    }
    fam = aliases.get(fam, fam)
    if fam in VALID_FAMILIES:
        return fam
    for candidate in VALID_FAMILIES:
        if candidate != "other" and candidate in fam:
            return candidate
    return None

def normalize_mutation(raw):
    if not raw:
        return "span_tweak"
    key = raw.lower().strip()
    key = re.sub(r"[`*_#>:\-.]+", "_", key)
    key = re.sub(r"\s+", "_", key).strip("_")
    aliases = {
        "parameter_tweak": "span_tweak",
        "param_tweak": "span_tweak",
        "normalization": "rank_norm",
        "rank_normalization": "rank_norm",
        "vol_scaling": "vol_scale",
        "volatility_scaling": "vol_scale",
        "regime_filter": "regime_gate",
        "factor_blend": "multi_factor",
        "multi_factor_blend": "multi_factor",
        "trend_momentum": "ts_momentum",
        "time_series_momentum": "ts_momentum",
        "volume_confirmation": "volume_confirm",
        "regime_filter_momentum": "regime_momentum",
    }
    key = aliases.get(key, key)
    return key if key in VALID_MUTATIONS else "span_tweak"

def extract_ewm_spans(code):
    spans = sorted({int(x) for x in _EWM_SPAN_RE.findall(code or "")})
    if not spans:
        return None, None
    if len(spans) == 1:
        return spans[0], spans[0]
    return spans[0], spans[-1]

def canonical_signal_shape(code):
    text = (code or "").lower()
    text = re.sub(r"#[^\n]*", "", text)
    text = _NUM_RE.sub("N", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def normalize_aux_params(params):
    params = params or {}
    out = {}
    for k, v in params.items():
        if isinstance(v, (int, float, np.integer, np.floating)):
            out[k] = float(round(float(v), 6))
        else:
            out[k] = v
    return dict(sorted(out.items()))

def deterministic_family_for_mutation(mutation_type):
    key = normalize_mutation(mutation_type)
    mapping = {
        "plain": ("ewm", "ewm"),
        "span_tweak": ("ewm", "ewm"),
        "rank_norm": ("ewm", "ewm"),
        "rank_normalization": ("ewm", "ewm"),
        "vol_scale": ("ewm_volscale", "ewm"),
        "vol_scaling": ("ewm_volscale", "ewm"),
        "volume_gate": ("ewm_volume", "ewm"),
        "regime_gate": ("ewm_regime", "ewm"),
        "ts_momentum": ("momentum", "momentum"),
        "short_reversal": ("mean_reversion", "mean_reversion"),
        "vol_adjusted": ("volatility_momentum", "momentum"),
        "volume_confirm": ("volume_momentum", "momentum"),
        "regime_momentum": ("regime_momentum", "momentum"),
        "multi_factor": ("multi_factor", "multi_factor"),
    }
    return mapping.get(key, ("ewm", "ewm"))

def cluster_id_for_signal(base_family, mutation_type, short_span, long_span, params=None):
    params = normalize_aux_params(params)
    key = normalize_mutation(mutation_type)
    family, mapped_base = deterministic_family_for_mutation(key)
    prefix = family or base_family or mapped_base or "signal"
    if short_span is None or long_span is None:
        return f"{prefix}:generic"
    if key in ("regime_gate", "regime_momentum"):
        thr = params.get("vix_threshold")
        return f"{prefix}:{short_span}:{long_span}:{thr}" if thr is not None else f"{prefix}:{short_span}:{long_span}"
    if key in ("vol_scale", "vol_scaling", "vol_adjusted", "multi_factor"):
        win = params.get("vol_window")
        return f"{prefix}:{short_span}:{long_span}:{win}" if win is not None else f"{prefix}:{short_span}:{long_span}"
    if key in ("volume_gate", "volume_confirm"):
        win = params.get("vol_gate_window")
        return f"{prefix}:{short_span}:{long_span}:{win}" if win is not None else f"{prefix}:{short_span}:{long_span}"
    if key == "ts_momentum":
        return f"{prefix}:{long_span}"
    if key == "short_reversal":
        return f"{prefix}:{short_span}"
    return f"{prefix}:{short_span}:{long_span}"

def signature_for_signal(family, mutation_type, code, short_span=None, long_span=None, base_family=None, params=None, cluster_id=None):
    if short_span is None or long_span is None:
        short_span, long_span = extract_ewm_spans(code)
    params = normalize_aux_params(params)
    base_family = base_family or family
    if cluster_id is None:
        cluster_id = cluster_id_for_signal(base_family, mutation_type, short_span, long_span, params=params)
    signature_payload = {
        "family": family,
        "base_family": base_family,
        "mutation_type": mutation_type,
        "short_span": short_span,
        "long_span": long_span,
        "params": params,
    }
    if short_span is None or long_span is None:
        signature_payload["shape"] = canonical_signal_shape(code)
    signature = hashlib.sha1(json.dumps(signature_payload, sort_keys=True, separators=(",", ":")).encode()).hexdigest()[:16]
    return {
        "signature": signature,
        "cluster_id": cluster_id,
        "short_span": short_span,
        "long_span": long_span,
        "base_family": base_family,
        "params": params,
    }

def classify_signal(entry):
    text = (entry.get("hypothesis", "") + " " + entry.get("code", "")).lower()
    found = [fam for fam, kws in SIGNAL_FAMILIES.items() if any(kw in text for kw in kws)]
    return found if found else ["other"]

_CAND_RE = re.compile(r"===\s*CANDIDATE\s*(\d+)\s*===(.*?)(?====\s*CANDIDATE|\Z)", re.DOTALL | re.IGNORECASE)
_CODE_RE = re.compile(r"```(?:python)?\s*\n(.*?)```", re.DOTALL)
_HYP_RE = re.compile(r"HYPOTHESIS:\s*(.+?)(?:\n|$)", re.IGNORECASE)
_FAM_RE = re.compile(r"FAMILY:\s*(.+?)(?:\n|$)", re.IGNORECASE)
_PARENT_RE = re.compile(r"PARENT_ID:\s*(.+?)(?:\n|$)", re.IGNORECASE)
_MUT_RE = re.compile(r"MUTATION_TYPE:\s*(.+?)(?:\n|$)", re.IGNORECASE)

def extract_candidates(text):
    results = []
    for m in _CAND_RE.finditer(text):
        body = m.group(2)
        cm = _CODE_RE.search(body)
        if not cm:
            continue
        hm = _HYP_RE.search(body)
        fm = _FAM_RE.search(body)
        pm = _PARENT_RE.search(body)
        mm = _MUT_RE.search(body)
        hyp = hm.group(1).strip() if hm else ""
        code = cm.group(1).strip()
        family = normalize_family(fm.group(1).strip() if fm else "")
        if family is None:
            family = classify_signal({"hypothesis": hyp, "code": code})[0]
        mutation_type = normalize_mutation(mm.group(1).strip() if mm else "span_tweak")
        meta = signature_for_signal(family, mutation_type, code)
        results.append({
            "idx": int(m.group(1)),
            "hypothesis": hyp,
            "family": family,
            "code": code,
            "parent_id": pm.group(1).strip() if pm else "unspecified",
            "mutation_type": mutation_type,
            **meta,
        })
    if not results:
        for i, cm in enumerate(_CODE_RE.finditer(text)):
            code_text = cm.group(1).strip()
            if "def signal" in code_text:
                family = classify_signal({"hypothesis": "", "code": code_text})[0]
                meta = signature_for_signal(family, "span_tweak", code_text)
                results.append({
                    "idx": i + 1,
                    "hypothesis": "",
                    "family": family,
                    "code": code_text,
                    "parent_id": "unspecified",
                    "mutation_type": "span_tweak",
                    **meta,
                })
    return results[:3]


## Cell 9 - Deterministic baseline sweep

Run a small hand-designed parameter sweep before the LLM loop.
This gives the notebook a sane starting frontier and lets the model improve
existing winners instead of rediscovering basic signal families from zero.


In [ ]:
BASELINE_RESULTS = []
DETERMINISTIC_RESULTS = []
BASELINE_FILE = OUT / "baseline_results.json"
SELECTION_OBJECTIVE = "market_neutral_net_sharpe"
BETA_LIMIT = 0.10
TURNOVER_LIMIT = 0.50
MIN_ACTIVE_TURNOVER = 0.002
MIN_SIGNAL_ACTIVITY = 0.50
MIN_RAW_CS_STD = 0.02
MIN_LONG_SHORT_FRAC = 0.08
WF_WINDOWS = 4
NEAR_DUP_SHORT = 2
NEAR_DUP_LONG = 6
EVOLVE_ENABLED = True
EVOLVE_MAX_GENERATIONS = 40
EVOLVE_TRIALS_PER_GENERATION = 160
EVOLVE_BEAM_WIDTH = 16
EVOLVE_SURVIVORS = 4
EVOLVE_MIN_GEN_GROWTH = 0.01
EVOLVE_MAX_STALLED_GENS = 20
EVOLVE_MIN_GENERATIONS = 8
EVOLVE_PARENT_MIN_TRIALS = 12
EVOLVE_PARENT_MAX_TRIALS = 60
EVOLVE_MIN_TRIALS_PER_GENERATION = 100
EVOLVE_MAX_TOTAL_EVALS = 9600
EVOLVE_MAX_WALLCLOCK_HOURS = 2.0
EVOLVE_RANDOM_EXPLORATION_FRAC = 0.20
EVOLVE_REQUIRE_NOVELTY_FOR_STALL = True
EVOLVE_MIN_NEW_CLUSTERS_PER_GEN = 1
EVOLVE_ROBUST_RESET_THRESHOLD = 20
EVOLVE_ZERO_ROBUST_PATIENCE = 3
EVOLVE_ZERO_ROBUST_MIN_GENERATIONS = 3
EVOLVE_ZERO_ROBUST_MIN_VALID_RATE = 0.03
EVOLVE_PARTIAL_WRITE_EVERY = 1
ROBUSTNESS_SCORE_FLOOR = 45.0
EVOLVE_PROGRAM_ROBUSTNESS_FLOOR = 60.0
EVOLVE_PROGRAM_MIN_VAL_WF_MEDIAN = 0.10
EVOLVE_PROGRAM_MIN_VAL_WF_MIN = -0.10
EVOLVE_VAL_WF_FLOOR_PENALTY = 2.00
EVOLVE_VAL_WF_MISSING_PENALTY = 0.35
EVOLVE_MIN_FOCUS_VARIANTS = 5
EVOLVE_PROXY_WF_SPREAD_PENALTY = 0.25
EVOLVE_PROXY_WFMIN_PENALTY = 0.20
EVOLVE_PROXY_CONSISTENCY_PENALTY = 0.10
EVOLVE_TOURNAMENT_K = 5
EVOLVE_DIVERSITY_PENALTY = 0.15
EVOLVE_CONVERGENCE_PATIENCE = 25
EVOLVE_STAGNATION_PATIENCE = 3
EVOLVE_HARD_STAGNATION_PATIENCE = 6
EVOLVE_MIN_BEST_GAIN = 0.003
EVOLVE_MIN_MEDIAN_GAIN = 0.002
EVOLVE_MIN_NOVELTY_GAIN = 1
EVOLVE_MIN_DIVERSITY_GAIN = 0.01
EVOLVE_BENCHMARK_SOURCE = "deterministic"
EVOLVE_BENCHMARK_MUTATION = "regime_momentum"
EVOLVE_CHAMPION_VARIANTS = ["regime_momentum", "volume_gate", "volume_confirm", "rank_norm", "plain", "regime_gate", "ts_momentum"]
EVOLVE_ADAPTIVE_DIAGNOSTIC_RUN = True
EVOLVE_PARENT_FAMILY_CAP = 0.40
EVOLVE_FAMILY_QUOTAS = {
    "volume_gate": 0.24,
    "volume_confirm": 0.20,
    "regime_momentum": 0.20,
    "rank_norm": 0.12,
    "plain": 0.08,
    "composite": 0.16,
}
EVOLVE_SAFE_COMPOSITE_PAIRS = [
    ("volume_gate", "rank_norm"),
    ("volume_confirm", "plain"),
    ("volume_gate", "regime_gate"),
    ("volume_confirm", "ts_momentum"),
]
EVOLVE_DIAGNOSIS_TOP_ROWS = 5
VAL_FRACTION = 0.20
EVAL_TIMEOUT_SEC = 25
EVOLVE_RESTART_ATTEMPTS = 6
EVOLVE_MIN_CANDIDATE_FLOOR = 32
EVOLVE_DETERMINISTIC_CHAMPIONS = 12
EVOLVE_COMPLEXITY_PENALTY = 0.025
EVOLVE_TRAIN_VAL_GAP_PENALTY = 1.25
EVOLVE_VOL_SCALE_PENALTY = 0.015
EVOLVE_BETA_EXCESS_PENALTY = 8.00
EVOLVE_TRAIN_WF_MIN_FLOOR = -0.20
EVOLVE_TRAIN_WF_MIN_PENALTY = 3.00
ECONOMIC_SHARPE_FLOOR = 0.50
ECONOMIC_EDGE_OVER_DETERMINISTIC = 0.05
EVOLUTION_SUMMARY_FILE = OUT / "evolution_summary.json"
EVOLUTION_LINEAGE_FILE = OUT / "evolution_lineage.json"
EVOLUTION_MEMORY_FILE = OUT / "evolution_memory.json"
EVOLUTION_PROGRAM_FILE = OUT / "evolution_program.json"
RUN_MANIFEST_FILE = OUT / "run_manifest.json"
PARTIAL_REPORT_FILE = OUT / "partial_run_report.json"
SCHEMA_VERSION = "evolution-v3"
_WARNINGS = []


def _warn(msg):
    _WARNINGS.append(str(msg))
    print(f"[warn] {msg}")


def _atomic_write_text(path_obj, text):
    tmp = path_obj.with_suffix(path_obj.suffix + ".tmp")
    tmp.write_text(text)
    tmp.replace(path_obj)


def _atomic_write_json(path_obj, obj):
    _atomic_write_text(path_obj, json.dumps(obj, indent=2))
    try:
        if "wandb_log" in globals():
            stem = getattr(path_obj, "stem", "json")
            payload = {}
            if isinstance(obj, list):
                rows = [r for r in obj if isinstance(r, dict)]
                payload[f"{stem}/rows"] = len(rows)
                payload[f"{stem}/errors"] = sum(1 for r in rows if r.get("error"))
                payload[f"{stem}/robust"] = sum(1 for r in rows if r.get("robust_ok"))
                scored = [r for r in rows if isinstance(r.get("score"), (int, float))]
                if scored:
                    best = max(scored, key=lambda r: r["score"])
                    payload[f"{stem}/best_score"] = float(best["score"])
                    for key in ("train_sharpe", "wf_median", "wf_min", "beta", "turnover"):
                        if isinstance(best.get(key), (int, float)):
                            payload[f"{stem}/best_{key}"] = float(best[key])
            elif isinstance(obj, dict):
                for key in ("generations_executed", "total_evals", "best_score"):
                    if isinstance(obj.get(key), (int, float)):
                        payload[f"{stem}/{key}"] = float(obj[key])
                latest = obj.get("latest") if isinstance(obj.get("latest"), dict) else {}
                for key in ("gen_id", "valid_count", "robust_count", "best_score", "best_gain", "diversity"):
                    if isinstance(latest.get(key), (int, float)):
                        payload[f"{stem}/latest_{key}"] = float(latest[key])
            if payload:
                wandb_log(payload)
    except Exception:
        pass


def row_identity(row, default_prefix="row"):
    if not isinstance(row, dict):
        return f"{default_prefix}:unknown"
    for key in ("parent_id", "iter", "program_hash", "signature", "cluster_id"):
        value = row.get(key)
        if value not in (None, ""):
            return str(value)
    return f"{default_prefix}:unknown"


def _clean_rows_for_json(rows):
    cleaned = []
    for row in rows:
        if not isinstance(row, dict):
            continue
        out = {k: v for k, v in row.items() if k != "_val_ret"}
        cleaned.append(out)
    return cleaned


def _score_sort_value(row):
    try:
        score = float(row.get("score", -999.0))
        if np.isfinite(score):
            return score
    except Exception:
        pass
    return -999.0


def _sorted_artifact_rows(rows):
    cleaned = _clean_rows_for_json(rows)
    deduped = {}
    ordered = []
    for row in cleaned:
        key = (
            row.get("signature")
            or row.get("program_hash")
            or row.get("program_sig")
            or row.get("cluster_id")
            or row_identity(row)
        )
        prev = deduped.get(key)
        if prev is None:
            deduped[key] = row
            ordered.append(key)
            continue
        prev_tuple = (
            bool(prev.get("robust_ok", False)),
            _score_sort_value(prev),
            float(prev.get("robustness_score", 0.0) or 0.0),
        )
        row_tuple = (
            bool(row.get("robust_ok", False)),
            _score_sort_value(row),
            float(row.get("robustness_score", 0.0) or 0.0),
        )
        if row_tuple > prev_tuple:
            deduped[key] = row
    return sorted(
        [deduped[key] for key in ordered],
        key=lambda r: (
            not bool(r.get("robust_ok", False)),
            -_score_sort_value(r),
            row_identity(r),
        ),
    )


def _best_score_from_rows(rows):
    scores = [_score_sort_value(r) for r in rows if isinstance(r, dict) and r.get("score") is not None]
    return max(scores) if scores else None


def _benchmark_policy_payload():
    return {
        "source": EVOLVE_BENCHMARK_SOURCE,
        "anchor": "best_deterministic_by_train_score",
        "fallback_seed_mutation_type": EVOLVE_BENCHMARK_MUTATION,
        "selection_objective": SELECTION_OBJECTIVE,
        "deterministic_champions": EVOLVE_DETERMINISTIC_CHAMPIONS,
        "economic_sharpe_floor": ECONOMIC_SHARPE_FLOOR,
        "economic_edge_over_deterministic": ECONOMIC_EDGE_OVER_DETERMINISTIC,
    }


def _evolution_policy_payload():
    return {
        "max_generations": EVOLVE_MAX_GENERATIONS,
        "trials_per_generation": EVOLVE_TRIALS_PER_GENERATION,
        "beam_width": EVOLVE_BEAM_WIDTH,
        "survivors": EVOLVE_SURVIVORS,
        "min_generations": EVOLVE_MIN_GENERATIONS,
        "min_trials_per_generation": EVOLVE_MIN_TRIALS_PER_GENERATION,
        "robust_reset_threshold": EVOLVE_ROBUST_RESET_THRESHOLD,
        "zero_robust_patience": EVOLVE_ZERO_ROBUST_PATIENCE,
        "zero_robust_min_generations": EVOLVE_ZERO_ROBUST_MIN_GENERATIONS,
        "zero_robust_min_valid_rate": EVOLVE_ZERO_ROBUST_MIN_VALID_RATE,
        "stagnation_patience": EVOLVE_STAGNATION_PATIENCE,
        "partial_write_every": EVOLVE_PARTIAL_WRITE_EVERY,
        "random_exploration_frac": EVOLVE_RANDOM_EXPLORATION_FRAC,
        "convergence_patience": EVOLVE_CONVERGENCE_PATIENCE,
        "min_best_gain": EVOLVE_MIN_BEST_GAIN,
        "min_median_gain": EVOLVE_MIN_MEDIAN_GAIN,
        "min_novelty_gain": EVOLVE_MIN_NOVELTY_GAIN,
        "min_diversity_gain": EVOLVE_MIN_DIVERSITY_GAIN,
        "diversity_penalty": EVOLVE_DIVERSITY_PENALTY,
        "complexity_penalty": EVOLVE_COMPLEXITY_PENALTY,
        "train_val_gap_penalty": EVOLVE_TRAIN_VAL_GAP_PENALTY,
        "vol_scale_penalty": EVOLVE_VOL_SCALE_PENALTY,
        "beta_excess_penalty": EVOLVE_BETA_EXCESS_PENALTY,
        "train_wf_min_floor": EVOLVE_TRAIN_WF_MIN_FLOOR,
        "train_wf_min_penalty": EVOLVE_TRAIN_WF_MIN_PENALTY,
        "robustness_score_floor": ROBUSTNESS_SCORE_FLOOR,
        "deterministic_champions": EVOLVE_DETERMINISTIC_CHAMPIONS,
        "economic_sharpe_floor": ECONOMIC_SHARPE_FLOOR,
        "economic_edge_over_deterministic": ECONOMIC_EDGE_OVER_DETERMINISTIC,
        "adaptive_diagnostic_run": EVOLVE_ADAPTIVE_DIAGNOSTIC_RUN,
        "family_quotas": dict(EVOLVE_FAMILY_QUOTAS),
        "parent_family_cap": EVOLVE_PARENT_FAMILY_CAP,
        "safe_composite_pairs": list(EVOLVE_SAFE_COMPOSITE_PAIRS),
        "max_total_evals": EVOLVE_MAX_TOTAL_EVALS,
        "max_wallclock_hours": EVOLVE_MAX_WALLCLOCK_HOURS,
        "train_val_firewall": True,
        "diagnosis_top_rows": EVOLVE_DIAGNOSIS_TOP_ROWS,
        "benchmark_policy": _benchmark_policy_payload(),
    }


def _robustness_failure_counts(rows):
    counts = {
        "train_sharpe": 0,
        "wf_median": 0,
        "wf_min": 0,
        "beta": 0,
        "turnover": 0,
        "signal_activity": 0,
        "raw_cs_std": 0,
        "long_short_balance": 0,
        "robustness_score": 0,
    }
    for row in rows or []:
        if not isinstance(row, dict):
            continue
        if row.get("train_sharpe", -99) <= 0:
            counts["train_sharpe"] += 1
        if row.get("wf_median", -99) <= 0:
            counts["wf_median"] += 1
        if row.get("wf_min", -99) <= -0.20:
            counts["wf_min"] += 1
        if abs(float(row.get("beta", 0.0) or 0.0)) > BETA_LIMIT:
            counts["beta"] += 1
        turnover = float(row.get("turnover", row.get("avg_turnover", 0.0)) or 0.0)
        if not (MIN_ACTIVE_TURNOVER <= turnover <= TURNOVER_LIMIT):
            counts["turnover"] += 1
        if float(row.get("signal_activity", 1.0) or 0.0) < MIN_SIGNAL_ACTIVITY:
            counts["signal_activity"] += 1
        if float(row.get("raw_cs_std", 1.0) or 0.0) < MIN_RAW_CS_STD:
            counts["raw_cs_std"] += 1
        if min(
            float(row.get("raw_long_frac", 1.0) or 0.0),
            float(row.get("raw_short_frac", 1.0) or 0.0),
        ) < MIN_LONG_SHORT_FRAC:
            counts["long_short_balance"] += 1
        if float(row.get("robustness_score", 0.0) or 0.0) < ROBUSTNESS_SCORE_FLOOR:
            counts["robustness_score"] += 1
    return counts


def _generation_diagnosis_events(generation, valid_rows, robust_rows, benchmark_row=None):
    generation = generation or {}
    valid_rows = [r for r in (valid_rows or []) if isinstance(r, dict)]
    robust_rows = [r for r in (robust_rows or []) if isinstance(r, dict)]

    def _snapshot_rows(rows):
        ranked = sorted(
            rows,
            key=lambda row: (-float(row.get("score", -999.0) or -999.0), str(row.get("parent_id") or row.get("signature") or "")),
        )
        snapped = []
        for row in ranked[: max(1, int(EVOLVE_DIAGNOSIS_TOP_ROWS))]:
            snapped.append(
                {
                    "parent_id": row.get("parent_id"),
                    "signature": row.get("signature"),
                    "cluster_id": row.get("cluster_id"),
                    "family": row.get("family"),
                    "mutation_type": row.get("mutation_type"),
                    "score": row.get("score"),
                    "robust_ok": bool(row.get("robust_ok")),
                    "train_sharpe": row.get("train_sharpe"),
                    "wf_median": row.get("wf_median"),
                    "wf_min": row.get("wf_min"),
                    "beta": row.get("beta"),
                    "turnover": row.get("turnover", row.get("avg_turnover")),
                    "signal_activity": row.get("signal_activity"),
                    "raw_cs_std": row.get("raw_cs_std"),
                    "raw_long_frac": row.get("raw_long_frac"),
                    "raw_short_frac": row.get("raw_short_frac"),
                    "robustness_score": row.get("robustness_score"),
                    "error": row.get("error"),
                }
            )
        return snapped

    benchmark_snapshot = None
    benchmark_score = None
    if isinstance(benchmark_row, dict):
        benchmark_snapshot = {
            "parent_id": benchmark_row.get("parent_id"),
            "signature": benchmark_row.get("signature"),
            "cluster_id": benchmark_row.get("cluster_id"),
            "family": benchmark_row.get("family"),
            "mutation_type": benchmark_row.get("mutation_type"),
            "score": benchmark_row.get("score"),
            "source": benchmark_row.get("source", EVOLVE_BENCHMARK_SOURCE),
        }
        try:
            benchmark_score = float(benchmark_row.get("score")) if benchmark_row.get("score") is not None else None
        except Exception:
            benchmark_score = None

    best_score = generation.get("best_score")
    best_so_far = generation.get("best_so_far")
    try:
        best_score_value = float(best_score) if best_score is not None else None
    except Exception:
        best_score_value = None
    try:
        best_so_far_value = float(best_so_far) if best_so_far is not None else None
    except Exception:
        best_so_far_value = None

    base_payload = {
        "gen_id": generation.get("gen_id"),
        "valid_count": int(generation.get("valid_count", len(valid_rows)) or 0),
        "valid_rate": float(generation.get("valid_rate", 0.0) or 0.0),
        "robust_count": int(generation.get("robust_count", len(robust_rows)) or 0),
        "best_score": best_score,
        "best_so_far": best_so_far,
        "benchmark_gap": None if best_score_value is None or benchmark_score is None else float(best_score_value - benchmark_score),
        "benchmark_policy": _benchmark_policy_payload(),
        "benchmark": benchmark_snapshot,
        "top_valid_rows": _snapshot_rows(valid_rows),
        "top_robust_rows": _snapshot_rows(robust_rows),
    }
    events = []

    best_so_far_streak = int(generation.get("best_so_far_streak", 0) or 0)
    if best_so_far_streak >= EVOLVE_STAGNATION_PATIENCE:
        events.append(
            {
                **base_payload,
                "kind": "stagnation",
                "trigger": "best_so_far_streak",
                "streak": best_so_far_streak,
                "new_cluster_count": int(generation.get("new_cluster_count", 0) or 0),
                "diversity": float(generation.get("diversity", 0.0) or 0.0),
            }
        )

    zero_robust_streak = int(generation.get("zero_robust_streak", 0) or 0)
    if generation.get("zero_robust_counted") and zero_robust_streak >= 1:
        failure_rows = [r for r in valid_rows if not r.get("robust_ok")]
        events.append(
            {
                **base_payload,
                "kind": "zero_robust",
                "trigger": "zero_robust_streak",
                "streak": zero_robust_streak,
                "failure_counts": _robustness_failure_counts(failure_rows),
            }
        )
    return events


def _run_flags_payload():
    return {
        "RUN_BASELINE_SWEEP": bool(globals().get("RUN_BASELINE_SWEEP", False)),
        "RUN_DETERMINISTIC_SEARCH": bool(globals().get("RUN_DETERMINISTIC_SEARCH", False)),
        "RUN_PARAM_SEARCH": bool(globals().get("RUN_PARAM_SEARCH", False)),
        "RUN_HELDOUT_EVAL": bool(globals().get("RUN_HELDOUT_EVAL", False)),
        "RUN_REPORTS": bool(globals().get("RUN_REPORTS", False)),
    }


def _portfolio_construction_payload():
    return {
        "beta_neutralize_positions": bool(globals().get("BETA_NEUTRALIZE_POSITIONS", False)),
        "beta_lookback": int(globals().get("BETA_NEUTRAL_LOOKBACK", 0) or 0),
        "beta_shifted": True,
    }


def _evolution_summary_payload(all_rows, generations, total_evals, stop_reason, diagnosis_events=None, partial=False, started=None):
    latest = generations[-1] if generations else {}
    diagnosis_events = list(diagnosis_events or [])
    payload = {
        "schema_version": SCHEMA_VERSION,
        "evolution_enabled": True,
        "artifact_status": "partial" if partial else "final",
        "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "policy": _evolution_policy_payload(),
        "portfolio_construction": _portfolio_construction_payload(),
        "stop_reason": stop_reason,
        "generations_executed": len(generations),
        "total_evals": int(total_evals),
        "best_score": _best_score_from_rows(all_rows),
        "warnings": list(_WARNINGS),
        "latest": latest,
        "generations": generations,
        "diagnosis_count": len(diagnosis_events),
        "latest_diagnosis": diagnosis_events[-1] if diagnosis_events else None,
        "diagnosis_events": diagnosis_events,
    }
    if started is not None:
        payload["elapsed_hours"] = float((time.time() - started) / 3600.0)
    return payload


def _run_manifest_payload(all_rows, generations, total_evals, stop_reason, diagnosis_events=None, partial=False, started=None):
    diagnosis_events = list(diagnosis_events or [])
    result = {
        "stop_reason": stop_reason,
        "artifact_status": "partial" if partial else "final",
        "generations_executed": len(generations),
        "total_evals": int(total_evals),
        "best_score": _best_score_from_rows(all_rows),
        "diagnosis_count": len(diagnosis_events),
        "latest_diagnosis": diagnosis_events[-1] if diagnosis_events else None,
    }
    if started is not None:
        result["elapsed_hours"] = float((time.time() - started) / 3600.0)
    return {
        "schema_version": SCHEMA_VERSION,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "python_version": sys.version,
        "selection_objective": SELECTION_OBJECTIVE,
        "benchmark_policy": _benchmark_policy_payload(),
        "portfolio_construction": _portfolio_construction_payload(),
        "flags": _run_flags_payload(),
        "result": result,
    }


def _partial_report_payload(all_rows, generations, total_evals, stop_reason, diagnosis_events=None, partial=False, started=None):
    artifact_rows = _sorted_artifact_rows(all_rows)
    robust_rows = [r for r in artifact_rows if r.get("robust_ok")]
    latest = generations[-1] if generations else {}
    diagnosis_events = list(diagnosis_events or [])
    return {
        "schema_version": SCHEMA_VERSION,
        "artifact_status": "partial" if partial else "final",
        "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "stop_reason": stop_reason,
        "generations_executed": len(generations),
        "total_evals": int(total_evals),
        "latest": latest,
        "best_score": _best_score_from_rows(artifact_rows),
        "robust_count": len(robust_rows),
        "top_rows": artifact_rows[:25],
        "top_robust_rows": robust_rows[:25],
        "diagnosis_count": len(diagnosis_events),
        "latest_diagnosis": diagnosis_events[-1] if diagnosis_events else None,
        "diagnosis_events": diagnosis_events,
        "warnings": list(_WARNINGS),
        "elapsed_hours": None if started is None else float((time.time() - started) / 3600.0),
    }


def _write_evolution_artifacts(all_rows, generations, memory_notes, programs, diagnosis_events, total_evals, stop_reason, partial=False, started=None, suppress_errors=False):
    def _write_all():
        artifact_rows = _sorted_artifact_rows(all_rows)
        _atomic_write_json(PARAM_SEARCH_FILE, artifact_rows)
        _atomic_write_json(EVOLUTION_LINEAGE_FILE, artifact_rows)
        _atomic_write_json(
            EVOLUTION_SUMMARY_FILE,
            _evolution_summary_payload(
                artifact_rows,
                generations,
                total_evals,
                stop_reason,
                diagnosis_events=diagnosis_events,
                partial=partial,
                started=started,
            ),
        )
        _atomic_write_json(
            PARTIAL_REPORT_FILE,
            _partial_report_payload(
                artifact_rows,
                generations,
                total_evals,
                stop_reason,
                diagnosis_events=diagnosis_events,
                partial=partial,
                started=started,
            ),
        )
        _atomic_write_json(
            EVOLUTION_MEMORY_FILE,
            {
                "schema_version": SCHEMA_VERSION,
                "artifact_status": "partial" if partial else "final",
                "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                "memo_seed": _safe_load_memo_text(),
                "generation_reflections": memory_notes,
                "generation_programs": programs,
                "diagnosis_events": diagnosis_events,
            },
        )
        _atomic_write_json(
            EVOLUTION_PROGRAM_FILE,
            {
                "schema_version": SCHEMA_VERSION,
                "artifact_status": "partial" if partial else "final",
                "updated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                "programs": programs,
            },
        )
        _atomic_write_json(
            RUN_MANIFEST_FILE,
            _run_manifest_payload(
                artifact_rows,
                generations,
                total_evals,
                stop_reason,
                diagnosis_events=diagnosis_events,
                partial=partial,
                started=started,
            ),
        )
        return artifact_rows

    if suppress_errors:
        try:
            return _write_all()
        except Exception as e:
            _warn(f"partial evolution artifact write failed: {e}")
            return _clean_rows_for_json(all_rows)
    return _write_all()


def robustness_score(metric_row):
    """0-100 diagnostic score for real, stable market-neutral behavior."""
    row = metric_row or {}
    train_sh = float(row.get("train_sharpe", row.get("sharpe", 0.0)) or 0.0)
    wf_median = float(row.get("wf_median", 0.0) or 0.0)
    wf_min = float(row.get("wf_min", 0.0) or 0.0)
    beta = abs(float(row.get("beta", 0.0) or 0.0))
    turnover = float(row.get("turnover", row.get("avg_turnover", 0.0)) or 0.0)
    consistency = float(row.get("consistency", 0.0) or 0.0)
    activity = float(row.get("signal_activity", 1.0) or 0.0)
    cs_std = float(row.get("raw_cs_std", 1.0) or 0.0)
    long_frac = float(row.get("raw_long_frac", 1.0) or 0.0)
    short_frac = float(row.get("raw_short_frac", 1.0) or 0.0)

    score = 0.0
    score += 22.0 * min(max(train_sh, 0.0), 1.0)
    score += 20.0 * min(max(wf_median, 0.0), 0.8) / 0.8
    score += 14.0 * min(max(wf_min + 0.35, 0.0), 0.55) / 0.55
    score += 12.0 * max(0.0, 1.0 - beta / max(BETA_LIMIT, 1e-6))
    score += 10.0 if MIN_ACTIVE_TURNOVER <= turnover <= TURNOVER_LIMIT else 0.0
    score += 8.0 * min(max(consistency, 0.0), 1.0)
    score += 6.0 * min(max(activity, 0.0), 1.0)
    score += 4.0 * min(max(cs_std, 0.0), 0.12) / 0.12
    score += 2.0 * min(max(min(long_frac, short_frac), 0.0), 0.25) / 0.25
    return float(round(score, 4))


def selection_score(sharpe, wf_median=0.0, consistency=0.0, beta=0.0, avg_turnover=0.0, wf_min=0.0, max_dd=0.0):
    """Score market-neutral net Sharpe, not benchmark-relative long-only spread."""
    turnover_floor_penalty = 5.0 * max(0.0, MIN_ACTIVE_TURNOVER - float(avg_turnover))
    wf_downside_penalty = 0.30 * max(0.0, -float(wf_min))
    dd_penalty = 0.08 * max(0.0, abs(float(max_dd)) - 0.25)
    return float(
        sharpe
        + 0.35 * wf_median
        + 0.15 * consistency
        - 2.5 * abs(beta)
        - 0.35 * avg_turnover
        - turnover_floor_penalty
        - wf_downside_penalty
        - dd_penalty
    )


def evolution_score(val_sharpe, val_wf_median, val_wf_min, consistency, beta, avg_turnover):
    """Evolution ranking score: optimize held-out-like Sharpe, not headline score inflation."""
    downside_wf = max(0.0, -float(val_wf_min))
    wf_floor_shortfall = max(0.0, EVOLVE_PROGRAM_MIN_VAL_WF_MEDIAN - float(val_wf_median))
    wf_min_shortfall = max(0.0, EVOLVE_PROGRAM_MIN_VAL_WF_MIN - float(val_wf_min))
    turnover_floor_penalty = 5.0 * max(0.0, MIN_ACTIVE_TURNOVER - float(avg_turnover))
    return float(
        1.25 * float(val_sharpe)
        + 0.55 * float(val_wf_median)
        + 0.10 * float(consistency)
        - 0.45 * downside_wf
        - EVOLVE_VAL_WF_FLOOR_PENALTY * wf_floor_shortfall
        - EVOLVE_VAL_WF_FLOOR_PENALTY * wf_min_shortfall
        - 2.5 * abs(float(beta))
        - 0.35 * float(avg_turnover)
        - turnover_floor_penalty
    )


def shortlist_ok(beta, avg_turnover):
    return bool(abs(beta) <= BETA_LIMIT and MIN_ACTIVE_TURNOVER <= avg_turnover <= TURNOVER_LIMIT)


def robust_ok(metric_row, wf_floor=-0.20):
    metric_row = metric_row or {}
    return bool(
        metric_row.get("train_sharpe", -99) > 0
        and metric_row.get("wf_median", -99) > 0
        and metric_row.get("wf_min", -99) > wf_floor
        and shortlist_ok(metric_row.get("beta", 0.0), metric_row.get("turnover", 0.0))
        and metric_row.get("signal_activity", 1.0) >= MIN_SIGNAL_ACTIVITY
        and metric_row.get("raw_cs_std", 1.0) >= MIN_RAW_CS_STD
        and metric_row.get("raw_long_frac", 1.0) >= MIN_LONG_SHORT_FRAC
        and metric_row.get("raw_short_frac", 1.0) >= MIN_LONG_SHORT_FRAC
        and robustness_score(metric_row) >= ROBUSTNESS_SCORE_FLOOR
    )


def wf_summary(windows):
    vals = [w["sharpe"] for w in windows if "sharpe" in w]
    if not vals:
        return 0.0, 0.0
    return float(np.median(vals)), float(min(vals))


def _finish_signal(out):
    return out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)


def baseline_signal_factory(kind, n1, n2=None):
    if kind == "momentum":
        def signal(close, volume, vix=None, tnx=None):
            r = close.pct_change(n1).rank(axis=1, pct=True)
            return _finish_signal((r - 0.5) * 2)
        return signal

    if kind == "mean_reversion":
        def signal(close, volume, vix=None, tnx=None):
            z = -(close / close.rolling(n1).mean() - 1.0)
            r = z.rank(axis=1, pct=True)
            return _finish_signal((r - 0.5) * 2)
        return signal

    if kind == "ewm":
        def signal(close, volume, vix=None, tnx=None):
            fast = close.ewm(span=n1).mean()
            slow = close.ewm(span=n2).mean()
            x = (fast / slow - 1.0).rank(axis=1, pct=True)
            return _finish_signal((x - 0.5) * 2)
        return signal

    if kind == "volume_momentum":
        def signal(close, volume, vix=None, tnx=None):
            mom = close.pct_change(n1)
            vr = volume / volume.rolling(n2).mean()
            x = (mom * vr).rank(axis=1, pct=True)
            return _finish_signal((x - 0.5) * 2)
        return signal

    if kind == "ts_momentum":
        def signal(close, volume, vix=None, tnx=None):
            core = close.pct_change(n1)
            out = (core.rank(axis=1, pct=True) - 0.5) * 2
            return _finish_signal(out)
        return signal

    if kind == "short_reversal":
        def signal(close, volume, vix=None, tnx=None):
            core = -close.pct_change(n1)
            out = (core.rank(axis=1, pct=True) - 0.5) * 2
            return _finish_signal(out)
        return signal

    if kind == "vol_adjusted_momentum":
        def signal(close, volume, vix=None, tnx=None):
            ret = close.pct_change(n1)
            vol = close.pct_change().rolling(n2).std().replace(0, np.nan)
            core = ret.div(vol.clip(lower=1e-6), fill_value=0.0).clip(-3, 3)
            out = (core.rank(axis=1, pct=True) - 0.5) * 2
            return _finish_signal(out)
        return signal

    if kind == "volume_confirmed_momentum":
        def signal(close, volume, vix=None, tnx=None):
            ret = close.pct_change(n1)
            vr = volume / volume.rolling(n2).mean()
            core = ret * vr.clip(lower=0.5, upper=1.8)
            out = (core.rank(axis=1, pct=True) - 0.5) * 2
            return _finish_signal(out)
        return signal

    if kind == "multi_factor":
        def signal(close, volume, vix=None, tnx=None):
            mom = close.pct_change(n1).rank(axis=1, pct=True) - 0.5
            rev = (-close.pct_change(max(5, n1 // 8))).rank(axis=1, pct=True) - 0.5
            vol = close.pct_change().rolling(n2).std().replace(0, np.nan)
            low_vol = (-vol).rank(axis=1, pct=True) - 0.5
            core = 0.55 * mom + 0.25 * rev + 0.20 * low_vol
            out = (core.rank(axis=1, pct=True) - 0.5) * 2
            return _finish_signal(out)
        return signal

    raise ValueError(kind)


def deterministic_signal_factory(short_span, long_span, variant="plain", aux=None):
    aux = aux or {}
    vol_win = int(aux.get("vol_window", 20))
    vol_cap = aux.get("vol_cap", 3.0)
    vol_gate = int(aux.get("vol_gate_window", 20))
    regime_thr = aux.get("vix_threshold", 24.0)

    def signal(close, volume, vix=None, tnx=None):
        fast = close.ewm(span=short_span, min_periods=short_span).mean().shift(1)
        slow = close.ewm(span=long_span, min_periods=long_span).mean().shift(1)
        core = fast / slow - 1.0
        if variant == "plain":
            x = core.rank(axis=1, pct=True)
            out = (x - 0.5) * 2
        elif variant == "rank_norm":
            out = (core.rank(axis=1, pct=True) - 0.5) * 2
        elif variant == "vol_scale":
            vol = close.pct_change().rolling(vol_win).std().replace(0, np.nan)
            scaled = core.div(vol.clip(lower=1e-6), fill_value=0.0).clip(-vol_cap, vol_cap)
            out = (scaled.rank(axis=1, pct=True) - 0.5) * 2
        elif variant == "volume_gate":
            vratio = volume / volume.rolling(vol_gate).mean()
            gated = core * vratio.clip(lower=0.5, upper=1.5)
            out = (gated.rank(axis=1, pct=True) - 0.5) * 2
        elif variant == "regime_gate":
            if vix is None:
                gated = core
            else:
                regime = (vix.rolling(5).mean() < regime_thr).astype(float).values[:, None]
                gated = core * regime
            out = (gated.rank(axis=1, pct=True) - 0.5) * 2
        elif variant == "ts_momentum":
            trend = close.pct_change(long_span)
            out = (trend.rank(axis=1, pct=True) - 0.5) * 2
        elif variant == "short_reversal":
            rev = -close.pct_change(short_span)
            out = (rev.rank(axis=1, pct=True) - 0.5) * 2
        elif variant == "vol_adjusted":
            trend = close.pct_change(long_span)
            vol = close.pct_change().rolling(vol_win).std().replace(0, np.nan)
            scaled = trend.div(vol.clip(lower=1e-6), fill_value=0.0).clip(-vol_cap, vol_cap)
            out = (scaled.rank(axis=1, pct=True) - 0.5) * 2
        elif variant == "volume_confirm":
            trend = close.pct_change(long_span)
            vratio = volume / volume.rolling(vol_gate).mean()
            confirmed = trend * vratio.clip(lower=0.5, upper=1.8)
            out = (confirmed.rank(axis=1, pct=True) - 0.5) * 2
        elif variant == "regime_momentum":
            trend_fast = close.pct_change(short_span)
            trend_slow = close.pct_change(long_span)
            trend = trend_fast - trend_slow
            if vix is not None:
                low_vix = (vix.rolling(5).mean() < regime_thr).astype(float).values[:, None]
                trend = trend * low_vix
            out = (trend.rank(axis=1, pct=True) - 0.5) * 2
        elif variant == "multi_factor":
            trend = close.pct_change(long_span).rank(axis=1, pct=True) - 0.5
            reversal = (-close.pct_change(short_span)).rank(axis=1, pct=True) - 0.5
            vol = close.pct_change().rolling(vol_win).std().replace(0, np.nan)
            low_vol = (-vol).rank(axis=1, pct=True) - 0.5
            score = 0.55 * trend + 0.25 * reversal + 0.20 * low_vol
            out = (score.rank(axis=1, pct=True) - 0.5) * 2
        else:
            raise ValueError(variant)
        return _finish_signal(out)

    return signal


def _normalized_mutation_key(mutation_type):
    key = str(mutation_type or "plain")
    aliases = {
        "volatility_scaling": "vol_scale",
        "vol_scaling": "vol_scale",
        "volume_confirmation": "volume_confirm",
        "regime_filter": "regime_gate",
        "regime_filter_momentum": "regime_momentum",
        "trend_momentum": "ts_momentum",
        "time_series_momentum": "ts_momentum",
        "factor_blend": "multi_factor",
        "multi_factor_blend": "multi_factor",
    }
    return aliases.get(key, key)


def deterministic_family_for_mutation(mutation_type):
    key = _normalized_mutation_key(mutation_type)
    mapping = {
        "plain": ("ewm", "ewm"),
        "span_tweak": ("ewm", "ewm"),
        "rank_norm": ("ewm", "ewm"),
        "rank_normalization": ("ewm", "ewm"),
        "vol_scale": ("ewm_volscale", "ewm"),
        "volume_gate": ("ewm_volume", "ewm"),
        "regime_gate": ("ewm_regime", "ewm"),
        "ts_momentum": ("momentum", "momentum"),
        "short_reversal": ("mean_reversion", "mean_reversion"),
        "vol_adjusted": ("volatility_momentum", "momentum"),
        "volume_confirm": ("volume_momentum", "momentum"),
        "regime_momentum": ("regime_momentum", "momentum"),
        "multi_factor": ("multi_factor", "multi_factor"),
    }
    return mapping.get(key, ("ewm", "ewm"))


def cluster_id_for_signal(base_family, mutation_type, short_span, long_span, params=None):
    params = normalize_aux_params(params)
    key = _normalized_mutation_key(mutation_type)
    family, mapped_base = deterministic_family_for_mutation(key)
    prefix = base_family if key == "span_tweak" and base_family else (family or base_family or mapped_base or "signal")
    if short_span is None or long_span is None:
        return f"{prefix}:generic"
    if key in ("regime_gate", "regime_momentum"):
        thr = params.get("vix_threshold")
        return f"{prefix}:{short_span}:{long_span}:{thr}" if thr is not None else f"{prefix}:{short_span}:{long_span}"
    if key in ("vol_scale", "vol_adjusted", "multi_factor"):
        win = params.get("vol_window")
        return f"{prefix}:{short_span}:{long_span}:{win}" if win is not None else f"{prefix}:{short_span}:{long_span}"
    if key in ("volume_gate", "volume_confirm"):
        win = params.get("vol_gate_window")
        return f"{prefix}:{short_span}:{long_span}:{win}" if win is not None else f"{prefix}:{short_span}:{long_span}"
    return f"{prefix}:{short_span}:{long_span}"


def baseline_walk_forward(fn, close_df, volume_df, vix_s=None, tnx_s=None, flipped=False, n_windows=WF_WINDOWS):
    n = len(close_df)
    sz = max(n // n_windows, 1)
    out = []
    for w in range(n_windows):
        lo = w * sz
        hi = (w + 1) * sz if w < n_windows - 1 else n
        sub_c = close_df.iloc[lo:hi]
        sub_v = volume_df.iloc[lo:hi]
        sub_vix = vix_s.iloc[lo:hi] if vix_s is not None else None
        sub_tnx = tnx_s.iloc[lo:hi] if tnx_s is not None else None
        if len(sub_c) < 40:
            continue
        try:
            sig = fn(sub_c, sub_v, sub_vix, sub_tnx)
            if flipped:
                sig = -sig
            m = backtest(sig, sub_c)
            out.append({"window": w, "sharpe": m["sharpe"], "beta": m["beta"], "ann_return": m["ann_return"], "max_dd": m["max_dd"]})
        except Exception as e:
            out.append({"window": w, "error": str(e)})
    return out


def run_baseline_sweep():
    grid = [
        ("ewm", 30, 100), ("ewm", 32, 110), ("ewm", 34, 115), ("ewm", 36, 120),
        ("ewm", 38, 122), ("ewm", 40, 120), ("ewm", 42, 125), ("ewm", 44, 128),
        ("ewm", 46, 130), ("ewm", 48, 135), ("ewm", 50, 140), ("ewm", 55, 150),
        ("ewm", 40, 160), ("ewm", 20, 80), ("ewm", 20, 100), ("ewm", 30, 120),
        ("volume_momentum", 20, 20), ("volume_momentum", 40, 20), ("volume_momentum", 60, 20),
        ("momentum", 40, None), ("momentum", 80, None), ("mean_reversion", 10, None),
        ("ts_momentum", 63, None), ("ts_momentum", 126, None), ("short_reversal", 5, None),
        ("short_reversal", 10, None), ("vol_adjusted_momentum", 63, 20),
        ("vol_adjusted_momentum", 126, 30), ("volume_confirmed_momentum", 63, 20),
        ("multi_factor", 126, 20),
    ]
    rows = []
    for kind, n1, n2 in grid:
        fn = baseline_signal_factory(kind, n1, n2)
        sig = fn(close_train, volume_train, vix_train, tnx_train)
        m = backtest(sig, close_train)
        flipped = bool(m["prefer_inverted"])
        beta = m["beta_inverted"] if flipped else m["beta"]
        train_sharpe = m["sharpe_inverted"] if flipped else m["sharpe"]
        ann_return = m["ann_return_inverted"] if flipped else m["ann_return"]
        dd = m["max_dd_inverted"] if flipped else m["max_dd"]
        consistency = m["consistency_inverted"] if flipped else m["consistency"]
        wf = baseline_walk_forward(fn, close_train, volume_train, vix_train, tnx_train, flipped=flipped)
        wf_median, wf_min = wf_summary(wf)
        turnover = m["avg_turnover"]
        score = selection_score(train_sharpe, wf_median, consistency, beta, turnover, wf_min=wf_min, max_dd=dd)
        family = "ewm" if kind == "ewm" else kind
        if kind == "ewm":
            meta = signature_for_signal(
                family,
                "span_tweak",
                f"ewm(span={n1}) ewm(span={n2})",
                short_span=n1,
                long_span=n2,
                base_family="ewm",
                params={},
            )
        else:
            meta = signature_for_signal(
                family,
                "span_tweak",
                f"{kind}:{n1}:{n2}",
                base_family=family,
                params={"n1": n1, "n2": n2},
            )
        priority_weight = 3.0 if kind == "ewm" and n1 >= 34 and n1 <= 48 and (n2 or 0) >= 115 and (n2 or 0) <= 135 else (2.0 if kind == "ewm" else 1.0)
        metric_payload = {
            "train_sharpe": train_sharpe,
            "wf_median": wf_median,
            "wf_min": wf_min,
            "beta": beta,
            "turnover": turnover,
            "consistency": consistency,
            "raw_cs_std": m.get("raw_cs_std", 1.0),
            "raw_long_frac": m.get("raw_long_frac", 1.0),
            "raw_short_frac": m.get("raw_short_frac", 1.0),
            "signal_activity": m.get("signal_activity", 1.0),
        }
        rows.append({
            "parent_id": f"base:{kind}:{n1}:{n2 if n2 is not None else '-'}",
            "family": kind,
            "n1": n1,
            "n2": n2,
            "train_sharpe": train_sharpe,
            "raw_sharpe": m["sharpe"],
            "inv_sharpe": m["sharpe_inverted"],
            "flipped": flipped,
            "beta": beta,
            "turnover": turnover,
            "consistency": consistency,
            "ann_return": ann_return,
            "dd": dd,
            "wf_median": wf_median,
            "wf_min": wf_min,
            "score": score,
            "robustness_score": robustness_score(metric_payload),
            "raw_cs_std": m.get("raw_cs_std", 1.0),
            "raw_long_frac": m.get("raw_long_frac", 1.0),
            "raw_short_frac": m.get("raw_short_frac", 1.0),
            "signal_activity": m.get("signal_activity", 1.0),
            "shortlist_ok": shortlist_ok(beta, turnover),
            "priority_weight": priority_weight,
            "robust_ok": robust_ok(metric_payload),
            "component_count": 1,
            "model_size": 1,
            "model_size_key": "components=1",
            **meta,
        })
    rows = sorted(rows, key=lambda r: -(r["score"] + 0.03 * r["priority_weight"]))
    _atomic_write_json(BASELINE_FILE, rows)
    return rows


def deterministic_variant_grid():
    variants = [("plain", {}), ("rank_norm", {})]
    variants += [("vol_scale", {"vol_window": w}) for w in (20, 30)]
    variants += [("volume_gate", {"vol_gate_window": w}) for w in (20, 40)]
    variants += [("regime_gate", {"vix_threshold": thr}) for thr in (20.0, 24.0, 28.0)]
    variants += [("ts_momentum", {}), ("short_reversal", {})]
    variants += [("vol_adjusted", {"vol_window": w}) for w in (20, 30, 40)]
    variants += [("volume_confirm", {"vol_gate_window": w}) for w in (20, 40, 60)]
    variants += [("regime_momentum", {"vix_threshold": thr}) for thr in (18.0, 22.0, 26.0)]
    variants += [("multi_factor", {"vol_window": 20}), ("multi_factor", {"vol_window": 40})]
    return variants


def parameter_search_variant_space():
    variants = [("plain", {}), ("rank_norm", {})]
    variants += [("vol_scale", {"vol_window": w}) for w in PARAM_SEARCH_VOL_WINDOWS]
    variants += [("volume_gate", {"vol_gate_window": w}) for w in PARAM_SEARCH_VOL_GATE_WINDOWS]
    variants += [("regime_gate", {"vix_threshold": thr}) for thr in PARAM_SEARCH_REGIME_THRESHOLDS]
    variants += [("ts_momentum", {}), ("short_reversal", {})]
    variants += [("vol_adjusted", {"vol_window": w}) for w in PARAM_SEARCH_VOL_WINDOWS]
    variants += [("volume_confirm", {"vol_gate_window": w}) for w in PARAM_SEARCH_VOL_GATE_WINDOWS]
    variants += [("regime_momentum", {"vix_threshold": thr}) for thr in PARAM_SEARCH_REGIME_THRESHOLDS]
    variants += [("multi_factor", {"vol_window": w}) for w in PARAM_SEARCH_VOL_WINDOWS]
    return variants


def sample_parameter_trials(n_trials=PARAM_SEARCH_TRIALS, seed=PARAM_SEARCH_SEED):
    rng = random.Random(seed)
    variants = parameter_search_variant_space()
    trials = []
    seen = set()
    max_attempts = max(n_trials * 20, 2000)
    attempts = 0
    while len(trials) < n_trials and attempts < max_attempts:
        attempts += 1
        short_span = rng.choice(PARAM_SEARCH_SHORT_SPANS)
        long_span = rng.choice(PARAM_SEARCH_LONG_SPANS)
        if long_span < short_span + 30:
            continue
        variant, aux = rng.choice(variants)
        aux_norm = normalize_aux_params(aux)
        trial_key = (variant, short_span, long_span, tuple(sorted(aux_norm.items())))
        if trial_key in seen:
            continue
        seen.add(trial_key)
        trials.append(
            {
                "mutation_type": variant,
                "short_span": short_span,
                "long_span": long_span,
                "params": aux_norm,
            }
        )
    return trials


def evaluate_structured_trial(trial, source="parameter_search"):
    short_span = trial["short_span"]
    long_span = trial["long_span"]
    variant = trial["mutation_type"]
    aux_norm = normalize_aux_params(trial.get("params", {}))
    fn = deterministic_signal_factory(short_span, long_span, variant=variant, aux=aux_norm)
    m = None
    sig = fn(close_train, volume_train, vix_train, tnx_train)
    m = backtest(sig, close_train)
    flipped = bool(m["prefer_inverted"])
    beta = m["beta_inverted"] if flipped else m["beta"]
    train_sharpe = m["sharpe_inverted"] if flipped else m["sharpe"]
    ann_return = m["ann_return_inverted"] if flipped else m["ann_return"]
    dd = m["max_dd_inverted"] if flipped else m["max_dd"]
    consistency = m["consistency_inverted"] if flipped else m["consistency"]
    wf = baseline_walk_forward(fn, close_train, volume_train, vix_train, tnx_train, flipped=flipped)
    wf_median, wf_min = wf_summary(wf)
    turnover = m["avg_turnover"]
    score = selection_score(train_sharpe, wf_median, consistency, beta, turnover, wf_min=wf_min, max_dd=dd)
    family, base_family = deterministic_family_for_mutation(variant)
    code_label = f"ewm(span={short_span}) ewm(span={long_span}) {variant} {aux_norm}"
    meta = signature_for_signal(
        family,
        variant,
        code_label,
        short_span=short_span,
        long_span=long_span,
        base_family=base_family,
        params=aux_norm,
    )
    aux_tag = ",".join(f"{k}={v}" for k, v in aux_norm.items()) if aux_norm else "base"
    metric_payload = {
        "train_sharpe": train_sharpe,
        "wf_median": wf_median,
        "wf_min": wf_min,
        "beta": beta,
        "turnover": turnover,
        "consistency": consistency,
        "raw_cs_std": m.get("raw_cs_std", 1.0),
        "raw_long_frac": m.get("raw_long_frac", 1.0),
        "raw_short_frac": m.get("raw_short_frac", 1.0),
        "signal_activity": m.get("signal_activity", 1.0),
    }
    return {
        "source": source,
        "parent_id": f"search:{variant}:{short_span}:{long_span}:{aux_tag}",
        "family": family,
        "base_family": base_family,
        "mutation_type": variant,
        "short_span": short_span,
        "long_span": long_span,
        "params": aux_norm,
        "train_sharpe": train_sharpe,
        "ann_return": ann_return,
        "beta": beta,
        "turnover": turnover,
        "max_dd": dd,
        "consistency": consistency,
        "wf_median": wf_median,
        "wf_min": wf_min,
        "score": score,
        "robustness_score": robustness_score(metric_payload),
        "raw_cs_std": m.get("raw_cs_std", 1.0),
        "raw_long_frac": m.get("raw_long_frac", 1.0),
        "raw_short_frac": m.get("raw_short_frac", 1.0),
        "signal_activity": m.get("signal_activity", 1.0),
        "flipped": flipped,
        "shortlist_ok": shortlist_ok(beta, turnover),
        "robust_ok": robust_ok(metric_payload),
        "component_count": 1,
        "model_size": 1,
        "model_size_key": "components=1",
        **meta,
    }


def _trial_key(trial):
    aux = normalize_aux_params(trial.get("params", {}))
    return (
        trial.get("mutation_type"),
        int(trial.get("short_span")),
        int(trial.get("long_span")),
        tuple(sorted(aux.items())),
    )


def _row_to_trial(row):
    return {
        "mutation_type": row["mutation_type"],
        "short_span": int(row["short_span"]),
        "long_span": int(row["long_span"]),
        "params": normalize_aux_params(row.get("params", {})),
    }


def _safe_load_rows_from_file(path_obj):
    try:
        if path_obj is not None and path_obj.exists():
            payload = json.loads(path_obj.read_text())
            if isinstance(payload, list):
                return payload
    except Exception:
        return []
    return []


def _safe_load_search_results():
    if "load_search_results" in globals():
        try:
            rows = load_search_results()
            if isinstance(rows, list):
                return rows
        except Exception:
            _warn("load_search_results() failed; using file-based fallback")
    rows = []
    if "load_parameter_search" in globals():
        try:
            rows.extend([r for r in load_parameter_search() if isinstance(r, dict)])
        except Exception:
            pass
    elif "PARAM_SEARCH_FILE" in globals():
        rows.extend(_safe_load_rows_from_file(PARAM_SEARCH_FILE))
    if "load_deterministic" in globals():
        try:
            rows.extend([r for r in load_deterministic() if isinstance(r, dict)])
        except Exception:
            pass
    elif "DETERMINISTIC_FILE" in globals():
        rows.extend(_safe_load_rows_from_file(DETERMINISTIC_FILE))
    return rows


def _safe_load_log_rows():
    if "load_log" in globals():
        try:
            rows = load_log()
            if isinstance(rows, list):
                return [r for r in rows if isinstance(r, dict)]
        except Exception:
            pass
    if "RESEARCH_LOG" in globals() and RESEARCH_LOG.exists():
        out = []
        for line in RESEARCH_LOG.read_text().splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict):
                    out.append(obj)
            except Exception:
                pass
        return out
    return []


def _safe_load_memo_text():
    if "load_memo" in globals():
        try:
            txt = load_memo()
            if isinstance(txt, str):
                return txt
        except Exception:
            pass
    if "MEMO_FILE" in globals() and MEMO_FILE.exists():
        try:
            return MEMO_FILE.read_text()
        except Exception:
            pass
    return "(No memo yet - deterministic-first mode.)"


def _entry_variant_tags(entry):
    tags = []
    allowed = {v for v, _ in parameter_search_variant_space()}

    def add(tag):
        if tag in allowed and tag not in tags:
            tags.append(tag)

    if not isinstance(entry, dict):
        return tags

    add(entry.get("mutation_type"))

    program = entry.get("program")
    if isinstance(program, dict):
        for comp in program.get("components", []):
            if isinstance(comp, dict):
                add(comp.get("mutation_type"))

    for comp in entry.get("program_components", []):
        if isinstance(comp, dict):
            add(comp.get("mutation_type"))

    text = " ".join(
        str(entry.get(key, ""))
        for key in ("cluster_id", "signature", "parent_id", "code", "hypothesis")
    ).lower()
    for variant in allowed:
        if variant in text:
            add(variant)
    return tags


def _variant_failure_bias():
    logs = _safe_load_log_rows()
    if not logs:
        return {}
    last = logs[-300:]
    counts = {v: 0.0 for v, _ in parameter_search_variant_space()}
    for e in last:
        err = str(e.get("error", "")).lower()
        if not err:
            continue
        weight = 1.0
        if "dead after market-neutral normalization" in err:
            weight = 3.0
        elif "not genuinely long-short" in err:
            weight = 3.0
        elif "beta" in err or "wf_min" in err or "robustness" in err:
            weight = 1.5

        tags = _entry_variant_tags(e)
        if not tags:
            code = str(e.get("code", "")).lower()
            hyp = str(e.get("hypothesis", "")).lower()
            text = code + " " + hyp
            if "vix" in text or "regime" in text:
                tags.extend(["regime_gate", "regime_momentum"])
            if "volume" in text:
                tags.extend(["volume_gate", "volume_confirm"])
            if "std" in text or "vol" in text:
                tags.extend(["vol_scale", "vol_adjusted"])
            if "rank" in text:
                tags.append("rank_norm")
            if "reversal" in text or "contrarian" in text:
                tags.append("short_reversal")
            if "multi" in text or "blend" in text or "composite" in text:
                tags.append("multi_factor")
            if "momentum" in text or "trend" in text:
                tags.append("ts_momentum")
            if not any(k in text for k in ("vix", "regime", "volume", "std", "vol", "rank")):
                tags.append("plain")

        for tag in set(tags):
            counts[tag] = counts.get(tag, 0.0) + weight
    total = sum(counts.values()) or 1
    return {k: counts[k] / total for k in counts}


def _summarize_generation_reflection(gen_id, gen_rows, survivors):
    valid = [r for r in gen_rows if r.get("score") is not None]
    robust = [r for r in valid if r.get("robust_ok")]
    fails = [r for r in gen_rows if r.get("error")]
    top = sorted(robust if robust else valid, key=lambda r: -r.get("score", -999))[:5]
    lines = [f"Generation {gen_id} reflection"]
    lines.append(f"- valid={len(valid)} robust={len(robust)} fails={len(fails)}")
    if top:
        lines.append("- winners:")
        for r in top:
            lines.append(
                f"  - {r.get('cluster_id')} score={r.get('score', 0.0):+.2f} "
                f"Sh={r.get('train_sharpe', 0.0):+.2f} wf={r.get('wf_median', 0.0):+.2f}/{r.get('wf_min', 0.0):+.2f}"
            )
    if fails:
        recent_fail = fails[-5:]
        lines.append("- recent failures:")
        for f in recent_fail:
            lines.append(f"  - {str(f.get('error'))[:140]}")
    lines.append("- survivor signatures:")
    for s in survivors:
        lines.append(f"  - {s.get('signature')}")
    return "\n".join(lines)


def _bootstrap_ci(values, n_boot=300, alpha=0.10, seed=42):
    vals = [float(v) for v in values if v is not None and np.isfinite(v)]
    if not vals:
        return (None, None)
    if len(vals) == 1:
        return (vals[0], vals[0])
    rng = random.Random(seed)
    means = []
    for _ in range(n_boot):
        sample = [vals[rng.randrange(0, len(vals))] for _ in range(len(vals))]
        means.append(float(np.mean(sample)))
    means.sort()
    lo_idx = int((alpha / 2) * len(means))
    hi_idx = int((1 - alpha / 2) * len(means)) - 1
    lo_idx = max(0, min(lo_idx, len(means) - 1))
    hi_idx = max(0, min(hi_idx, len(means) - 1))
    return (means[lo_idx], means[hi_idx])


def _build_generation_program(gen_id, parents, recent_rows):
    fail_bias = _variant_failure_bias()
    memo = _safe_load_memo_text().lower()
    valid = [r for r in recent_rows if r.get("score") is not None]
    robust = [r for r in valid if r.get("robust_ok")]
    ranked = sorted(robust if robust else valid, key=lambda r: -_heldout_proxy_score(r))
    top = ranked[:8]
    top_clusters = [r.get("cluster_id") for r in top if r.get("cluster_id")]
    dominant_cluster_frac = 0.0
    if top_clusters:
        dominant_cluster_frac = max(top_clusters.count(cid) for cid in set(top_clusters)) / max(len(top_clusters), 1)

    if top:
        short_center = int(round(float(np.median([r.get("short_span", 54) for r in top]))))
        long_center = int(round(float(np.median([r.get("long_span", 90) for r in top]))))
    elif parents:
        short_center = int(round(float(np.median([p.get("short_span", 54) for p in parents]))))
        long_center = int(round(float(np.median([p.get("long_span", 90) for p in parents]))))
    else:
        short_center, long_center = 54, 90

    short_deltas = [-9, -6, -3, 0, 3, 6, 9]
    long_deltas = [-18, -12, -6, 0, 6, 12, 18]
    if "faster" in memo or "short horizon" in memo:
        short_deltas = [-6, -3, 0, 3, 6]
    if "slower" in memo or "long horizon" in memo:
        long_deltas = [-24, -18, -12, -6, 0, 6, 12]

    variant_space = [v for v, _ in parameter_search_variant_space()]
    if fail_bias:
        variant_priority = sorted(variant_space, key=lambda v: fail_bias.get(v, 0.0))
    else:
        variant_priority = list(variant_space)

    avoid_variants = [v for v, b in fail_bias.items() if b > 0.22]
    focus_variants = [v for v in variant_priority if v not in avoid_variants][:3]
    fallback_focus = ["regime_momentum", "volume_confirm", "volume_gate", "rank_norm", "plain", "regime_gate", "ts_momentum"]
    if dominant_cluster_frac >= 0.50:
        fallback_focus = ["volume_confirm", "volume_gate", "rank_norm", "plain", "regime_momentum", "regime_gate", "ts_momentum"]
    for variant in fallback_focus:
        if variant in variant_space and variant not in focus_variants and variant not in avoid_variants:
            focus_variants.append(variant)
        if len(focus_variants) >= EVOLVE_MIN_FOCUS_VARIANTS:
            break
    if not focus_variants:
        focus_variants = ["regime_momentum", "volume_confirm", "volume_gate", "rank_norm", "regime_gate"]

    program = {
        "gen_id": gen_id,
        "short_center": short_center,
        "long_center": long_center,
        "short_deltas": short_deltas,
        "long_deltas": long_deltas,
        "focus_variants": focus_variants,
        "avoid_variants": avoid_variants,
        "fail_bias": fail_bias,
        "dominant_cluster_frac": float(dominant_cluster_frac),
        "notes": "reflection-driven mutation program",
    }
    return program


def _program_variant_mix(base_variant, program):
    focus = list(program.get("focus_variants", []))
    avoid = set(program.get("avoid_variants", []))
    all_variants = [v for v, _ in parameter_search_variant_space()]
    ordered = [base_variant] + focus + [v for v in all_variants if v != base_variant and v not in focus]
    out = []
    for v in ordered:
        if v in avoid and v != base_variant:
            continue
        if v not in out:
            out.append(v)
    return out[:4]


def _heldout_proxy_score(row):
    base = float(row.get("score", -999.0))
    wf_median = float(row.get("wf_median", 0.0))
    wf_min = float(row.get("wf_min", 0.0))
    consistency = float(row.get("consistency", 0.0))
    spread = max(0.0, wf_median - wf_min)
    neg_wfmin = max(0.0, -wf_min)
    low_consistency = max(0.0, 0.5 - consistency)
    proxy = (
        base
        - EVOLVE_PROXY_WF_SPREAD_PENALTY * spread
        - EVOLVE_PROXY_WFMIN_PENALTY * neg_wfmin
        - EVOLVE_PROXY_CONSISTENCY_PENALTY * low_consistency
    )
    return float(proxy)


def _top_parents_from_rows(rows, k):
    valid = [r for r in rows if r.get("score") is not None]
    if not valid:
        return []
    ranked = sorted(valid, key=lambda r: -_heldout_proxy_score(r))
    out = []
    seen_sig = set()
    for r in ranked:
        sig = r.get("signature")
        if sig and sig in seen_sig:
            continue
        if sig:
            seen_sig.add(sig)
        out.append(r)
        if len(out) >= k:
            break
    return out


def _diverse_top_parents_from_rows(rows, k):
    valid = [r for r in rows if r.get("score") is not None]
    if not valid:
        return []
    ranked = sorted(valid, key=lambda r: -_heldout_proxy_score(r))
    out = []
    seen_sig = set()

    def add(row):
        sig = row.get("signature") or row_identity(row, "parent")
        if sig in seen_sig:
            return False
        seen_sig.add(sig)
        out.append(row)
        return True

    for variant in EVOLVE_CHAMPION_VARIANTS:
        family_rows = [r for r in ranked if r.get("mutation_type") == variant]
        if family_rows:
            add(family_rows[0])
        if len(out) >= k:
            return out[:k]
    for row in ranked:
        if len(out) >= k:
            break
        add(row)
    return out[:k]


def _program_variants(program):
    comps = program.get("components", []) if isinstance(program, dict) else []
    return [str(_sanitize_component(c).get("mutation_type", EVOLVE_BENCHMARK_MUTATION)) for c in comps]


def _program_primary_family(program):
    variants = _program_variants(program)
    if len(variants) >= 2:
        return "composite"
    return variants[0] if variants else EVOLVE_BENCHMARK_MUTATION


def _row_family(row):
    if not isinstance(row, dict):
        return "unknown"
    if isinstance(row.get("program"), dict):
        return _program_primary_family(row["program"])
    return str(row.get("candidate_family") or row.get("mutation_type") or row.get("family") or "unknown")


def _family_capped_rows(rows, limit, cap_frac=EVOLVE_PARENT_FAMILY_CAP):
    rows = [r for r in rows or [] if isinstance(r, dict)]
    if not rows or limit <= 0:
        return []
    cap = max(1, int(np.ceil(float(limit) * float(cap_frac))))
    families = { _row_family(r) for r in rows }
    if len(families) <= 1:
        return rows[:limit]
    selected = []
    family_counts = {}
    deferred = []
    for row in rows:
        fam = _row_family(row)
        if family_counts.get(fam, 0) >= cap:
            deferred.append(row)
            continue
        selected.append(row)
        family_counts[fam] = family_counts.get(fam, 0) + 1
        if len(selected) >= limit:
            return selected
    for row in deferred:
        if len(selected) >= limit:
            break
        selected.append(row)
    return selected[:limit]


def _aux_candidates_for_variant(variant, base_params):
    base = normalize_aux_params(base_params)
    if variant == "regime_gate":
        vals = sorted(set(PARAM_SEARCH_REGIME_THRESHOLDS + [float(base.get("vix_threshold", 20.0))]))
        return [{"vix_threshold": float(v)} for v in vals]
    if variant == "vol_scale":
        vals = sorted(set(PARAM_SEARCH_VOL_WINDOWS + [int(base.get("vol_window", 20))]))
        return [{"vol_window": int(v)} for v in vals]
    if variant == "vol_adjusted":
        vals = sorted(set(PARAM_SEARCH_VOL_WINDOWS + [int(base.get("vol_window", 20))]))
        return [{"vol_window": int(v)} for v in vals]
    if variant == "multi_factor":
        vals = sorted(set(PARAM_SEARCH_VOL_WINDOWS + [int(base.get("vol_window", 20))]))
        return [{"vol_window": int(v)} for v in vals]
    if variant == "volume_gate":
        vals = sorted(set(PARAM_SEARCH_VOL_GATE_WINDOWS + [int(base.get("vol_gate_window", 20))]))
        return [{"vol_gate_window": int(v)} for v in vals]
    if variant == "volume_confirm":
        vals = sorted(set(PARAM_SEARCH_VOL_GATE_WINDOWS + [int(base.get("vol_gate_window", 20))]))
        return [{"vol_gate_window": int(v)} for v in vals]
    if variant == "regime_momentum":
        vals = sorted(set(PARAM_SEARCH_REGIME_THRESHOLDS + [float(base.get("vix_threshold", 20.0))]))
        return [{"vix_threshold": float(v)} for v in vals]
    return [{}]


def _allocate_parent_quotas(parents, total_trials):
    n = len(parents)
    if n == 0:
        return []
    if n == 1:
        return [max(total_trials, 0)]
    min_q = EVOLVE_PARENT_MIN_TRIALS
    max_q = EVOLVE_PARENT_MAX_TRIALS
    if total_trials <= n * min_q:
        base = total_trials // n
        rem = total_trials - base * n
        return [base + (1 if i < rem else 0) for i in range(n)]
    quotas = [min_q] * n
    extra = total_trials - n * min_q
    ranks = list(range(n, 0, -1))
    sw = sum(ranks)
    fractional = []
    for i, w in enumerate(ranks):
        add_f = extra * (w / sw)
        add_i = int(add_f)
        quotas[i] += add_i
        fractional.append((add_f - add_i, i))
    rem = total_trials - sum(quotas)
    for _, i in sorted(fractional, reverse=True):
        if rem <= 0:
            break
        quotas[i] += 1
        rem -= 1
    overflow = 0
    for i in range(n):
        if quotas[i] > max_q:
            overflow += quotas[i] - max_q
            quotas[i] = max_q
    if overflow > 0:
        for i in range(n):
            room = max_q - quotas[i]
            if room <= 0:
                continue
            take = min(room, overflow)
            quotas[i] += take
            overflow -= take
            if overflow <= 0:
                break
    return quotas


def _build_branch_trials(parent, quota, gen_id, branch_id, rng, program):
    base_trial = _row_to_trial(parent)
    base_short = int(base_trial["short_span"])
    base_long = int(base_trial["long_span"])
    variant = base_trial["mutation_type"]
    variant_mix = _program_variant_mix(variant, program)
    short_deltas = list(program.get("short_deltas", [-9, -6, -3, 0, 3, 6, 9]))
    long_deltas = list(program.get("long_deltas", [-18, -12, -6, 0, 6, 12, 18]))
    short_center = int(program.get("short_center", base_short))
    long_center = int(program.get("long_center", base_long))
    trials = []
    seen = set()
    for d_short in short_deltas:
        for d_long in long_deltas:
            blended_short = int(round((base_short + short_center) / 2.0))
            blended_long = int(round((base_long + long_center) / 2.0))
            short_span = min(max(blended_short + d_short, min(PARAM_SEARCH_SHORT_SPANS)), max(PARAM_SEARCH_SHORT_SPANS))
            long_span = min(max(blended_long + d_long, min(PARAM_SEARCH_LONG_SPANS)), max(PARAM_SEARCH_LONG_SPANS))
            if long_span < short_span + 30:
                continue
            for v in variant_mix[:3]:
                aux_variants = _aux_candidates_for_variant(v, base_trial["params"])
                rng.shuffle(aux_variants)
                for aux in aux_variants[:2]:
                    trial = {
                        "mutation_type": v,
                        "short_span": short_span,
                        "long_span": long_span,
                        "params": normalize_aux_params(aux),
                        "origin": "branch_local",
                        "gen_id": gen_id,
                        "branch_id": branch_id,
                        "parent_id": row_identity(parent, "parent"),
                        "parent_signature": parent.get("signature"),
                        "parent_score": parent.get("score"),
                    }
                    k = _trial_key(trial)
                    if k in seen:
                        continue
                    seen.add(k)
                    trials.append(trial)
                    if len(trials) >= quota:
                        return trials
    return trials[:quota]


def _seed_parents_for_generation():
    search_rows = [r for r in _safe_load_search_results() if r.get("score") is not None]
    robust = [r for r in search_rows if r.get("robust_ok")]
    return _diverse_top_parents_from_rows(robust if robust else search_rows, EVOLVE_BEAM_WIDTH)


def _run_generation_trials(gen_id, parents, remaining_cap, seen_trial_keys, program):
    rng = random.Random(PARAM_SEARCH_SEED + 191 * gen_id)
    target_trials = min(EVOLVE_TRIALS_PER_GENERATION, remaining_cap)
    random_budget = int(target_trials * EVOLVE_RANDOM_EXPLORATION_FRAC)
    branch_budget = max(target_trials - random_budget, 0)
    parents_ranked = sorted(parents, key=lambda r: -r.get("score", -999))
    quotas = _allocate_parent_quotas(parents_ranked, branch_budget)
    planned = []
    for i, (parent, quota) in enumerate(zip(parents_ranked, quotas)):
        branch_trials = _build_branch_trials(parent, quota, gen_id, i, rng, program)
        for t in branch_trials:
            k = _trial_key(t)
            if k in seen_trial_keys:
                continue
            seen_trial_keys.add(k)
            t["allocation_quota"] = quota
            planned.append(t)
    if len(planned) < target_trials:
        filler_n = target_trials - len(planned)
        fillers = sample_parameter_trials(n_trials=filler_n * 3, seed=PARAM_SEARCH_SEED + 9973 * gen_id)
        for t in fillers:
            ft = {
                **t,
                "origin": "random_fill",
                "gen_id": gen_id,
                "branch_id": -1,
                "parent_id": f"random_fill:g{gen_id}",
                "parent_signature": None,
                "parent_score": None,
                "allocation_quota": 0,
            }
            k = _trial_key(ft)
            if k in seen_trial_keys:
                continue
            seen_trial_keys.add(k)
            planned.append(ft)
            if len(planned) >= target_trials:
                break

    rows = []
    for trial in planned:
        try:
            row = evaluate_structured_trial(trial, source="parameter_search_evolution")
            row["gen_id"] = gen_id
            row["branch_id"] = trial.get("branch_id")
            row["origin"] = trial.get("origin", "branch_local")
            row["parent_id"] = trial.get("parent_id")
            row["parent_signature"] = trial.get("parent_signature")
            row["allocation_quota"] = trial.get("allocation_quota", 0)
            parent_score = trial.get("parent_score")
            row["score_gain_vs_parent"] = float(row["score"] - parent_score) if parent_score is not None else None
        except Exception as e:
            row = {
                "source": "parameter_search_evolution",
                "gen_id": gen_id,
                "branch_id": trial.get("branch_id"),
                "origin": trial.get("origin", "branch_local"),
                "parent_id": trial.get("parent_id"),
                "parent_signature": trial.get("parent_signature"),
                "allocation_quota": trial.get("allocation_quota", 0),
                "mutation_type": trial["mutation_type"],
                "short_span": trial["short_span"],
                "long_span": trial["long_span"],
                "params": normalize_aux_params(trial.get("params", {})),
                "error": str(e),
            }
        rows.append(row)
    return rows, planned


def _sanitize_component(comp):
    c = dict(comp)
    c["short_span"] = int(c.get("short_span", 54))
    c["long_span"] = int(c.get("long_span", 90))
    if c["long_span"] < c["short_span"] + 30:
        c["long_span"] = c["short_span"] + 30
    c["mutation_type"] = c.get("mutation_type", EVOLVE_BENCHMARK_MUTATION)
    params = normalize_aux_params(c.get("params", {}))
    if c["mutation_type"] in ("vol_scale", "vol_adjusted", "multi_factor"):
        params["vol_window"] = int(params.get("vol_window", 20))
    if c["mutation_type"] in ("volume_gate", "volume_confirm"):
        params["vol_gate_window"] = int(params.get("vol_gate_window", 20))
    if c["mutation_type"] in ("regime_gate", "regime_momentum"):
        params["vix_threshold"] = float(params.get("vix_threshold", 20.0))
    c["params"] = params
    c["weight"] = float(c.get("weight", 1.0))
    return c


def _program_identity(program):
    comps = [_sanitize_component(c) for c in program.get("components", [])]
    if not comps:
        comps = [_sanitize_component({"mutation_type": EVOLVE_BENCHMARK_MUTATION, "short_span": 54, "long_span": 90, "params": {"vix_threshold": 26.0}, "weight": 1.0})]
    payload = [
        {
            "mutation_type": c["mutation_type"],
            "short_span": int(c["short_span"]),
            "long_span": int(c["long_span"]),
            "params": normalize_aux_params(c.get("params", {})),
            "weight": round(float(c.get("weight", 1.0)), 6),
        }
        for c in comps
    ]
    fingerprint = hashlib.sha1(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()).hexdigest()[:16]
    variants = "+".join(sorted({c["mutation_type"] for c in comps}))
    short_bucket = int(round(float(np.median([c["short_span"] for c in comps])) / 3.0) * 3)
    long_bucket = int(round(float(np.median([c["long_span"] for c in comps])) / 3.0) * 3)
    cluster_id = f"program:{variants}:{short_bucket}:{long_bucket}"
    return payload, fingerprint, cluster_id


def _program_code(program):
    comps = [_sanitize_component(c) for c in program.get("components", [])]
    if not comps:
        comps = [_sanitize_component({"mutation_type": EVOLVE_BENCHMARK_MUTATION, "short_span": 54, "long_span": 90, "params": {"vix_threshold": 26.0}, "weight": 1.0})]
    lines = ["import numpy as np", "def signal(close, volume, vix=None, tnx=None):", "    parts = []", "    weights = []"]
    for i, c in enumerate(comps):
        ss = int(c["short_span"])
        ls = int(c["long_span"])
        var = c["mutation_type"]
        w = float(c.get("weight", 1.0))
        p = c.get("params", {})
        lines.append(f"    # comp_{i}: {var} {ss}/{ls}")
        lines.append(f"    fast_{i} = close.ewm(span={ss}, min_periods={ss}).mean().shift(1)")
        lines.append(f"    slow_{i} = close.ewm(span={ls}, min_periods={ls}).mean().shift(1)")
        lines.append(f"    core_{i} = fast_{i} / slow_{i} - 1.0")
        if var == "vol_scale":
            vw = int(p.get("vol_window", 20))
            lines.append(f"    vol_{i} = close.pct_change().rolling({vw}).std().replace(0, np.nan)")
            lines.append(f"    sig_{i} = (core_{i}.div(vol_{i}.clip(lower=1e-6), fill_value=0.0).clip(-3,3).rank(axis=1,pct=True)-0.5)*2")
        elif var == "vol_adjusted":
            vw = int(p.get("vol_window", 20))
            lines.append(f"    ret_{i} = close.pct_change({ls})")
            lines.append(f"    vol_{i} = close.pct_change().rolling({vw}).std().replace(0, np.nan)")
            lines.append(f"    sig_{i} = (ret_{i}.div(vol_{i}.clip(lower=1e-6), fill_value=0.0).clip(-3,3).rank(axis=1,pct=True)-0.5)*2")
        elif var == "volume_gate":
            gw = int(p.get("vol_gate_window", 20))
            lines.append(f"    vr_{i} = volume / volume.rolling({gw}).mean()")
            lines.append(f"    sig_{i} = ((core_{i} * vr_{i}.clip(lower=0.5, upper=1.5)).rank(axis=1,pct=True)-0.5)*2")
        elif var == "volume_confirm":
            gw = int(p.get("vol_gate_window", 20))
            lines.append(f"    ret_{i} = close.pct_change({ls})")
            lines.append(f"    vr_{i} = volume / volume.rolling({gw}).mean()")
            lines.append(f"    sig_{i} = ((ret_{i} * vr_{i}.clip(lower=0.5, upper=1.8)).rank(axis=1,pct=True)-0.5)*2")
        elif var == "regime_gate":
            thr = float(p.get("vix_threshold", 20.0))
            lines.append(f"    reg_{i} = 1.0 if vix is None else (vix.rolling(5).mean() < {thr}).astype(float).values[:, None]")
            lines.append(f"    sig_{i} = ((core_{i} * reg_{i}).rank(axis=1,pct=True)-0.5)*2")
        elif var == "regime_momentum":
            thr = float(p.get("vix_threshold", 20.0))
            lines.append(f"    ret_fast_{i} = close.pct_change({ss})")
            lines.append(f"    ret_slow_{i} = close.pct_change({ls})")
            lines.append(f"    reg_{i} = 1.0 if vix is None else (vix.rolling(5).mean() < {thr}).astype(float).values[:, None]")
            lines.append(f"    sig_{i} = (((ret_fast_{i} - ret_slow_{i}) * reg_{i}).rank(axis=1,pct=True)-0.5)*2")
        elif var == "ts_momentum":
            lines.append(f"    ret_{i} = close.pct_change({ls})")
            lines.append(f"    sig_{i} = (ret_{i}.rank(axis=1,pct=True)-0.5)*2")
        elif var == "short_reversal":
            lines.append(f"    rev_{i} = -close.pct_change({ss})")
            lines.append(f"    sig_{i} = (rev_{i}.rank(axis=1,pct=True)-0.5)*2")
        elif var == "multi_factor":
            vw = int(p.get("vol_window", 20))
            lines.append(f"    mom_{i} = close.pct_change({ls}).rank(axis=1,pct=True)-0.5")
            lines.append(f"    rev_{i} = (-close.pct_change({ss})).rank(axis=1,pct=True)-0.5")
            lines.append(f"    vol_{i} = close.pct_change().rolling({vw}).std().replace(0, np.nan)")
            lines.append(f"    lowvol_{i} = (-vol_{i}).rank(axis=1,pct=True)-0.5")
            lines.append(f"    combo_{i} = 0.55*mom_{i} + 0.25*rev_{i} + 0.20*lowvol_{i}")
            lines.append(f"    sig_{i} = (combo_{i}.rank(axis=1,pct=True)-0.5)*2")
        else:
            lines.append(f"    sig_{i} = (core_{i}.rank(axis=1,pct=True)-0.5)*2")
        lines.append(f"    parts.append(sig_{i}); weights.append({w})")
    lines += [
        "    wsum = sum(weights) if len(weights) else 1.0",
        "    out = sum(p * w for p, w in zip(parts, weights)) / wsum",
        "    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)",
        "    return out",
    ]
    return "\n".join(lines)


def _row_to_program(row):
    comp = {
        "mutation_type": row.get("mutation_type", EVOLVE_BENCHMARK_MUTATION),
        "short_span": int(row.get("short_span", 54)),
        "long_span": int(row.get("long_span", 90)),
        "params": normalize_aux_params(row.get("params", {})),
        "weight": 1.0,
    }
    return {"components": [comp]}


def _mutate_program(parent_program, gen_program, rng):
    program = {"components": [dict(c) for c in parent_program.get("components", [])]}
    if not program["components"]:
        program["components"] = [{"mutation_type": "regime_momentum", "short_span": 54, "long_span": 90, "params": {"vix_threshold": 26.0}, "weight": 1.0}]
    op_choices = ["tweak", "tweak", "tweak", "swap_variant", "swap_variant", "reweight", "add"]
    if len(program["components"]) > 1:
        op_choices.append("drop")
    op = rng.choice(op_choices)
    focus = list(gen_program.get("focus_variants", ["regime_momentum", "volume_confirm", "rank_norm"]))
    avoid = set(gen_program.get("avoid_variants", []))
    idx = rng.randrange(0, len(program["components"]))
    if op == "tweak":
        c = dict(program["components"][idx])
        c["short_span"] = int(c.get("short_span", 54)) + rng.choice([-6, -3, 3, 6])
        c["long_span"] = int(c.get("long_span", 90)) + rng.choice([-12, -6, 6, 12])
        program["components"][idx] = _sanitize_component(c)
    elif op == "add" and len(program["components"]) < 4:
        base = dict(program["components"][idx])
        base["mutation_type"] = rng.choice([v for v in focus if v not in avoid] or ["regime_momentum"])
        base["short_span"] = int(base.get("short_span", 54)) + rng.choice([-9, -6, 6, 9])
        base["long_span"] = int(base.get("long_span", 90)) + rng.choice([-18, -12, 12, 18])
        base["weight"] = rng.uniform(0.6, 1.4)
        program["components"].append(_sanitize_component(base))
    elif op == "drop" and len(program["components"]) > 1:
        del program["components"][idx]
    elif op == "reweight":
        for i in range(len(program["components"])):
            c = dict(program["components"][i])
            c["weight"] = float(max(0.2, min(2.0, c.get("weight", 1.0) * rng.uniform(0.8, 1.25))))
            program["components"][i] = _sanitize_component(c)
    elif op == "swap_variant":
        c = dict(program["components"][idx])
        c["mutation_type"] = rng.choice([v for v in focus if v not in avoid] or ["regime_momentum"])
        program["components"][idx] = _sanitize_component(c)
    program["components"] = [_sanitize_component(c) for c in program["components"]]
    return program


def _critic_assess_program(program):
    issues = []
    score_adj = 0.0
    comps = program.get("components", [])
    allowed_variants = {v for v, _ in parameter_search_variant_space()}
    core_variants = {"regime_momentum", "volume_confirm", "volume_gate"}
    composite_support = core_variants | {"rank_norm", "plain", "regime_gate", "ts_momentum"}
    if len(comps) == 0:
        issues.append("empty_program")
        score_adj -= 1.0
    if len(comps) > 4:
        issues.append("too_many_components")
        score_adj -= 0.15
    if len(comps) > 2:
        issues.append("bad_component_count")
        score_adj -= 0.35
    comp_variants = []
    for c in comps:
        ss = int(c.get("short_span", 54))
        ls = int(c.get("long_span", 90))
        variant = c.get("mutation_type", EVOLVE_BENCHMARK_MUTATION)
        comp_variants.append(variant)
        if ls < ss + 30:
            issues.append("invalid_span_gap")
            score_adj -= 0.20
        if variant not in allowed_variants:
            issues.append("unknown_variant")
            score_adj -= 0.50
            continue
        p = normalize_aux_params(c.get("params", {}))
        if variant in ("vol_scale", "vol_adjusted", "multi_factor"):
            if "vol_window" in p and int(p["vol_window"]) <= 1:
                issues.append("bad_vol_window")
                score_adj -= 0.30
            if "vol_window" in p and int(p["vol_window"]) > 252:
                issues.append("oversized_vol_window")
                score_adj -= 0.05
        if variant in ("volume_gate", "volume_confirm"):
            if "vol_gate_window" in p and int(p["vol_gate_window"]) <= 1:
                issues.append("bad_vol_gate_window")
                score_adj -= 0.30
        if variant in ("regime_gate", "regime_momentum"):
            threshold = float(p.get("vix_threshold", 20.0))
            if threshold <= 5.0 or threshold >= 80.0:
                issues.append("bad_vix_threshold")
                score_adj -= 0.30
    if len(comp_variants) >= 2:
        if len(set(comp_variants)) != len(comp_variants):
            issues.append("bad_duplicate_variant_composite")
            score_adj -= 0.25
        if not any(v in core_variants for v in comp_variants):
            issues.append("bad_composite_anchor")
            score_adj -= 0.40
        if any(v not in composite_support for v in comp_variants):
            issues.append("bad_composite_variant_mix")
            score_adj -= 0.30
        if any(v in ("vol_scale", "vol_adjusted", "multi_factor") for v in comp_variants):
            issues.append("bad_fragile_vol_composite")
            score_adj -= 0.45
    reject_issues = [i for i in issues if i.startswith("invalid_") or i.startswith("bad_") or i.startswith("unknown_")]
    return {"ok": len(reject_issues) == 0, "issues": issues, "score_adjust": float(score_adj)}


def _component_with_variant(template, variant):
    c = dict(template or {})
    c["mutation_type"] = variant
    params = normalize_aux_params(c.get("params", {}))
    if variant in ("volume_gate", "volume_confirm"):
        params = {"vol_gate_window": int(params.get("vol_gate_window", 20))}
    elif variant in ("regime_gate", "regime_momentum"):
        params = {"vix_threshold": float(params.get("vix_threshold", 22.0))}
    elif variant in ("vol_scale", "vol_adjusted", "multi_factor"):
        params = {"vol_window": int(params.get("vol_window", 20))}
    else:
        params = {}
    c["params"] = params
    return _sanitize_component(c)


def _repair_program_pre_eval(program, gen_program=None, rng=None):
    rng = rng or random.Random(PARAM_SEARCH_SEED)
    comps = [_sanitize_component(c) for c in (program or {}).get("components", [])]
    if not comps:
        comps = [_sanitize_component({"mutation_type": "volume_gate", "short_span": 60, "long_span": 105, "params": {"vol_gate_window": 20}, "weight": 1.0})]
    if len(comps) > 2:
        comps = comps[:2]
    seen = set()
    repaired = []
    replacement_pool = ["volume_gate", "volume_confirm", "rank_norm", "plain", "regime_gate", "ts_momentum"]
    for c in comps:
        variant = c.get("mutation_type", EVOLVE_BENCHMARK_MUTATION)
        if variant in seen:
            variant = next((v for v in replacement_pool if v not in seen), "volume_gate")
            c = _component_with_variant(c, variant)
        seen.add(c.get("mutation_type", variant))
        repaired.append(_sanitize_component(c))
    return {"components": repaired}


def _repair_program_after_failure(program, error_text, gen_program=None, rng=None):
    err = str(error_text or "").lower()
    comps = [_sanitize_component(c) for c in (program or {}).get("components", [])]
    base = comps[0] if comps else {"short_span": 60, "long_span": 105, "weight": 1.0}
    if "beta drift" in err:
        return None
    if "dead after market-neutral" in err or "not genuinely long-short" in err or "low signal activity" in err:
        repaired = _component_with_variant(base, "volume_confirm")
        repaired["params"] = {"vol_gate_window": int(repaired.get("params", {}).get("vol_gate_window", 20))}
        return {"components": [_sanitize_component(repaired)]}
    if "duplicate" in err or "bad_composite" in err:
        a = _component_with_variant(base, "volume_gate")
        b = _component_with_variant(base, "rank_norm")
        b["short_span"] = int(max(36, min(66, int(b.get("short_span", 54)) - 3)))
        b["long_span"] = int(max(b["short_span"] + 30, min(150, int(b.get("long_span", 90)) + 6)))
        return _repair_program_pre_eval({"components": [a, b]}, gen_program=gen_program, rng=rng)
    return None


def _split_train_validation():
    n = len(close_train)
    n_val = max(int(n * VAL_FRACTION), 252)
    n_val = min(max(60, n_val), max(60, n - 120))
    cut = n - n_val
    core_idx = close_train.index[:cut]
    val_idx = close_train.index[cut:]
    return (
        close_train.loc[core_idx],
        volume_train.loc[core_idx],
        (vix_train.loc[core_idx] if "vix_train" in globals() and vix_train is not None else None),
        (tnx_train.loc[core_idx] if "tnx_train" in globals() and tnx_train is not None else None),
        close_train.loc[val_idx],
        volume_train.loc[val_idx],
        (vix_train.loc[val_idx] if "vix_train" in globals() and vix_train is not None else None),
        (tnx_train.loc[val_idx] if "tnx_train" in globals() and tnx_train is not None else None),
    )


def _eval_program_trial(program, context):
    code = _program_code(program)
    c_core, v_core, x_core, t_core, c_val, v_val, x_val, t_val = context
    sig_core, err_core = run_signal_code(code, c_core, v_core, vix_s=x_core, tnx_s=t_core, timeout=EVAL_TIMEOUT_SEC)
    if err_core:
        return {"error": f"core_eval: {err_core}", "code": code}
    m_core = backtest(sig_core, c_core)
    flipped = bool(m_core.get("prefer_inverted", False))
    if flipped:
        sig_core = -sig_core
        m_core = backtest(sig_core, c_core)

    sig_val, err_val = run_signal_code(code, c_val, v_val, vix_s=x_val, tnx_s=t_val, timeout=EVAL_TIMEOUT_SEC)
    if err_val:
        return {"error": f"val_eval: {err_val}", "code": code}
    if flipped:
        sig_val = -sig_val
    m_val = backtest(sig_val, c_val)

    raw_cs_std = float(m_val.get("raw_cs_std", m_core.get("raw_cs_std", 1.0)) or 0.0)
    raw_long_frac = float(m_val.get("raw_long_frac", m_core.get("raw_long_frac", 1.0)) or 0.0)
    raw_short_frac = float(m_val.get("raw_short_frac", m_core.get("raw_short_frac", 1.0)) or 0.0)
    signal_activity = float(m_val.get("signal_activity", m_core.get("signal_activity", 1.0)) or 0.0)
    beta_val = float(m_val.get("beta", 0.0))
    beta_raw = float(m_val.get("beta_raw", beta_val) or 0.0)
    beta_reduction = float(m_val.get("beta_reduction", abs(beta_raw) - abs(beta_val)) or 0.0)
    beta_neutralized = bool(m_val.get("beta_neutralized", globals().get("BETA_NEUTRALIZE_POSITIONS", False)))
    beta_gate_status = "scored_beta_high" if abs(beta_val) > BETA_LIMIT else "ok"
    if signal_activity < MIN_SIGNAL_ACTIVITY:
        return {"error": f"val_eval: SIGNAL_VALIDATION: low signal activity before walk-forward (active_frac={signal_activity:.2f})", "code": code}
    if raw_cs_std < MIN_RAW_CS_STD:
        return {"error": f"val_eval: SIGNAL_VALIDATION: weak cross-section before walk-forward (raw_cs_std={raw_cs_std:.3f})", "code": code}
    if min(raw_long_frac, raw_short_frac) < MIN_LONG_SHORT_FRAC:
        return {"error": f"val_eval: SIGNAL_VALIDATION: not genuinely long-short before walk-forward (long={raw_long_frac:.2f}, short={raw_short_frac:.2f})", "code": code}
    wf = []
    n = len(c_core)
    sz = max(n // WF_WINDOWS, 1)
    for w in range(WF_WINDOWS):
        lo = w * sz
        hi = (w + 1) * sz if w < WF_WINDOWS - 1 else n
        sc = c_core.iloc[lo:hi]
        sig_w = sig_core.iloc[lo:hi]
        mw = backtest(sig_w, sc)
        wf.append({"window": w, "sharpe": mw.get("sharpe", 0.0)})
    wf_median, wf_min = wf_summary(wf)

    val_wf = []
    n_val = len(c_val)
    val_windows = min(WF_WINDOWS, max(1, n_val // 126))
    val_sz = max(n_val // val_windows, 1)
    for w in range(val_windows):
        lo = w * val_sz
        hi = (w + 1) * val_sz if w < val_windows - 1 else n_val
        sc = c_val.iloc[lo:hi]
        if len(sc) < 60:
            continue
        sig_w = sig_val.iloc[lo:hi]
        mw = backtest(sig_w, sc)
        val_wf.append({"window": w, "sharpe": mw.get("sharpe", 0.0)})
    val_wf_median, val_wf_min = wf_summary(val_wf)
    val_wf_count = len(val_wf)

    train_sh = float(m_core.get("sharpe", 0.0))
    val_sh = float(m_val.get("sharpe", 0.0))
    train_cons = float(m_core.get("consistency", 0.0))
    val_cons = float(m_val.get("consistency", 0.0))
    to_val = float(m_val.get("avg_turnover", 0.0))
    max_dd_val = float(m_val.get("max_dd", 0.0))
    comps = [_sanitize_component(c) for c in program.get("components", [])]
    n_components = max(len(comps), 1)
    has_vol_scale = any(c.get("mutation_type") in ("vol_scale", "vol_adjusted") for c in comps)
    train_score = selection_score(
        train_sh,
        wf_median,
        train_cons,
        float(m_core.get("beta", 0.0)),
        float(m_core.get("avg_turnover", 0.0)),
        wf_min=wf_min,
        max_dd=float(m_core.get("max_dd", 0.0)),
    )
    raw_val_score = evolution_score(val_sh, val_wf_median, val_wf_min, val_cons, beta_val, to_val)
    val_score = raw_val_score
    val_wf_missing_penalty = EVOLVE_VAL_WF_MISSING_PENALTY if val_wf_count < 2 else 0.0
    beta_excess_penalty = EVOLVE_BETA_EXCESS_PENALTY * max(0.0, abs(beta_val) - BETA_LIMIT)
    train_wf_min_penalty = EVOLVE_TRAIN_WF_MIN_PENALTY * max(0.0, EVOLVE_TRAIN_WF_MIN_FLOOR - wf_min)
    val_score -= EVOLVE_COMPLEXITY_PENALTY * max(0, n_components - 1)
    val_score -= EVOLVE_VOL_SCALE_PENALTY if has_vol_scale else 0.0
    val_score -= val_wf_missing_penalty
    val_score -= beta_excess_penalty
    val_score -= train_wf_min_penalty
    val_score -= EVOLVE_TRAIN_VAL_GAP_PENALTY * max(0.0, train_score - raw_val_score)
    robustness_payload = {
        "train_sharpe": val_sh,
        "wf_median": val_wf_median,
        "wf_min": val_wf_min,
        "beta": beta_val,
        "beta_neutralized": beta_neutralized,
        "beta_raw": beta_raw,
        "beta_neutralized_value": beta_val,
        "beta_reduction": beta_reduction,
        "beta_gate_status": beta_gate_status,
        "turnover": to_val,
        "consistency": val_cons,
        "raw_cs_std": raw_cs_std,
        "raw_long_frac": raw_long_frac,
        "raw_short_frac": raw_short_frac,
        "signal_activity": signal_activity,
    }
    program_robustness_score = robustness_score(robustness_payload)
    robust = (
        train_sh > 0
        and val_sh > 0
        and val_wf_median >= EVOLVE_PROGRAM_MIN_VAL_WF_MEDIAN
        and val_wf_min >= EVOLVE_PROGRAM_MIN_VAL_WF_MIN
        and wf_min > -0.20
        and robust_ok(robustness_payload, wf_floor=-0.25)
        and program_robustness_score >= EVOLVE_PROGRAM_ROBUSTNESS_FLOOR
    )

    comp0 = _sanitize_component(program.get("components", [{}])[0])
    program_payload, program_hash, program_cluster = _program_identity(program)
    meta = signature_for_signal(
        "program_evolution",
        "program",
        code,
        short_span=comp0.get("short_span", 54),
        long_span=comp0.get("long_span", 90),
        base_family="program_evolution",
        params={"n_components": len(program_payload), "program_hash": program_hash},
        cluster_id=program_cluster,
    )

    val_ret = (sig_val.shift(1).fillna(0) * c_val.pct_change().fillna(0)).mean(axis=1)
    return {
        "source": "parameter_search_evolution",
        "family": "program_evolution",
        "base_family": "program_evolution",
        "mutation_type": comp0.get("mutation_type", "program"),
        "short_span": int(comp0.get("short_span", 54)),
        "long_span": int(comp0.get("long_span", 90)),
        "params": {"n_components": len(program_payload), "program_hash": program_hash},
        "component_params": normalize_aux_params(comp0.get("params", {})),
        "program": program,
        "program_components": program_payload,
        "program_hash": program_hash,
        "code": code,
        "train_sharpe": train_sh,
        "val_sharpe": val_sh,
        "consistency": val_cons,
        "wf_median": wf_median,
        "wf_min": wf_min,
        "val_wf_median": val_wf_median,
        "val_wf_min": val_wf_min,
        "val_wf_count": val_wf_count,
        "beta": beta_val,
        "beta_neutralized": beta_neutralized,
        "beta_raw": beta_raw,
        "beta_neutralized_value": beta_val,
        "beta_reduction": beta_reduction,
        "beta_gate_status": beta_gate_status,
        "turnover": to_val,
        "max_dd": max_dd_val,
        "score_train": train_score,
        "score_val_raw": raw_val_score,
        "score_val": val_score,
        "complexity_penalty": EVOLVE_COMPLEXITY_PENALTY * max(0, n_components - 1),
        "val_wf_missing_penalty": val_wf_missing_penalty,
        "beta_excess_penalty": beta_excess_penalty,
        "train_wf_min_penalty": train_wf_min_penalty,
        "train_val_gap_penalty": EVOLVE_TRAIN_VAL_GAP_PENALTY * max(0.0, train_score - raw_val_score),
        "score": val_score,
        "robustness_score": program_robustness_score,
        "raw_cs_std": raw_cs_std,
        "raw_long_frac": raw_long_frac,
        "raw_short_frac": raw_short_frac,
        "signal_activity": signal_activity,
        "robust_ok": bool(robust),
        "component_count": n_components,
        "model_size": n_components,
        "model_size_key": f"components={n_components}",
        "flipped": flipped,
        "_val_ret": val_ret,
        **meta,
    }


def _candidate_programs(parents, gen_program, n_trials, rng):
    if not parents:
        parents = [{
            "parent_id": f"seed:{EVOLVE_BENCHMARK_MUTATION}:54:90",
            "program": {"components": [{"mutation_type": "regime_momentum", "short_span": 54, "long_span": 90, "params": {"vix_threshold": 26.0}, "weight": 1.0}]},
        }]
    candidates = []
    seen = set()
    parent_records = []
    for p in parents:
        parent_records.append(
            {
                "row": p,
                "program": p.get("program") if isinstance(p.get("program"), dict) else _row_to_program(p),
                "parent_id": row_identity(p, "parent"),
                "parent_signature": p.get("signature"),
                "parent_program_hash": p.get("program_hash"),
                "parent_score": p.get("score"),
            }
        )
    min_short = int(min(PARAM_SEARCH_SHORT_SPANS))
    max_short = int(max(PARAM_SEARCH_SHORT_SPANS))
    min_long = int(min(PARAM_SEARCH_LONG_SPANS))
    max_long = int(max(PARAM_SEARCH_LONG_SPANS))
    regime_thresholds = [float(x) for x in PARAM_SEARCH_REGIME_THRESHOLDS] if "PARAM_SEARCH_REGIME_THRESHOLDS" in globals() else [20.0, 24.0, 28.0]
    vol_windows = [int(x) for x in PARAM_SEARCH_VOL_WINDOWS] if "PARAM_SEARCH_VOL_WINDOWS" in globals() else [20, 30]
    vol_gate_windows = [int(x) for x in PARAM_SEARCH_VOL_GATE_WINDOWS] if "PARAM_SEARCH_VOL_GATE_WINDOWS" in globals() else [20, 40]
    force_explore = bool(gen_program.get("force_explore", False))
    allowed_variants = {v for v, _ in parameter_search_variant_space()}
    focus_variants = [v for v in gen_program.get("focus_variants", ["regime_momentum"]) if v in allowed_variants]
    fallback_focus = ["regime_momentum", "volume_confirm", "volume_gate", "rank_norm", "plain", "regime_gate", "ts_momentum"]
    if float(gen_program.get("dominant_cluster_frac", 0.0) or 0.0) >= 0.50:
        fallback_focus = ["volume_confirm", "volume_gate", "rank_norm", "plain", "regime_momentum", "regime_gate", "ts_momentum"]
    for variant in fallback_focus:
        if variant in allowed_variants and variant not in focus_variants:
            focus_variants.append(variant)
        if len(focus_variants) >= EVOLVE_MIN_FOCUS_VARIANTS:
            break
    if not focus_variants:
        focus_variants = ["regime_momentum", "volume_confirm", "volume_gate", "rank_norm", "regime_gate"]

    def _sample_component_for_variant(var):
        ss = int(rng.randint(min_short, max_short))
        ls = int(rng.randint(max(min_long, ss + 30), max_long))
        params = {}
        if var in ("regime_gate", "regime_momentum"):
            params["vix_threshold"] = float(rng.choice(regime_thresholds))
        elif var in ("vol_scale", "vol_adjusted", "multi_factor"):
            params["vol_window"] = int(rng.choice(vol_windows))
        elif var in ("volume_gate", "volume_confirm"):
            params["vol_gate_window"] = int(rng.choice(vol_gate_windows))
        return _sanitize_component(
            {
                "mutation_type": var,
                "short_span": ss,
                "long_span": ls,
                "params": params,
                "weight": float(rng.uniform(0.6, 1.4)),
            }
        )

    def _sample_component():
        return _sample_component_for_variant(rng.choice(focus_variants))

    def _quota_program(family):
        if family == "composite":
            a, b = rng.choice(EVOLVE_SAFE_COMPOSITE_PAIRS)
            return {"components": [_sample_component_for_variant(a), _sample_component_for_variant(b)]}
        return {"components": [_sample_component_for_variant(family)]}

    def _add_candidate(prog, parent_meta, origin):
        prog = _repair_program_pre_eval(prog, gen_program, rng)
        _, sig, _ = _program_identity(prog)
        if sig in seen:
            return False
        seen.add(sig)
        candidates.append(
            {
                "program": prog,
                "program_sig": sig,
                "candidate_family": _program_primary_family(prog),
                "parent_id": parent_meta.get("parent_id"),
                "parent_signature": parent_meta.get("parent_signature"),
                "parent_program_hash": parent_meta.get("parent_program_hash"),
                "parent_score": parent_meta.get("parent_score"),
                "origin": origin,
            }
        )
        return True

    quota_total = max(0, min(n_trials, int(round(n_trials * 0.65))))
    quota_items = list(EVOLVE_FAMILY_QUOTAS.items())
    quota_counts = {family: max(1, int(round(quota_total * frac))) for family, frac in quota_items}
    while sum(quota_counts.values()) > quota_total and quota_counts:
        family = max(quota_counts, key=lambda k: quota_counts[k])
        quota_counts[family] -= 1
    for family, quota in quota_counts.items():
        if family not in allowed_variants and family != "composite":
            continue
        attempts = 0
        while len(candidates) < n_trials and quota > 0 and attempts < max(quota * 20, 50):
            attempts += 1
            parent_meta = {
                "parent_id": f"quota:{family}:g{gen_program.get('gen_id', 'unknown')}",
                "parent_signature": None,
                "parent_program_hash": None,
                "parent_score": None,
            }
            if _add_candidate(_quota_program(family), parent_meta, f"family_quota:{family}"):
                quota -= 1

    while len(candidates) < n_trials:
        parent_meta = {
            "parent_id": f"explore:g{gen_program.get('gen_id', 'unknown')}",
            "parent_signature": None,
            "parent_program_hash": None,
            "parent_score": None,
        }
        if force_explore or rng.random() < EVOLVE_RANDOM_EXPLORATION_FRAC:
            n_comp = 1 if rng.random() < 0.88 else 2
            base = {"components": [_sample_component() for _ in range(n_comp)]}
            prog = _mutate_program(base, gen_program, rng)
        else:
            parent_meta = parent_records[rng.randrange(0, len(parent_records))]
            parent_prog = parent_meta["program"]
            prog = _mutate_program(parent_prog, gen_program, rng)
        _add_candidate(prog, parent_meta, "program_mutation")
        if len(seen) > max(n_trials * 60, 20000):
            break
    return candidates


def _deterministic_champion_candidates(global_rows, limit):
    anchors = [
        r for r in global_rows
        if r.get("source") == EVOLVE_BENCHMARK_SOURCE
        and r.get("robust_ok")
        and r.get("score") is not None
    ]
    anchors = sorted(anchors, key=lambda r: (-_heldout_proxy_score(r), abs(int(r.get("long_span", 90)) - 90)))
    candidates = []
    seen = set()
    
    def add_row(row):
        program = _row_to_program(row)
        _, program_hash, _ = _program_identity(program)
        if program_hash in seen:
            return False
        seen.add(program_hash)
        candidates.append(
            {
                "program": program,
                "program_sig": program_hash,
                "origin": "deterministic_champion",
                "parent_id": row_identity(row, "deterministic"),
                "parent_signature": row.get("signature"),
                "parent_program_hash": row.get("program_hash"),
                "parent_score": row.get("score"),
            }
        )
        return True

    for variant in EVOLVE_CHAMPION_VARIANTS:
        for row in [r for r in anchors if r.get("mutation_type") == variant]:
            add_row(row)
            break
        if len(candidates) >= limit:
            return candidates[:limit]
    for row in anchors:
        if len(candidates) >= limit:
            break
        add_row(row)
    return candidates


def _tournament_select(rows, k_survivors, rng):
    pool = [r for r in rows if r.get("robust_ok") and r.get("score") is not None]
    if not pool:
        pool = [r for r in rows if r.get("score") is not None]
    original_pool = list(pool)
    selected = []
    used = set()
    used_clusters = set()
    while len(selected) < k_survivors and pool:
        contenders = rng.sample(pool, k=min(EVOLVE_TOURNAMENT_K, len(pool)))
        best = None
        best_u = -1e9
        for c in contenders:
            u = float(c.get("score", -999.0)) + 0.15 * float(c.get("score_train", c.get("score", -999.0)))
            if selected and c.get("_val_ret") is not None:
                cors = []
                for s in selected:
                    if s.get("_val_ret") is None:
                        continue
                    ix = c["_val_ret"].index.intersection(s["_val_ret"].index)
                    if len(ix) < 30:
                        continue
                    corr = c["_val_ret"].loc[ix].corr(s["_val_ret"].loc[ix])
                    if pd.notna(corr):
                        cors.append(abs(float(corr)))
                if cors:
                    u -= EVOLVE_DIVERSITY_PENALTY * float(np.mean(cors))
            if u > best_u:
                best_u = u
                best = c
        if best is None:
            break
        sig = best.get("signature")
        if sig and sig in used:
            pool = [p for p in pool if p is not best]
            continue
        cid = best.get("cluster_id")
        if cid and cid in used_clusters and len(pool) > 1:
            pool = [p for p in pool if p is not best]
            continue
        if sig:
            used.add(sig)
        if cid:
            used_clusters.add(cid)
        selected.append(best)
        pool = [p for p in pool if p is not best]
    return _family_capped_rows(selected + [p for p in original_pool if p not in selected], k_survivors)


def _merge_parent_candidates(*groups, limit):
    merged = []
    seen = set()
    for group in groups:
        for row in group or []:
            ident = (
                row.get("program_hash")
                or row.get("signature")
                or row.get("parent_id")
                or row.get("cluster_id")
                or row_identity(row, "parent")
            )
            if ident in seen:
                continue
            seen.add(ident)
            merged.append(row)
    return _family_capped_rows(merged, limit)


def _select_next_generation_parents(current_parents, valid, robust, global_rows, rng):
    reseed = []
    did_reseed = False
    if len(robust) < EVOLVE_ROBUST_RESET_THRESHOLD:
        robust_global = [r for r in global_rows if r.get("robust_ok")]
        reseed = _diverse_top_parents_from_rows(robust_global, EVOLVE_BEAM_WIDTH)
        did_reseed = bool(reseed)

    survivor_pool = robust if robust else valid
    survivors = _tournament_select(survivor_pool, EVOLVE_SURVIVORS, rng)

    if did_reseed:
        parents = _merge_parent_candidates(reseed, survivors, current_parents, limit=EVOLVE_BEAM_WIDTH)
    elif survivors:
        parents = survivors
    else:
        parents = list(current_parents or [])
    return parents, survivors, did_reseed


def _inc_count(mapping, key, n=1):
    key = str(key or "unknown")
    mapping[key] = int(mapping.get(key, 0)) + int(n)


def _generation_family_telemetry(candidates, rows, survivors):
    telemetry = {
        "generated": {},
        "critic_rejected": {},
        "validation_failed": {},
        "scored": {},
        "scored_beta_high": {},
        "robust": {},
        "survivor": {},
        "failure_reasons": {},
    }
    for cand in candidates or []:
        _inc_count(telemetry["generated"], cand.get("candidate_family") or _program_primary_family(cand.get("program", {})))
    for row in rows or []:
        fam = _row_family(row)
        err = row.get("error")
        if row.get("score") is not None:
            _inc_count(telemetry["scored"], fam)
            if row.get("beta_gate_status") == "scored_beta_high":
                _inc_count(telemetry["scored_beta_high"], fam)
            if row.get("robust_ok"):
                _inc_count(telemetry["robust"], fam)
        elif err:
            bucket = "critic_rejected" if str(err).startswith("critic_reject:") else "validation_failed"
            _inc_count(telemetry[bucket], fam)
            reason = str(err).split(":", 1)[0] if ":" in str(err) else str(err)
            reason_map = telemetry["failure_reasons"].setdefault(fam, {})
            _inc_count(reason_map, reason)
    for row in survivors or []:
        _inc_count(telemetry["survivor"], _row_family(row))
    return telemetry


def _should_stop_for_zero_robust(gen_id, zero_robust_streak, valid_rate):
    """Stop only when zero-robust also indicates generator validity collapse."""
    return bool(
        gen_id >= EVOLVE_ZERO_ROBUST_MIN_GENERATIONS
        and zero_robust_streak >= EVOLVE_ZERO_ROBUST_PATIENCE
        and float(valid_rate or 0.0) < EVOLVE_ZERO_ROBUST_MIN_VALID_RATE
    )


def run_evolution_search():
    all_rows = []
    generations = []
    memory_notes = []
    programs = []
    diagnosis_events = []
    parents = _seed_parents_for_generation()
    rng = random.Random(PARAM_SEARCH_SEED)
    c_core, v_core, x_core, t_core, c_val, v_val, x_val, t_val = _split_train_validation()
    eval_context = (c_core, v_core, x_core, t_core, c_val, v_val, x_val, t_val)
    total_evals = 0
    stop_reason = "max_generations_reached"
    seen_clusters = set([p.get("cluster_id") for p in parents if p.get("cluster_id")])
    started = time.time()
    prev_best = -1e9
    prev_median = -1e9
    prev_div = 1.0
    patience_bad = 0
    zero_robust_streak = 0
    best_so_far_streak = 0
    global_rows = [r for r in _safe_load_search_results() if r.get("score") is not None]

    for gen_id in range(1, EVOLVE_MAX_GENERATIONS + 1):
        if total_evals >= EVOLVE_MAX_TOTAL_EVALS:
            stop_reason = "max_total_evals_reached"
            break
        if EVOLVE_MAX_WALLCLOCK_HOURS and (time.time() - started) / 3600.0 >= EVOLVE_MAX_WALLCLOCK_HOURS:
            stop_reason = "max_wallclock_reached"
            break

        recent_rows = all_rows[-400:] if all_rows else global_rows[:400]
        gen_program = _build_generation_program(gen_id, parents, recent_rows)
        programs.append(gen_program)

        remaining = EVOLVE_MAX_TOTAL_EVALS - total_evals
        target = min(EVOLVE_TRIALS_PER_GENERATION, remaining)
        candidates = _candidate_programs(parents, gen_program, target, rng)
        champion_candidates = _deterministic_champion_candidates(global_rows, EVOLVE_DETERMINISTIC_CHAMPIONS)
        if champion_candidates:
            seen_programs = {c.get("program_sig") for c in candidates}
            injected = []
            for cand in champion_candidates:
                if cand.get("program_sig") in seen_programs:
                    continue
                injected.append(cand)
                seen_programs.add(cand.get("program_sig"))
            candidates = injected + candidates
            candidates = candidates[:target]
        if len(candidates) < EVOLVE_MIN_TRIALS_PER_GENERATION:
            min_needed = EVOLVE_MIN_TRIALS_PER_GENERATION
            attempts = 0
            while len(candidates) < min_needed and attempts < EVOLVE_RESTART_ATTEMPTS:
                attempts += 1
                refill_program = dict(gen_program)
                refill_program["force_explore"] = True
                refill_target = max(min_needed - len(candidates), EVOLVE_MIN_TRIALS_PER_GENERATION)
                extra = _candidate_programs(parents, refill_program, refill_target, rng)
                if not extra:
                    continue
                seen_prog = {c.get("program_sig") for c in candidates}
                for c in extra:
                    ps = c.get("program_sig")
                    if ps in seen_prog:
                        continue
                    seen_prog.add(ps)
                    candidates.append(c)
                if len(candidates) >= min_needed:
                    break
            if len(candidates) < min_needed:
                floor = min(EVOLVE_MIN_CANDIDATE_FLOOR, target)
                if len(candidates) < floor:
                    stop_reason = "search_space_exhausted"
                    break
                _warn(f"g{gen_id}: candidate floor relaxed to {len(candidates)} due sparse search space")

        gen_rows = []
        for cand in candidates:
            critique = _critic_assess_program(cand["program"])
            if not critique["ok"]:
                repaired_program = _repair_program_after_failure(
                    cand["program"],
                    "critic_reject:" + ",".join(critique["issues"]),
                    gen_program=gen_program,
                    rng=rng,
                )
                if repaired_program is not None:
                    repaired_payload, repaired_hash, _ = _program_identity(repaired_program)
                    if repaired_hash != cand.get("program_sig"):
                        cand["program"] = repaired_program
                        cand["program_sig"] = repaired_hash
                        cand["candidate_family"] = _program_primary_family(repaired_program)
                        cand["repair_applied"] = "critic_pre_eval"
                        critique = _critic_assess_program(cand["program"])
                if not critique["ok"]:
                    program_payload, program_hash, program_cluster = _program_identity(cand["program"])
                    primary_family = _program_primary_family(cand["program"])
                    gen_rows.append(
                        {
                            "source": "parameter_search_evolution",
                            "family": "program_evolution",
                            "base_family": "program_evolution",
                            "mutation_type": primary_family,
                            "candidate_family": cand.get("candidate_family", primary_family),
                            "error": "critic_reject:" + ",".join(critique["issues"]),
                            "gen_id": gen_id,
                            "parent_id": cand.get("parent_id") or f"unknown:g{gen_id}",
                            "parent_signature": cand.get("parent_signature"),
                            "parent_program_hash": cand.get("parent_program_hash"),
                            "program_sig": cand.get("program_sig"),
                            "program_hash": program_hash,
                            "cluster_id": program_cluster,
                            "program": cand["program"],
                            "program_components": program_payload,
                            "origin": cand.get("origin", "program_mutation"),
                            "repair_applied": cand.get("repair_applied"),
                        }
                    )
                    continue
            row = _eval_program_trial(cand["program"], eval_context)
            if row.get("score") is None and row.get("error"):
                repaired_program = _repair_program_after_failure(cand["program"], row.get("error"), gen_program=gen_program, rng=rng)
                if repaired_program is not None:
                    repaired_payload, repaired_hash, _ = _program_identity(repaired_program)
                    if repaired_hash != cand.get("program_sig"):
                        repaired_row = _eval_program_trial(repaired_program, eval_context)
                        repaired_row["repair_source_error"] = row.get("error")
                        repaired_row["repair_applied"] = "failure_repair"
                        cand["program"] = repaired_program
                        cand["program_sig"] = repaired_hash
                        cand["candidate_family"] = _program_primary_family(repaired_program)
                        row = repaired_row
            row.setdefault("source", "parameter_search_evolution")
            row.setdefault("family", "program_evolution")
            row.setdefault("base_family", "program_evolution")
            row["mutation_type"] = row.get("mutation_type") or _program_primary_family(cand["program"])
            row["candidate_family"] = cand.get("candidate_family", _program_primary_family(cand["program"]))
            row["gen_id"] = gen_id
            row["parent_id"] = cand.get("parent_id") or row.get("parent_id") or f"unknown:g{gen_id}"
            row["parent_signature"] = cand.get("parent_signature")
            row["parent_program_hash"] = cand.get("parent_program_hash")
            row["program_sig"] = cand["program_sig"]
            row["origin"] = cand.get("origin", "program_mutation")
            if row.get("score") is not None:
                row["score"] = float(row["score"] + critique.get("score_adjust", 0.0))
            parent_score = cand.get("parent_score")
            row["score_gain_vs_parent"] = float(row["score"] - parent_score) if row.get("score") is not None and parent_score is not None else None
            gen_rows.append(row)

        total_evals += len(gen_rows)
        valid = [r for r in gen_rows if r.get("score") is not None]
        robust = [r for r in valid if r.get("robust_ok")]
        valid_rate = float(len(valid) / len(gen_rows)) if gen_rows else 0.0
        zero_robust_counted = len(robust) == 0
        zero_robust_streak = zero_robust_streak + 1 if zero_robust_counted else 0
        parents, survivors, did_reseed = _select_next_generation_parents(parents, valid, robust, global_rows, rng)
        if did_reseed:
            _warn(f"g{gen_id}: robust_count={len(robust)} below threshold; reseeded parents")

        gen_scores = [r.get("score") for r in robust if r.get("score") is not None]
        ci_lo, ci_hi = _bootstrap_ci(gen_scores, n_boot=300, alpha=0.10, seed=PARAM_SEARCH_SEED + gen_id)
        best_score = max(gen_scores, default=max([r.get("score", -1e9) for r in valid], default=-1e9))
        med_score = float(np.median(gen_scores)) if gen_scores else -1e9
        new_clusters = set([r.get("cluster_id") for r in valid if r.get("cluster_id") and r.get("cluster_id") not in seen_clusters])
        new_cluster_count = len(new_clusters)
        seen_clusters.update(new_clusters)
        prior_best = None if prev_best <= -1e8 else float(prev_best)
        if prior_best is None or best_score > prior_best:
            best_so_far_streak = 0
        else:
            best_so_far_streak += 1
        best_so_far = None if max(prev_best, best_score) <= -1e8 else float(max(prev_best, best_score))
        diversity = 0.0
        if len(survivors) >= 2:
            cors = []
            for i in range(len(survivors)):
                for j in range(i + 1, len(survivors)):
                    si, sj = survivors[i].get("_val_ret"), survivors[j].get("_val_ret")
                    if si is None or sj is None:
                        continue
                    ix = si.index.intersection(sj.index)
                    if len(ix) < 30:
                        continue
                    c = si.loc[ix].corr(sj.loc[ix])
                    if pd.notna(c):
                        cors.append(abs(float(c)))
            diversity = float(1.0 - np.mean(cors)) if cors else 0.0

        family_telemetry = _generation_family_telemetry(candidates, gen_rows, survivors)
        best_gain = 0.0 if prev_best <= -1e8 else float(best_score - prev_best)
        median_gain = 0.0 if prev_median <= -1e8 else float(med_score - prev_median)
        diversity_gain = float(diversity - prev_div)
        prev_best = max(prev_best, best_score)
        prev_median = med_score
        prev_div = diversity

        no_growth = (
            best_gain < EVOLVE_MIN_BEST_GAIN
            and median_gain < EVOLVE_MIN_MEDIAN_GAIN
            and new_cluster_count < EVOLVE_MIN_NOVELTY_GAIN
            and diversity_gain < EVOLVE_MIN_DIVERSITY_GAIN
        )
        patience_bad = patience_bad + 1 if no_growth else 0

        for r in gen_rows:
            if "_val_ret" in r:
                del r["_val_ret"]
        all_rows.extend(gen_rows)
        global_rows.extend([r for r in valid if r.get("score") is not None])

        benchmark_candidates = [
            r for r in global_rows
            if r.get("source") == EVOLVE_BENCHMARK_SOURCE
            and r.get("score") is not None
        ]
        benchmark_candidates = sorted(
            benchmark_candidates,
            key=lambda row: (not bool(row.get("robust_ok")), -float(row.get("score", -999.0) or -999.0)),
        )
        benchmark_row = benchmark_candidates[0] if benchmark_candidates else None

        generation_payload = {
            "gen_id": gen_id,
            "planned_trials": len(candidates),
            "eval_count": len(gen_rows),
            "valid_count": len(valid),
            "valid_rate": valid_rate,
            "robust_count": len(robust),
            "zero_robust_streak": int(zero_robust_streak),
            "zero_robust_counted": bool(zero_robust_counted),
            "best_score": None if best_score <= -1e8 else float(best_score),
            "best_so_far": best_so_far,
            "best_so_far_streak": int(best_so_far_streak),
            "median_score": None if med_score <= -1e8 else float(med_score),
            "best_gain": float(best_gain),
            "median_gain": float(median_gain),
            "new_cluster_count": int(new_cluster_count),
            "diversity": float(diversity),
            "score_ci_90": [ci_lo, ci_hi],
            "survivor_count": len(survivors),
            "survivor_signatures": [s.get("signature") for s in survivors],
            "survivor_families": family_telemetry.get("survivor", {}),
            "family_telemetry": family_telemetry,
            "program_focus_variants": list(gen_program.get("focus_variants", [])),
            "adaptive_diagnostic_run": bool(EVOLVE_ADAPTIVE_DIAGNOSTIC_RUN),
        }
        gen_diagnoses = _generation_diagnosis_events(generation_payload, valid, robust, benchmark_row=benchmark_row)
        generation_payload["diagnoses"] = gen_diagnoses
        diagnosis_events.extend(gen_diagnoses)
        for event in gen_diagnoses:
            if event.get("kind") == "stagnation":
                _warn(
                    f"g{gen_id}: stagnation diagnosis triggered after "
                    f"{event.get('streak', 0)} generations without best-so-far improvement"
                )
            elif event.get("kind") == "zero_robust":
                _warn(
                    f"g{gen_id}: zero-robust diagnosis "
                    f"{event.get('failure_counts', {})}"
                )

        reflection_text = _summarize_generation_reflection(gen_id, gen_rows, survivors)
        memory_notes.append({
            "gen_id": gen_id,
            "reflection": reflection_text,
            "best_gain": best_gain,
            "median_gain": median_gain,
            "new_clusters": new_cluster_count,
            "diversity": diversity,
            "valid_rate": valid_rate,
            "zero_robust_streak": zero_robust_streak,
            "best_so_far_streak": best_so_far_streak,
            "diagnoses": gen_diagnoses,
        })

        generations.append(generation_payload)

        stop_after_generation = False
        if _should_stop_for_zero_robust(gen_id, zero_robust_streak, valid_rate):
            stop_reason = "zero_robust_streak_reached"
            stop_after_generation = True
        elif gen_id >= EVOLVE_MIN_GENERATIONS and best_so_far_streak >= EVOLVE_HARD_STAGNATION_PATIENCE:
            stop_reason = "stagnation_reached"
            stop_after_generation = True
        elif gen_id >= EVOLVE_MIN_GENERATIONS and patience_bad >= EVOLVE_CONVERGENCE_PATIENCE:
            stop_reason = "convergence_reached"
            stop_after_generation = True

        if EVOLVE_PARTIAL_WRITE_EVERY and gen_id % max(1, int(EVOLVE_PARTIAL_WRITE_EVERY)) == 0:
            _write_evolution_artifacts(
                all_rows,
                generations,
                memory_notes,
                programs,
                diagnosis_events,
                total_evals,
                stop_reason,
                partial=True,
                started=started,
                suppress_errors=True,
            )

        if stop_after_generation:
            break

    all_rows = _write_evolution_artifacts(
        all_rows,
        generations,
        memory_notes,
        programs,
        diagnosis_events,
        total_evals,
        stop_reason,
        partial=False,
        started=started,
    )
    return all_rows


def run_parameter_search(n_trials=PARAM_SEARCH_TRIALS):
    if EVOLVE_ENABLED:
        return run_evolution_search()
    trials = sample_parameter_trials(n_trials=n_trials)
    rows = []
    seen_sig = set()
    for trial in trials:
        try:
            row = evaluate_structured_trial(trial, source="parameter_search")
        except Exception as e:
            row = {
                "source": "parameter_search",
                "parent_id": f"search:{trial['mutation_type']}:{trial['short_span']}:{trial['long_span']}",
                "family": deterministic_family_for_mutation(trial["mutation_type"])[0],
                "base_family": "ewm",
                "mutation_type": trial["mutation_type"],
                "short_span": trial["short_span"],
                "long_span": trial["long_span"],
                "params": normalize_aux_params(trial.get("params", {})),
                "error": str(e),
            }
        sig = row.get("signature")
        if sig and sig in seen_sig:
            continue
        if sig:
            seen_sig.add(sig)
        rows.append(row)
    rows = sorted(rows, key=lambda r: (not r.get("robust_ok", False), -r.get("score", -999)))
    _atomic_write_json(PARAM_SEARCH_FILE, rows)
    return rows


def run_deterministic_search():
    short_grid = [36, 39, 42, 45, 48, 51, 54, 57, 60]
    long_grid = [90, 95, 100, 105, 110, 120, 130, 140]
    variants = deterministic_variant_grid()
    rows = []
    for short_span in short_grid:
        for long_span in long_grid:
            if long_span < short_span + 30:
                continue
            for variant, aux in variants:
                fn = deterministic_signal_factory(short_span, long_span, variant=variant, aux=aux)
                try:
                    sig = fn(close_train, volume_train, vix_train, tnx_train)
                    m = backtest(sig, close_train)
                    flipped = bool(m["prefer_inverted"])
                    beta = m["beta_inverted"] if flipped else m["beta"]
                    train_sharpe = m["sharpe_inverted"] if flipped else m["sharpe"]
                    ann_return = m["ann_return_inverted"] if flipped else m["ann_return"]
                    dd = m["max_dd_inverted"] if flipped else m["max_dd"]
                    consistency = m["consistency_inverted"] if flipped else m["consistency"]
                    wf = baseline_walk_forward(fn, close_train, volume_train, vix_train, tnx_train, flipped=flipped)
                    wf_median, wf_min = wf_summary(wf)
                    turnover = m["avg_turnover"]
                    score = selection_score(train_sharpe, wf_median, consistency, beta, turnover, wf_min=wf_min, max_dd=dd)
                    family, base_family = deterministic_family_for_mutation(variant)
                    aux_norm = normalize_aux_params(aux)
                    code_label = f"ewm(span={short_span}) ewm(span={long_span}) {variant} {aux_norm}"
                    meta = signature_for_signal(
                        family,
                        variant,
                        code_label,
                        short_span=short_span,
                        long_span=long_span,
                        base_family=base_family,
                        params=aux_norm,
                    )
                    aux_tag = ",".join(f"{k}={v}" for k, v in aux_norm.items()) if aux_norm else "base"
                    metric_payload = {
                        "train_sharpe": train_sharpe,
                        "wf_median": wf_median,
                        "wf_min": wf_min,
                        "beta": beta,
                        "turnover": turnover,
                        "consistency": consistency,
                        "raw_cs_std": m.get("raw_cs_std", 1.0),
                        "raw_long_frac": m.get("raw_long_frac", 1.0),
                        "raw_short_frac": m.get("raw_short_frac", 1.0),
                        "signal_activity": m.get("signal_activity", 1.0),
                    }
                    row = {
                        "source": "deterministic",
                        "parent_id": f"det:{variant}:{short_span}:{long_span}:{aux_tag}",
                        "family": family,
                        "base_family": base_family,
                        "mutation_type": variant,
                        "short_span": short_span,
                        "long_span": long_span,
                        "params": aux_norm,
                        "train_sharpe": train_sharpe,
                        "ann_return": ann_return,
                        "beta": beta,
                        "turnover": turnover,
                        "max_dd": dd,
                        "consistency": consistency,
                        "wf_median": wf_median,
                        "wf_min": wf_min,
                        "score": score,
                        "robustness_score": robustness_score(metric_payload),
                        "raw_cs_std": m.get("raw_cs_std", 1.0),
                        "raw_long_frac": m.get("raw_long_frac", 1.0),
                        "raw_short_frac": m.get("raw_short_frac", 1.0),
                        "signal_activity": m.get("signal_activity", 1.0),
                        "flipped": flipped,
                        "shortlist_ok": shortlist_ok(beta, turnover),
                        "robust_ok": robust_ok(metric_payload),
                        "component_count": 1,
                        "model_size": 1,
                        "model_size_key": "components=1",
                        **meta,
                    }
                    rows.append(row)
                except Exception as e:
                    rows.append({
                        "source": "deterministic",
                        "parent_id": f"det:{variant}:{short_span}:{long_span}",
                        "family": deterministic_family_for_mutation(variant)[0],
                        "base_family": "ewm",
                        "mutation_type": variant,
                        "short_span": short_span,
                        "long_span": long_span,
                        "params": normalize_aux_params(aux),
                        "error": str(e),
                    })
    rows = sorted(rows, key=lambda r: (not r.get("robust_ok", False), -r.get("score", -999)))
    _atomic_write_json(DETERMINISTIC_FILE, rows)
    return rows


def _cols_present(df, cols):
    return [c for c in cols if c in df.columns]


if RUN_BASELINE_SWEEP:
    BASELINE_RESULTS = run_baseline_sweep()
    baseline_df = pd.DataFrame(BASELINE_RESULTS)
    display(baseline_df.head(12))
    print("baseline summary:\n" + baseline_df.head(12).to_string(index=False))
else:
    print("baseline sweep paused")

if RUN_DETERMINISTIC_SEARCH:
    DETERMINISTIC_RESULTS = run_deterministic_search()
    deterministic_df = pd.DataFrame([r for r in DETERMINISTIC_RESULTS if r.get("score") is not None])
    if not deterministic_df.empty:
        raw_cols = [
            "score", "robustness_score", "train_sharpe", "wf_median", "wf_min", "beta", "turnover",
            "family", "base_family", "mutation_type", "short_span", "long_span",
            "cluster_id", "signature", "params", "model_size_key", "robust_ok",
        ]
        display(deterministic_df.sort_values("score", ascending=False)[_cols_present(deterministic_df, raw_cols)].head(12))
        frontier_df = deterministic_df[deterministic_df["robust_ok"]].sort_values("score", ascending=False)
        if not frontier_df.empty:
            frontier_cols = [
                "score", "robustness_score", "train_sharpe", "wf_median", "wf_min", "beta", "turnover",
                "family", "mutation_type", "short_span", "long_span", "cluster_id", "signature", "params", "model_size_key",
            ]
            frontier_cols = _cols_present(frontier_df, frontier_cols)
            display(frontier_df[frontier_cols].head(12))
            print("frontier after filters:\n" + frontier_df[frontier_cols].head(12).to_string(index=False))
        else:
            print("frontier after filters: no deterministic rows passed robust gates")
    else:
        print("deterministic search produced no valid rows")
else:
    print("deterministic search paused")

if RUN_PARAM_SEARCH:
    PARAM_SEARCH_RESULTS = run_parameter_search()
    parameter_df = pd.DataFrame([r for r in PARAM_SEARCH_RESULTS if r.get("score") is not None])
    if EVOLVE_ENABLED and EVOLUTION_SUMMARY_FILE.exists():
        evo_summary = json.loads(EVOLUTION_SUMMARY_FILE.read_text())
        gens = evo_summary.get("generations", [])
        best_evo = evo_summary.get("best_score")
        best_evo_txt = f"{best_evo:+.2f}" if isinstance(best_evo, (int, float)) else "n/a"
        print(
            f"evolution: gens={evo_summary.get('generations_executed', 0)} "
            f"stop={evo_summary.get('stop_reason', 'n/a')} "
            f"best={best_evo_txt} evals={evo_summary.get('total_evals', 0)}"
        )
        for gg in gens:
            gg_best = gg.get("best_score")
            gg_best_txt = f"{gg_best:+.2f}" if isinstance(gg_best, (int, float)) else "n/a"
            print(
                f"  g{gg.get('gen_id')}: trials={gg.get('planned_trials')} valid={gg.get('valid_count')} "
                f"valid_rate={gg.get('valid_rate', 0.0):.2%} robust={gg.get('robust_count')} "
                f"zero_streak={gg.get('zero_robust_streak', 0)} best={gg_best_txt} "
                f"best_gain={gg.get('best_gain', 0.0):+.3f} median_gain={gg.get('median_gain', 0.0):+.3f}"
            )
    if not parameter_df.empty:
        param_cols = [
            "score", "robustness_score", "train_sharpe", "wf_median", "wf_min", "beta", "beta_raw",
            "beta_reduction", "beta_gate_status", "turnover",
            "family", "mutation_type", "short_span", "long_span", "cluster_id", "signature", "params", "model_size_key", "robust_ok",
        ]
        param_cols = _cols_present(parameter_df, param_cols)
        display(parameter_df.sort_values(["robust_ok", "score"], ascending=[False, False])[param_cols].head(12))
        print("parameter search frontier:\n" + parameter_df.sort_values(["robust_ok", "score"], ascending=[False, False])[param_cols].head(12).to_string(index=False))
    else:
        print("parameter search produced no valid rows")
else:
    print("parameter search paused")

## Cell 9 â€” Research log + reflection memo

Append-only JSONL log. The log is the model's **memory**: it sees its own
past experiments (ranked by Sharpe) plus recent failures. The reflection
memo is a separate file updated every few batches.


In [ ]:
import threading as _threading
_LOG_LOCK = _threading.Lock()

def append_log(entry):
    with _LOG_LOCK:
        with open(RESEARCH_LOG, "a") as f:
            f.write(json.dumps(entry, default=str) + "\n")
    try:
        if "wandb_log_candidate" in globals():
            wandb_log_candidate(entry)
    except Exception:
        pass

def load_log():
    if not RESEARCH_LOG.exists():
        return []
    out = []
    for line in open(RESEARCH_LOG):
        line = line.strip()
        if not line:
            continue
        try:
            out.append(json.loads(line))
        except Exception:
            pass
    return out

def load_deterministic():
    if not DETERMINISTIC_FILE.exists():
        return []
    try:
        return json.loads(DETERMINISTIC_FILE.read_text())
    except Exception:
        return []

def load_parameter_search():
    if not PARAM_SEARCH_FILE.exists():
        return []
    try:
        return json.loads(PARAM_SEARCH_FILE.read_text())
    except Exception:
        return []

def load_search_results():
    rows = []
    rows.extend(load_deterministic())
    rows.extend(load_parameter_search())
    return rows

def save_memo(text):
    MEMO_FILE.write_text(text)

def load_memo():
    if MEMO_FILE.exists():
        return MEMO_FILE.read_text()
    return "(No memo yet - deterministic-first mode.)"

def eligible_entry(entry):
    if entry.get("score") is None:
        return False
    return shortlist_ok(entry.get("beta", 0.0), entry.get("turnover", 0.0))

def family_saturation(log):
    good = [e for e in log if e.get("sharpe") is not None]
    counts = {f: 0 for f in VALID_FAMILIES}
    for e in good:
        for fam in classify_signal(e):
            counts[fam] = counts.get(fam, 0) + 1
    return counts

def cluster_summary(rows, top_k=6):
    good = [e for e in rows if e.get("score") is not None]
    if not good:
        return []
    groups = {}
    for e in good:
        cid = e.get("cluster_id", "unknown")
        groups.setdefault(cid, []).append(e)
    out = []
    for cid, items in groups.items():
        best = max(items, key=lambda e: e["score"])
        out.append({
            "cluster_id": cid,
            "count": len(items),
            "best_score": best["score"],
            "best_sharpe": best.get("sharpe", best.get("train_sharpe", 0.0)),
            "family": best.get("family", "?"),
            "mutation_type": best.get("mutation_type", "?"),
        })
    return sorted(out, key=lambda r: -r["best_score"])[:top_k]

def winner_anchor_summary(log):
    det = [r for r in load_search_results() if r.get("robust_ok")]
    if det:
        best = max(det, key=lambda r: r["score"])
        return f"{best['cluster_id']}  score={best['score']:+.2f}  trainSh={best['train_sharpe']:+.2f}  mut={best['mutation_type']}  source={best['parent_id']}"
    rows = cluster_summary(log, top_k=1)
    if rows:
        r = rows[0]
        return f"{r['cluster_id']}  bestScore={r['best_score']:+.2f}  trainSh={r['best_sharpe']:+.2f}  count={r['count']}  mut={r['mutation_type']}"
    baselines = load_baselines()
    if not baselines:
        return "(no winner anchor yet)"
    best = next((r for r in baselines if r.get("family") == "ewm"), baselines[0])
    return f"{best['cluster_id']}  score={best['score']:+.2f}  trainSh={best['train_sharpe']:+.2f}  source={best['parent_id']}"

def log_summary_for_prompt(log, top_k=8, code_lines=12):
    good = [e for e in log if e.get("score") is not None]
    if not good:
        return "(empty - deterministic-first mode, no LLM runs yet)"
    top = sorted(good, key=lambda e: -e["score"])[:top_k]
    lines = []
    for i, e in enumerate(top):
        lines.append(
            f"#{i+1}  id=log:{e['iter']}  score={e['score']:+.2f}  trainSh={e['sharpe']:+.2f}  "
            f"wf={e.get('wf_median', 0):+.2f}/{e.get('wf_min', 0):+.2f}  beta={e.get('beta', 0):+.2f}  "
            f"to={e.get('turnover', 0):.2f}  fam={e.get('family', '?')}  mut={e.get('mutation_type', '?')}  cluster={e.get('cluster_id', '?')}"
        )
        if e.get("parent_id"):
            lines.append(f"    PARENT: {e['parent_id']}")
        if e.get("hypothesis"):
            lines.append(f"    HYP: {e['hypothesis'][:150]}")
        code_snip = "\n".join("    " + ln for ln in e["code"].splitlines()[:code_lines])
        lines.append(code_snip)
        if len(e["code"].splitlines()) > code_lines:
            lines.append("    ...")
        lines.append("")
    return "\n".join(lines)

def failures_for_prompt(log, n_reject=6, n_runtime=3):
    fails = [e for e in log if e.get("error")]
    chosen = fails[-(n_reject + n_runtime):]
    if not chosen:
        return ""
    lines = ["\n## Recent failures (avoid these mistakes)"]
    for e in chosen:
        lines.append(f"- iter {e.get('iter', '?')}: {str(e['error'])[:180]}")
        if e.get("cluster_id"):
            lines.append(f"    cluster: {e['cluster_id']}")
    return "\n".join(lines)

def load_baselines():
    if not BASELINE_FILE.exists():
        return []
    try:
        return json.loads(BASELINE_FILE.read_text())
    except Exception:
        return []

def baseline_summary_for_prompt(top_k=8):
    rows = load_baselines()
    if not rows:
        return "(no deterministic baselines yet)"
    top = sorted(rows, key=lambda r: -(r["score"] + 0.03 * r.get("priority_weight", 0)))[:top_k]
    lines = []
    for i, r in enumerate(top):
        n2 = "-" if r.get("n2") is None else r.get("n2")
        lines.append(
            f"#{i+1}  id={r['parent_id']}  cluster={r.get('cluster_id', '?')}  w={r.get('priority_weight', 1.0):.1f}  "
            f"n1={r['n1']}  n2={n2}  score={r['score']:+.2f}  trainSh={r['train_sharpe']:+.2f}  "
            f"wf={r.get('wf_median', 0):+.2f}/{r.get('wf_min', 0):+.2f}  beta={r['beta']:+.2f}  to={r.get('turnover', 0):.2f}"
        )
    return "\n".join(lines)

def build_parent_pool(log, top_logs=3, top_baselines=5, top_det=8):
    det = [r for r in load_search_results() if r.get("robust_ok")]
    det_rows = sorted(det, key=lambda r: -r["score"])[:top_det]
    baselines = [r for r in load_baselines() if r.get("shortlist_ok")]
    base_rows = sorted(baselines, key=lambda r: -(r["score"] + 0.03 * r.get("priority_weight", 0)))[:top_baselines]
    good_logs = [e for e in log if eligible_entry(e)]
    log_rows = sorted(good_logs, key=lambda e: -e["score"])[:top_logs]
    parents = []
    for r in det_rows:
        parents.append({
            "parent_id": r["parent_id"],
            "family": r["family"],
            "priority_weight": 4.0,
            "cluster_id": r.get("cluster_id"),
            "summary": f"{r['parent_id']}  cluster={r.get('cluster_id', '?')}  fam={r['family']}  w=4.0  score={r['score']:+.2f}  trainSh={r['train_sharpe']:+.2f}  wf={r.get('wf_median', 0):+.2f}/{r.get('wf_min', 0):+.2f}",
        })
    for r in base_rows:
        n2 = "-" if r.get("n2") is None else r.get("n2")
        parents.append({
            "parent_id": r["parent_id"],
            "family": r["family"],
            "priority_weight": r.get("priority_weight", 1.0),
            "cluster_id": r.get("cluster_id"),
            "summary": f"{r['parent_id']}  cluster={r.get('cluster_id', '?')}  fam={r['family']}  params=({r['n1']},{n2})  w={r.get('priority_weight', 1.0):.1f}  score={r['score']:+.2f}  trainSh={r['train_sharpe']:+.2f}",
        })
    for e in log_rows:
        parents.append({
            "parent_id": f"log:{e['iter']}",
            "family": e.get("family", "?"),
            "priority_weight": 2.0,
            "cluster_id": e.get("cluster_id"),
            "summary": f"log:{e['iter']}  cluster={e.get('cluster_id', '?')}  fam={e.get('family', '?')}  score={e['score']:+.2f}  trainSh={e['sharpe']:+.2f}  mut={e.get('mutation_type', '?')}",
        })
    return parents

def parent_summary_for_prompt(log):
    parents = build_parent_pool(log)
    if not parents:
        return "(no shortlisted parents yet; prioritise deterministic survivors)"
    return "\n".join(f"- {p['summary']}" for p in parents)

def mutation_success_priors(log):
    stats = {m: {"n": 0, "robust": 0} for m in VALID_MUTATIONS}
    for e in log:
        mut = normalize_mutation(e.get("mutation_type", "span_tweak"))
        if mut not in stats:
            stats[mut] = {"n": 0, "robust": 0}
        if e.get("sharpe") is None:
            continue
        stats[mut]["n"] += 1
        if e.get("robust_ok"):
            stats[mut]["robust"] += 1
    out = {}
    for mut, s in stats.items():
        out[mut] = (s["robust"] + 1.0) / (s["n"] + 2.0)
    return out

def mutation_priors_text(log):
    pri = mutation_success_priors(log)
    parts = [f"{k}:{pri.get(k, 0.5):.2f}" for k in sorted(pri)]
    return " | ".join(parts)

def expert_success_stats(log):
    stats = {}
    for e in log:
        expert = e.get("mutation_type")
        if not expert:
            continue
        s = stats.setdefault(expert, {"n": 0, "robust": 0})
        if e.get("sharpe") is None:
            continue
        s["n"] += 1
        if e.get("robust_ok"):
            s["robust"] += 1
    return stats

def expert_budget_text(log):
    stats = expert_success_stats(log)
    if not stats:
        return "(cold start: equal budget per mutation)"
    lines = []
    for expert in sorted(stats):
        n = stats[expert]["n"]
        robust = stats[expert]["robust"]
        rate = robust / max(n, 1)
        lines.append(f"{expert}: robust={robust}/{n} ({rate:.0%})")
    return " | ".join(lines)


## Cell 10 â€” The AutoResearch loop

Each batch:
1. Build prompt from research log + reflection memo + failures
2. Generate 3 candidates via LLM
3. Sandbox â†’ validate â†’ lookahead check â†’ backtest each
4. Append results to log

Every `REFLECT_EVERY` batches the model reviews its full log and writes
a **research memo** â€” what's working, what's not, what to try next.
This memo is included in all future prompts: the model learns from itself.


In [ ]:
def parameter_search_summary(rows):
    good = [r for r in rows if isinstance(r, dict) and r.get("score") is not None]
    robust = [r for r in good if r.get("robust_ok")]
    if not good:
        return "(parameter search has not produced any valid rows)"
    best = max(good, key=lambda r: r.get("score", -1e99))
    if "row_identity" in globals():
        best_id = row_identity(best, "parameter_search")
    else:
        best_id = (
            best.get("parent_id")
            or best.get("signature")
            or best.get("cluster_id")
            or best.get("program_hash")
            or "parameter_search:unknown"
        )
    score = float(best.get("score", 0.0) or 0.0)
    train_sharpe = float(best.get("train_sharpe", best.get("sharpe", 0.0)) or 0.0)
    wf_median = float(best.get("wf_median", 0.0) or 0.0)
    wf_min = float(best.get("wf_min", 0.0) or 0.0)
    lines = [
        f"rows={len(rows)} valid={len(good)} robust={len(robust)}",
        f"best={best_id} score={score:+.2f} trainSh={train_sharpe:+.2f} wf={wf_median:+.2f}/{wf_min:+.2f}",
    ]
    return "\n".join(lines)

PARAMETER_SEARCH_RESULTS = load_parameter_search()
print("parameter search snapshot:\n" + parameter_search_summary(PARAMETER_SEARCH_RESULTS))
if 'EVOLUTION_SUMMARY_FILE' in globals() and EVOLUTION_SUMMARY_FILE.exists():
    evo = json.loads(EVOLUTION_SUMMARY_FILE.read_text())
    print(
        f"evolution snapshot: gens={evo.get('generations_executed', 0)} "
        f"stop={evo.get('stop_reason', 'n/a')} best={evo.get('best_score', 'n/a')} evals={evo.get('total_evals', 0)}"
    )


## Cell 11 â€” Held-out evaluation (2023â€“2024)

Take the top-K train signals and re-run on untouched test data.
Walk-forward: slice held-out into quarters and check consistency.


In [ ]:
TOP_K = 5
HELDOUT_SHORTLIST_K = 20
HELDOUT_MAX_PER_CLUSTER = 2
HELDOUT_MIN_DETERMINISTIC = 5
HELDOUT_MIN_EVOLUTION = 10
HELDOUT_MIN_EVOLUTION_CHAMPIONS = 5
HELDOUT_MAX_PER_MUTATION = 4
COST_STRESS_BPS = (5.0, 10.0, 15.0)
ENSEMBLE_TOP_N = 3
ECONOMIC_SHARPE_FLOOR = globals().get("ECONOMIC_SHARPE_FLOOR", 0.50)
ECONOMIC_EDGE_OVER_DETERMINISTIC = globals().get("ECONOMIC_EDGE_OVER_DETERMINISTIC", 0.05)
APPROVED_WINNER_EDGE_OVER_DETERMINISTIC = globals().get("APPROVED_WINNER_EDGE_OVER_DETERMINISTIC", 0.10)
CHRONOLOGICAL_HOLDOUT_SEGMENTS = max(0, int(globals().get("CHRONOLOGICAL_HOLDOUT_SEGMENTS", 3) or 0))
ROLLING_HOLDOUT_SEGMENTS = max(0, int(globals().get("ROLLING_HOLDOUT_SEGMENTS", 3) or 0))
ROLLING_HOLDOUT_MIN_POINTS = max(20, int(globals().get("ROLLING_HOLDOUT_MIN_POINTS", 126) or 126))
SUBPERIOD_WINDOWS = (
    ("2015-2018", "2015-01-01", "2018-12-31"),
    ("2019-2021", "2019-01-01", "2021-12-31"),
    ("2022-2024", "2022-01-01", "2024-12-31"),
)
EVOLUTION_DEEP_DIVE_FILE = OUT / "evolution_deep_dive.md"
EVOLUTION_TLDR_FILE = OUT / "evolution_tldr.md"


def _float_or(value, default=0.0):
    try:
        if value is None:
            return default
        return float(value)
    except Exception:
        return default


def _safe_json_value(path, default):
    if path is None:
        return default
    try:
        if (not path.exists()) or path.stat().st_size == 0:
            return default
        text = path.read_text()
        if not text.strip():
            return default
        value = json.loads(text)
        return default if value is None else value
    except Exception:
        return default


def _as_row_list(value):
    if isinstance(value, list):
        return [r for r in value if isinstance(r, dict)]
    if isinstance(value, dict):
        for key in ("rows", "results", "candidates", "lineage", "programs"):
            rows = value.get(key)
            if isinstance(rows, list):
                return [r for r in rows if isinstance(r, dict)]
        return [value] if value.get("score") is not None else []
    return []


def _safe_json_rows(path):
    return _as_row_list(_safe_json_value(path, []))


def _safe_json_dict(path):
    value = _safe_json_value(path, {})
    return value if isinstance(value, dict) else {}


def _runtime_scope_metadata():
    path = globals().get("RUNTIME_METADATA_FILE")
    return _safe_json_dict(path) if path is not None else {}


def _runtime_family_scope_lists():
    runtime_metadata = _runtime_scope_metadata()
    configured = runtime_metadata.get("configured_method_families")
    if not isinstance(configured, list) or not configured:
        configured = list(globals().get("ROADMAP_METHOD_FAMILIES", globals().get("BENCHMARK_METHOD_FAMILIES", [])))
    executed = runtime_metadata.get("executed_method_families")
    if not isinstance(executed, list) or not executed:
        executed = list(globals().get("EXECUTED_METHOD_FAMILIES", []))
    configured = [str(x) for x in configured]
    executed = [str(x) for x in executed]
    deferred = [family for family in configured if family not in executed]
    active_sources = runtime_metadata.get("active_result_sources")
    if not isinstance(active_sources, list) or not active_sources:
        active_sources = list(globals().get("ACTIVE_RESULT_SOURCES", ["deterministic", "parameter_search_evolution"]))
    active_scope = runtime_metadata.get("active_execution_scope") or globals().get(
        "ACTIVE_EXECUTION_SCOPE",
        "deterministic/classical/autoresearch only",
    )
    return {
        "configured": configured,
        "executed": executed,
        "deferred": deferred,
        "active_sources": [str(x) for x in active_sources],
        "active_scope": str(active_scope),
    }


def _row_family_tokens(row):
    if not isinstance(row, dict):
        return set()
    keys = (
        "source",
        "family",
        "base_family",
        "benchmark_family",
        "model_family",
        "study_family",
        "study_type",
        "label",
        "name",
    )
    tokens = set()
    for key in keys:
        value = row.get(key)
        if value in (None, ""):
            continue
        text = str(value).strip().lower()
        if text:
            tokens.add(text)
    return tokens


def _row_matches_deferred_stage_family(row):
    deferred_aliases = {
        "lstm",
        "lstm_sharpe",
        "tabular",
        "tabular_ml",
        "gbt",
        "gradient_boosted_trees",
        "combination",
        "combination_studies",
        "ensemble_combination",
    }
    tokens = _row_family_tokens(row)
    return any(alias in token for token in tokens for alias in deferred_aliases)


def _partial_report_path():
    path = globals().get("PARTIAL_REPORT_FILE")
    if path is None and "OUT" in globals():
        path = OUT / "partial_run_report.json"
    return path


def _load_partial_run_report():
    return _safe_json_dict(_partial_report_path())


def _row_identity_safe(row, default_prefix="row"):
    helper = globals().get("row_identity")
    if callable(helper):
        try:
            return str(helper(row, default_prefix=default_prefix))
        except TypeError:
            try:
                return str(helper(row, default_prefix))
            except Exception:
                pass
        except Exception:
            pass
    if not isinstance(row, dict):
        return f"{default_prefix}:unknown"
    for key in ("parent_id", "iter", "program_hash", "program_sig", "signature", "cluster_id"):
        value = row.get(key)
        if value not in (None, ""):
            return str(value)
    return f"{default_prefix}:unknown"


def _heldout_iter_label(row):
    if not isinstance(row, dict):
        return "row:unknown"
    source = row.get("source", "unknown")
    signature = str(row.get("signature") or row.get("program_hash") or row.get("program_sig") or "nosig")[:8]
    cluster_id = str(row.get("cluster_id", "unknown"))
    if source == "parameter_search_evolution":
        return f"evo:{cluster_id}:{signature}"
    if source == "deterministic":
        return str(row.get("parent_id") or _row_identity_safe(row, default_prefix="det"))
    return _row_identity_safe(row, default_prefix=str(source or default_prefix))


def _selection_score_safe(sharpe, wf_median, consistency, beta, avg_turnover, wf_min=0.0, max_dd=0.0):
    try:
        return selection_score(
            sharpe,
            wf_median,
            consistency,
            beta,
            avg_turnover,
            wf_min=wf_min,
            max_dd=max_dd,
        )
    except TypeError:
        return selection_score(sharpe, wf_median, consistency, beta, avg_turnover)


def _robustness_score_safe(metric_row):
    helper = globals().get("robustness_score")
    if callable(helper):
        try:
            return helper(metric_row)
        except Exception:
            return None
    return None


def _median_value(values):
    vals = sorted([float(v) for v in values if v is not None])
    if not vals:
        return None
    mid = len(vals) // 2
    if len(vals) % 2:
        return vals[mid]
    return 0.5 * (vals[mid - 1] + vals[mid])


def _window_median_sharpe(windows):
    if not isinstance(windows, list):
        return None
    return _median_value([_float_or(w.get("sharpe"), None) for w in windows if isinstance(w, dict)])


def _cost_stress_avg_sharpe(row):
    stress = row.get("cost_stress", {}) if isinstance(row, dict) else {}
    vals = []
    if isinstance(stress, dict):
        for payload in stress.values():
            if isinstance(payload, dict):
                vals.append(_float_or(payload.get("sharpe"), None))
    vals = [v for v in vals if v is not None]
    if not vals:
        return None
    return float(sum(vals) / len(vals))


def _build_chronological_window_specs(index_like, segments, min_points=60):
    index = list(index_like) if index_like is not None else []
    n = len(index)
    if segments <= 0 or n < min_points:
        return []
    window_len = max(n // segments, min_points)
    if window_len > n:
        return []
    specs = []
    lo = 0
    slot = 1
    while lo < n and len(specs) < segments:
        hi = n if len(specs) == segments - 1 else min(n, lo + window_len)
        if hi - lo < min_points:
            break
        specs.append(
            {
                "label": f"chrono_{slot}",
                "lo": lo,
                "hi": hi,
                "start": str(index[lo])[:10],
                "end": str(index[hi - 1])[:10],
            }
        )
        lo = hi
        slot += 1
    return specs


def _build_rolling_window_specs(index_like, segments, min_points=126):
    index = list(index_like) if index_like is not None else []
    n = len(index)
    if segments <= 0 or n < min_points:
        return []
    window_len = max(min_points, n // max(segments, 1))
    if window_len > n:
        return []
    max_start = max(n - window_len, 0)
    if segments == 1:
        starts = [max_start]
    else:
        stride = max(max_start // max(segments - 1, 1), 1)
        starts = []
        for i in range(segments):
            starts.append(min(i * stride, max_start))
        starts = sorted(set(starts))
        if starts[-1] != max_start:
            starts.append(max_start)
    specs = []
    for idx, lo in enumerate(starts, start=1):
        hi = min(n, lo + window_len)
        if hi - lo < min_points:
            continue
        specs.append(
            {
                "label": f"roll_{idx}",
                "lo": lo,
                "hi": hi,
                "start": str(index[lo])[:10],
                "end": str(index[hi - 1])[:10],
            }
        )
        if len(specs) >= segments:
            break
    return specs


def _supporting_diagnostic_report(candidate, baseline):
    diagnostics = []
    if not isinstance(candidate, dict) or not isinstance(baseline, dict):
        return {"diagnostics": diagnostics, "wins": 0, "losses": 0, "ties": 0, "comparable": 0, "not_losing_majority": False}

    metric_specs = (
        ("test_score", "composite score", _float_or(candidate.get("test_score"), None), _float_or(baseline.get("test_score"), None)),
        ("test_sharpe", "test Sharpe", _float_or(candidate.get("test_sharpe"), None), _float_or(baseline.get("test_sharpe"), None)),
        ("wf_median", "WF median", _float_or(candidate.get("wf_median"), None), _float_or(baseline.get("wf_median"), None)),
        ("wf_min", "WF min", _float_or(candidate.get("wf_min"), None), _float_or(baseline.get("wf_min"), None)),
        (
            "test_robustness_score",
            "robustness score",
            _float_or(candidate.get("test_robustness_score", candidate.get("train_robustness_score")), None),
            _float_or(baseline.get("test_robustness_score", baseline.get("train_robustness_score")), None),
        ),
        (
            "test_benchmark_spread_sharpe",
            "benchmark-spread Sharpe",
            _float_or(candidate.get("test_benchmark_spread_sharpe"), None),
            _float_or(baseline.get("test_benchmark_spread_sharpe"), None),
        ),
        ("cost_stress_avg_sharpe", "cost-stress avg Sharpe", _cost_stress_avg_sharpe(candidate), _cost_stress_avg_sharpe(baseline)),
        ("subperiod_median_sharpe", "subperiod median Sharpe", _window_median_sharpe(candidate.get("subperiods")), _window_median_sharpe(baseline.get("subperiods"))),
        (
            "chronological_median_sharpe",
            "chronological holdout median Sharpe",
            _window_median_sharpe(candidate.get("chronological_holdout")),
            _window_median_sharpe(baseline.get("chronological_holdout")),
        ),
        (
            "rolling_median_sharpe",
            "rolling holdout median Sharpe",
            _window_median_sharpe(candidate.get("rolling_holdout")),
            _window_median_sharpe(baseline.get("rolling_holdout")),
        ),
    )

    wins = 0
    losses = 0
    ties = 0
    for key, label, candidate_value, baseline_value in metric_specs:
        if candidate_value is None or baseline_value is None:
            continue
        delta = float(candidate_value - baseline_value)
        if abs(delta) <= 1e-9:
            verdict = "tie"
            ties += 1
        elif delta > 0:
            verdict = "win"
            wins += 1
        else:
            verdict = "loss"
            losses += 1
        diagnostics.append(
            {
                "key": key,
                "label": label,
                "candidate": float(candidate_value),
                "baseline": float(baseline_value),
                "delta": delta,
                "verdict": verdict,
            }
        )

    comparable = wins + losses + ties
    return {
        "diagnostics": diagnostics,
        "wins": wins,
        "losses": losses,
        "ties": ties,
        "comparable": comparable,
        "not_losing_majority": bool(comparable and losses * 2 <= comparable),
    }


def _heldout_rule_reason_lines(heldout_report):
    if not isinstance(heldout_report, dict):
        return ["held-out winner rule not evaluated"]
    reasons = []
    leader = heldout_report.get("leader")
    baseline = heldout_report.get("best_deterministic")
    comparison = heldout_report.get("comparison", {})
    if leader is None:
        return ["held-out evaluation produced no leader"]
    if baseline is None:
        reasons.append("no deterministic baseline available for held-out comparison")
    elif leader.get("source") == "deterministic":
        reasons.append("highest composite-score candidate is still the deterministic baseline")
    edge = heldout_report.get("score_edge_vs_deterministic")
    required_edge = heldout_report.get("required_edge", APPROVED_WINNER_EDGE_OVER_DETERMINISTIC)
    if edge is None:
        reasons.append("score edge vs deterministic baseline unavailable")
    elif edge < required_edge:
        reasons.append(f"composite-score edge {edge:+.2f} is below required {required_edge:+.2f}")
    if not comparison.get("not_losing_majority"):
        reasons.append(
            "supporting diagnostics lost majority "
            f"({comparison.get('losses', 0)} losses across {comparison.get('comparable', 0)} comparable diagnostics)"
        )
    return reasons or ["winner rule met"]


def _build_heldout_report(test_rows):
    rows = [r for r in test_rows if isinstance(r, dict)]
    ranked = sorted(rows, key=lambda r: -_float_or(r.get("test_score"), -999.0))
    leader = ranked[0] if ranked else None
    best_deterministic = max(
        [r for r in rows if r.get("source") == "deterministic"],
        key=lambda r: _float_or(r.get("test_score"), -999.0),
        default=None,
    )
    best_evolution = max(
        [r for r in rows if r.get("source") == "parameter_search_evolution"],
        key=lambda r: _float_or(r.get("test_score"), -999.0),
        default=None,
    )
    score_edge = None
    if leader and best_deterministic:
        score_edge = float(_float_or(leader.get("test_score")) - _float_or(best_deterministic.get("test_score")))
    comparison = _supporting_diagnostic_report(leader, best_deterministic)
    edge_ok = bool(best_deterministic and score_edge is not None and score_edge >= APPROVED_WINNER_EDGE_OVER_DETERMINISTIC)
    source_ok = bool(leader and leader.get("source") != "deterministic")
    diagnostics_ok = bool(comparison.get("not_losing_majority"))
    winner = leader if leader and source_ok and edge_ok and diagnostics_ok else None
    return {
        "ranked": ranked,
        "leader": leader,
        "winner": winner,
        "best_deterministic": best_deterministic,
        "best_evolution": best_evolution,
        "score_edge_vs_deterministic": score_edge,
        "required_edge": APPROVED_WINNER_EDGE_OVER_DETERMINISTIC,
        "comparison": comparison,
        "reasons": _heldout_rule_reason_lines(
            {
                "leader": leader,
                "best_deterministic": best_deterministic,
                "comparison": comparison,
                "score_edge_vs_deterministic": score_edge,
                "required_edge": APPROVED_WINNER_EDGE_OVER_DETERMINISTIC,
            }
        ),
        "status": "approved_winner" if winner else "no_winner_yet",
    }


def walk_forward(code_str, close_df, volume_df, flipped=False, n_windows=WF_WINDOWS):
    N = len(close_df)
    sz = max(N // n_windows, 1)
    per_window = []
    for w in range(n_windows):
        lo = w * sz
        hi = (w + 1) * sz if w < n_windows - 1 else N
        sub_c, sub_v = close_df.iloc[lo:hi], volume_df.iloc[lo:hi]
        sig, err = run_signal_code(code_str, sub_c, sub_v, vix_s=vix_test.iloc[lo:hi] if vix_test is not None else None, tnx_s=tnx_test.iloc[lo:hi] if tnx_test is not None else None, timeout=20)
        if err:
            per_window.append({"window": w, "error": err})
            continue
        if flipped:
            sig = -sig
        m = backtest(sig, sub_c)
        per_window.append({"window": w, "sharpe": m.get("sharpe", 0.0), "benchmark_spread_sharpe": m.get("benchmark_spread_sharpe", m.get("excess_sharpe", 0.0)), "ann_return": m.get("ann_return", 0.0), "beta": m.get("beta", 0.0), "max_dd": m.get("max_dd", 0.0), "turnover": m.get("avg_turnover", 0.0)})
    return per_window


def deterministic_code(row):
    if row.get("code"):
        return row["code"]
    short_span = row["short_span"]
    long_span = row["long_span"]
    variant = row["mutation_type"]
    params = normalize_aux_params(row.get("params", {}))
    vol_window = int(params.get("vol_window", 20))
    vol_gate_window = int(params.get("vol_gate_window", 20))
    vol_cap = float(params.get("vol_cap", 3.0))
    vix_threshold = float(params.get("vix_threshold", 24.0))
    if variant == "plain":
        return f"""def signal(close, volume, vix=None, tnx=None):
    fast = close.ewm(span={short_span}, min_periods={short_span}).mean().shift(1)
    slow = close.ewm(span={long_span}, min_periods={long_span}).mean().shift(1)
    core = fast / slow - 1.0
    out = (core.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant == "rank_norm":
        return f"""def signal(close, volume, vix=None, tnx=None):
    fast = close.ewm(span={short_span}, min_periods={short_span}).mean().shift(1)
    slow = close.ewm(span={long_span}, min_periods={long_span}).mean().shift(1)
    core = fast / slow - 1.0
    out = (core.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant == "vol_scale":
        return f"""def signal(close, volume, vix=None, tnx=None):
    fast = close.ewm(span={short_span}, min_periods={short_span}).mean().shift(1)
    slow = close.ewm(span={long_span}, min_periods={long_span}).mean().shift(1)
    core = fast / slow - 1.0
    vol = close.pct_change().rolling({vol_window}).std().replace(0, np.nan)
    scaled = core.div(vol.clip(lower=1e-6), fill_value=0.0).clip(-3.0, 3.0)
    out = (scaled.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant == "volume_gate":
        return f"""def signal(close, volume, vix=None, tnx=None):
    fast = close.ewm(span={short_span}, min_periods={short_span}).mean().shift(1)
    slow = close.ewm(span={long_span}, min_periods={long_span}).mean().shift(1)
    core = fast / slow - 1.0
    vratio = volume / volume.rolling({vol_gate_window}).mean()
    gated = core * vratio.clip(lower=0.5, upper=1.5)
    out = (gated.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant == "regime_gate":
        return f"""def signal(close, volume, vix=None, tnx=None):
    fast = close.ewm(span={short_span}, min_periods={short_span}).mean().shift(1)
    slow = close.ewm(span={long_span}, min_periods={long_span}).mean().shift(1)
    core = fast / slow - 1.0
    regime = 1.0 if vix is None else (vix.rolling(5).mean() < {vix_threshold}).astype(float).values[:, None]
    gated = core * regime
    out = (gated.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant == "ts_momentum":
        return f"""def signal(close, volume, vix=None, tnx=None):
    trend = close.pct_change({long_span})
    out = (trend.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant == "short_reversal":
        return f"""def signal(close, volume, vix=None, tnx=None):
    reversal = -close.pct_change({short_span})
    out = (reversal.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant in ("vol_adjusted", "vol_adjusted_momentum"):
        return f"""def signal(close, volume, vix=None, tnx=None):
    trend = close.pct_change({long_span})
    vol = close.pct_change().rolling({vol_window}).std().replace(0, np.nan)
    scaled = trend.div(vol.clip(lower=1e-6), fill_value=0.0).clip(-{vol_cap}, {vol_cap})
    out = (scaled.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant in ("volume_confirm", "volume_confirmed_momentum"):
        return f"""def signal(close, volume, vix=None, tnx=None):
    trend = close.pct_change({long_span})
    vratio = volume / volume.rolling({vol_gate_window}).mean()
    confirmed = trend * vratio.clip(lower=0.5, upper=1.8)
    out = (confirmed.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant == "regime_momentum":
        return f"""def signal(close, volume, vix=None, tnx=None):
    trend_fast = close.pct_change({short_span})
    trend_slow = close.pct_change({long_span})
    trend = trend_fast - trend_slow
    if vix is not None:
        regime = (vix.rolling(5).mean() < {vix_threshold}).astype(float).values[:, None]
        trend = trend * regime
    out = (trend.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant == "multi_factor":
        return f"""def signal(close, volume, vix=None, tnx=None):
    trend = close.pct_change({long_span}).rank(axis=1, pct=True) - 0.5
    reversal = (-close.pct_change({short_span})).rank(axis=1, pct=True) - 0.5
    vol = close.pct_change().rolling({vol_window}).std().replace(0, np.nan)
    low_vol = (-vol).rank(axis=1, pct=True) - 0.5
    combined = 0.55 * trend + 0.25 * reversal + 0.20 * low_vol
    out = (combined.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    raise ValueError(variant)


def _oriented_metrics(backtest_row, flipped=False):
    if flipped:
        return {
            "sharpe": backtest_row["sharpe_inverted"],
            "ann_return": backtest_row["ann_return_inverted"],
            "max_dd": backtest_row["max_dd_inverted"],
            "beta": backtest_row["beta_inverted"],
            "consistency": backtest_row["consistency_inverted"],
            "equity": backtest_row["equity_inverted"],
        }
    return {
        "sharpe": backtest_row["sharpe"],
        "ann_return": backtest_row["ann_return"],
        "max_dd": backtest_row["max_dd"],
        "beta": backtest_row["beta"],
        "consistency": backtest_row["consistency"],
        "equity": backtest_row["equity"],
    }


def _concat_optional(train_series, test_series):
    if train_series is None and test_series is None:
        return None
    if train_series is None:
        return test_series
    if test_series is None:
        return train_series
    return pd.concat([train_series, test_series]).sort_index()


def _full_test_panels():
    close_full = globals().get("close_all")
    if close_full is None:
        close_full = pd.concat([close_train, close_test]).sort_index()
    volume_full = globals().get("volume_all")
    if volume_full is None:
        volume_full = pd.concat([volume_train, volume_test]).sort_index()
    vix_full = globals().get("vix_all")
    if vix_full is None:
        vix_full = _concat_optional(globals().get("vix_train"), globals().get("vix_test"))
    tnx_full = globals().get("tnx_all")
    if tnx_full is None:
        tnx_full = _concat_optional(globals().get("tnx_train"), globals().get("tnx_test"))
    return close_full, volume_full, vix_full, tnx_full


def _oriented_signal_code(code_str, flipped=False):
    """Return executable signal code in the same orientation used during evaluation."""
    code_str = str(code_str or "")
    if not flipped:
        return code_str
    if "def signal(" not in code_str:
        return code_str
    raw_code = code_str.replace("def signal(", "def _raw_signal(", 1)
    return (
        raw_code.rstrip()
        + "\n\n\n"
        + "def signal(close, volume, vix=None, tnx=None):\n"
        + "    out = -_raw_signal(close, volume, vix=vix, tnx=tnx)\n"
        + "    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)\n"
        + "    return out\n"
    )


def _ensemble_code(rows):
    blocks = []
    calls = []
    for i, row in enumerate(rows):
        fn_code = row.get("code") or deterministic_code(row)
        if not row.get("code_is_oriented"):
            fn_code = _oriented_signal_code(fn_code, bool(row.get("flipped")))
        fn_code = fn_code.replace("def signal(", f"def _signal_{i}(", 1)
        blocks.append(fn_code)
        calls.append(f"    s{i} = _signal_{i}(close, volume, vix=vix, tnx=tnx)")
    avg_expr = " + ".join([f"s{i}" for i in range(len(rows))])
    meta_lines = "\n".join(
        [
            f"# member_{i + 1}: iter={row.get('iter')} cluster={row.get('cluster_id')} test_sc={row.get('test_score', 0.0):+.2f}"
            for i, row in enumerate(rows)
        ]
    )
    return (
        f"# objective=market_neutral_net_sharpe ensemble_top_n={len(rows)}\n"
        f"{meta_lines}\n\n"
        + "import numpy as np\n\n"
        + "\n\n".join(blocks)
        + "\n\n"
        + "def signal(close, volume, vix=None, tnx=None):\n"
        + "\n".join(calls)
        + "\n"
        + f"    out = ({avg_expr}) / {float(len(rows)):.1f}\n"
        + "    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)\n"
        + "    return out\n"
    )


def _safe_load_search_results_local():
    if "load_search_results" in globals():
        try:
            rows = _as_row_list(load_search_results())
            if rows:
                return rows
        except Exception:
            pass
    rows = []
    if "load_parameter_search" in globals():
        try:
            rows.extend(_as_row_list(load_parameter_search()))
        except Exception:
            pass
    elif "PARAM_SEARCH_FILE" in globals() and PARAM_SEARCH_FILE.exists():
        rows.extend(_safe_json_rows(PARAM_SEARCH_FILE))
    if "load_deterministic" in globals():
        try:
            rows.extend(_as_row_list(load_deterministic()))
        except Exception:
            pass
    elif "DETERMINISTIC_FILE" in globals() and DETERMINISTIC_FILE.exists():
        rows.extend(_safe_json_rows(DETERMINISTIC_FILE))
    return rows


def _rank_candidates(rows):
    candidates = [r for r in rows if r.get("score") is not None]
    robust = [r for r in candidates if r.get("robust_ok")]
    return sorted(robust if robust else candidates, key=lambda r: (not r.get("robust_ok", False), -_float_or(r.get("score"), -999.0)))


def _pick_diverse(rows, limit, seen_sig=None, per_cluster=None, per_mutation=None):
    picked = []
    seen_sig = seen_sig if seen_sig is not None else set()
    per_cluster = per_cluster if per_cluster is not None else {}
    per_mutation = per_mutation if per_mutation is not None else {}
    for row in _rank_candidates(rows):
        sig = row.get("signature")
        cid = row.get("cluster_id", "unknown")
        mutation_type = row.get("mutation_type", "unknown")
        if sig and sig in seen_sig:
            continue
        if per_cluster.get(cid, 0) >= HELDOUT_MAX_PER_CLUSTER:
            continue
        if per_mutation.get(mutation_type, 0) >= HELDOUT_MAX_PER_MUTATION:
            continue
        picked.append(row)
        if sig:
            seen_sig.add(sig)
        per_cluster[cid] = per_cluster.get(cid, 0) + 1
        per_mutation[mutation_type] = per_mutation.get(mutation_type, 0) + 1
        if len(picked) >= limit:
            break
    return picked


def _evolution_health_lines(evo_summary, partial_report=None):
    partial_report = partial_report if isinstance(partial_report, dict) else {}
    generations = evo_summary.get("generations", []) if isinstance(evo_summary, dict) else []
    generations = [g for g in generations if isinstance(g, dict)]
    eval_total = sum(int(_float_or(g.get("eval_count"), 0.0)) for g in generations)
    valid_total = sum(int(_float_or(g.get("valid_count"), 0.0)) for g in generations)
    robust_total = sum(int(_float_or(g.get("robust_count"), 0.0)) for g in generations)

    valid_rate = partial_report.get("valid_rate", partial_report.get("overall_valid_rate"))
    if valid_rate is None and eval_total:
        valid_rate = valid_total / max(eval_total, 1)

    zero_robust = partial_report.get("zero_robust_generations")
    if zero_robust is None and generations:
        zero_robust = [g.get("gen_id") for g in generations if int(_float_or(g.get("robust_count"), 0.0)) == 0]

    zero_robust_streak = partial_report.get("zero_robust_streak")
    if zero_robust_streak is None and generations:
        zero_robust_streak = 0
        for g in reversed(generations):
            if int(_float_or(g.get("robust_count"), 0.0)) == 0:
                zero_robust_streak += 1
            else:
                break

    lines = []
    if valid_rate is not None:
        try:
            rate_text = f"{float(valid_rate):.1%}"
        except Exception:
            rate_text = str(valid_rate)
        suffix = f" ({valid_total}/{eval_total} valid/evaluated)" if eval_total else ""
        lines.append(f"valid_rate={rate_text}{suffix}")
    if zero_robust is not None:
        if isinstance(zero_robust, list):
            tail = zero_robust[-5:] if zero_robust else []
            lines.append(f"zero_robust_generations={len(zero_robust)} latest={tail}")
        else:
            lines.append(f"zero_robust_generations={zero_robust}")
    if zero_robust_streak:
        lines.append(f"zero_robust_streak={zero_robust_streak}")
    if eval_total:
        lines.append(f"robust_total={robust_total}/{valid_total} valid rows")
    return lines


def _partial_report_line(partial_report):
    if not isinstance(partial_report, dict) or not partial_report:
        return None
    fields = []
    for key in (
        "status",
        "phase",
        "gen_id",
        "generation",
        "generations_executed",
        "valid_rate",
        "robust_count",
        "zero_robust_streak",
        "stop_reason",
        "updated_at",
        "timestamp",
    ):
        if key in partial_report:
            fields.append(f"{key}={partial_report.get(key)}")
    if not fields:
        fields.append("keys=" + ",".join(list(partial_report.keys())[:8]))
    return " ".join(fields)


def _write_evolution_analysis(test_rows):
    evo_summary = _safe_json_dict(globals().get("EVOLUTION_SUMMARY_FILE"))
    partial_report = _load_partial_run_report()
    if not evo_summary and not partial_report:
        return

    search_rows = [r for r in _safe_load_search_results_local() if r.get("score") is not None]
    robust_rows = [r for r in search_rows if r.get("robust_ok")]
    by_cluster = {}
    for r in robust_rows:
        cid = r.get("cluster_id", "unknown")
        prev = by_cluster.get(cid)
        if prev is None or _float_or(r.get("score"), -999.0) > _float_or(prev.get("score"), -999.0):
            by_cluster[cid] = r
    top_clusters = sorted(by_cluster.values(), key=lambda r: -_float_or(r.get("score"), -999.0))[:8]
    generations = [g for g in evo_summary.get("generations", []) if isinstance(g, dict)]
    health_lines = _evolution_health_lines(evo_summary, partial_report)
    partial_line = _partial_report_line(partial_report)

    deep = []
    deep.append("# Evolution Deep Dive\n")
    deep.append("## Run Policy")
    p = evo_summary.get("policy", {}) if isinstance(evo_summary.get("policy"), dict) else {}
    deep.append(
        f"- generations={evo_summary.get('generations_executed')} stop_reason={evo_summary.get('stop_reason')} "
        f"max_generations={p.get('max_generations')} trials_per_generation={p.get('trials_per_generation')} "
        f"beam={p.get('beam_width')} survivors={p.get('survivors')} min_gain={p.get('min_gen_growth')}"
    )
    if health_lines:
        deep.append("\n## Validity / Robustness Watch")
        deep.extend([f"- {line}" for line in health_lines])
    if partial_line:
        deep.append("\n## Partial Run Report")
        deep.append(f"- {partial_line}")
    deep.append("\n## Generation Trace")
    for rr in generations:
        deep.append(
            f"- g{rr.get('gen_id')}: valid={rr.get('valid_count')} robust={rr.get('robust_count')} "
            f"best={rr.get('best_score')} gain={rr.get('improvement_vs_prev_best')} stalled={rr.get('stalled')} "
            f"survivors={rr.get('survivor_count')}"
        )
    deep.append("\n## Best Robust Clusters (Train)")
    for r in top_clusters:
        line = (
            f"- {r.get('cluster_id')}: score={_float_or(r.get('score')):+.2f} "
            f"trainSh={_float_or(r.get('train_sharpe')):+.2f} wf={_float_or(r.get('wf_median')):+.2f}/{_float_or(r.get('wf_min')):+.2f} "
            f"beta={_float_or(r.get('beta')):+.2f} to={_float_or(r.get('turnover')):.2f}"
        )
        if r.get("robustness_score") is not None:
            line += f" robustScore={_float_or(r.get('robustness_score')):.1f}"
        deep.append(line)
    if test_rows:
        heldout_report = _build_heldout_report(test_rows)
        leader = heldout_report.get("leader")
        winner = heldout_report.get("winner")
        best_det = heldout_report.get("best_deterministic")
        comparison = heldout_report.get("comparison", {})
        score_edge = heldout_report.get("score_edge_vs_deterministic")
        best_evo = heldout_report.get("best_evolution")
        evo_edge = None if not (best_det and best_evo) else float(best_evo["test_score"] - best_det["test_score"])
        econ_success = bool(
            best_evo
            and best_evo["test_sharpe"] >= ECONOMIC_SHARPE_FLOOR
            and (best_det is None or evo_edge >= ECONOMIC_EDGE_OVER_DETERMINISTIC)
        )
        deep.append("\n## Held-out Verdict")
        if leader:
            deep.append(
                f"- Leader by composite score: {leader['cluster_id']} ({leader['iter']}) "
                f"score={leader['test_score']:+.2f} testSh={leader['test_sharpe']:+.2f} "
                f"dd={leader['test_dd']:+.1%} beta={leader['test_beta']:+.2f}"
            )
        if best_det:
            deep.append(
                f"- Best deterministic baseline: {best_det['cluster_id']} ({best_det['iter']}) "
                f"score={best_det['test_score']:+.2f} testSh={best_det['test_sharpe']:+.2f}"
            )
        if winner:
            deep.append(
                f"- Approved winner: {winner['cluster_id']} ({winner['iter']}) "
                f"edge_vs_det={_float_or(score_edge):+.2f} diagnostics="
                f"{comparison.get('wins', 0)}W/{comparison.get('losses', 0)}L/{comparison.get('ties', 0)}T"
            )
        else:
            deep.append("- Approved winner: no winner yet")
            deep.extend([f"- Rule gap: {reason}" for reason in heldout_report.get("reasons", [])])
        deep.append(
            f"- Winner rule: highest composite score, edge >= {APPROVED_WINNER_EDGE_OVER_DETERMINISTIC:+.2f} "
            "vs best deterministic baseline, and no majority loss across supporting diagnostics."
        )
        deep.append(
            f"- Economic success: {econ_success} "
            f"(requires evolved Sharpe >= {ECONOMIC_SHARPE_FLOOR:.2f} and edge >= {ECONOMIC_EDGE_OVER_DETERMINISTIC:+.2f})"
        )
        if best_det and leader:
            deep.append(
                f"- Leader vs deterministic baseline: leader={leader['test_score']:+.2f}/{leader['test_sharpe']:+.2f} "
                f"det={best_det['test_score']:+.2f}/{best_det['test_sharpe']:+.2f} "
                f"edge={_float_or(score_edge):+.2f} diagnostics="
                f"{comparison.get('wins', 0)}W/{comparison.get('losses', 0)}L/{comparison.get('ties', 0)}T"
            )
            for diag in comparison.get("diagnostics", []):
                deep.append(
                    f"  - {diag['label']}: candidate={diag['candidate']:+.2f} "
                    f"baseline={diag['baseline']:+.2f} delta={diag['delta']:+.2f} [{diag['verdict']}]"
                )
        cs = (leader or {}).get("cost_stress", {})
        if cs:
            deep.append(
                "- Cost stress Sharpe: "
                + " | ".join([f"{k}:{v['sharpe']:+.2f}" for k, v in cs.items()])
            )
        subs = (leader or {}).get("subperiods", [])
        if subs:
            deep.append(
                "- Subperiod Sharpe: "
                + " | ".join([f"{s['label']}:{s['sharpe']:+.2f}" for s in subs])
            )

    EVOLUTION_DEEP_DIVE_FILE.write_text("\n".join(deep).strip() + "\n")

    tldr = []
    tldr.append("# Evolution TLDR")
    tldr.append(
        f"- Stop reason: {evo_summary.get('stop_reason')} after {evo_summary.get('generations_executed')} generations."
    )
    if health_lines:
        tldr.append("- Robustness/validity: " + "; ".join(health_lines[:3]) + ".")
    if partial_line:
        tldr.append(f"- Partial run report: {partial_line}.")
    if top_clusters:
        best_train = top_clusters[0]
        tldr.append(
            f"- Best train robust cluster: {best_train.get('cluster_id')} "
            f"(score {_float_or(best_train.get('score')):+.2f}, Sh {_float_or(best_train.get('train_sharpe')):+.2f})."
        )
    if test_rows:
        heldout_report = _build_heldout_report(test_rows)
        leader = heldout_report.get("leader")
        winner = heldout_report.get("winner")
        best_det = heldout_report.get("best_deterministic")
        best_evo = heldout_report.get("best_evolution")
        evo_edge = None if not (best_det and best_evo) else float(best_evo["test_score"] - best_det["test_score"])
        econ_success = bool(
            best_evo
            and best_evo["test_sharpe"] >= ECONOMIC_SHARPE_FLOOR
            and (best_det is None or evo_edge >= ECONOMIC_EDGE_OVER_DETERMINISTIC)
        )
        if winner:
            tldr.append(
                f"- Held-out winner: {winner['cluster_id']} (test score {winner['test_score']:+.2f}, "
                f"test Sharpe {winner['test_sharpe']:+.2f}, edge vs det {heldout_report.get('score_edge_vs_deterministic', 0.0):+.2f})."
            )
        elif leader:
            tldr.append(
                f"- Held-out verdict: no winner yet. Leader is {leader['cluster_id']} "
                f"(score {leader['test_score']:+.2f}, test Sharpe {leader['test_sharpe']:+.2f})."
            )
            tldr.extend([f"- Rule gap: {reason}." for reason in heldout_report.get("reasons", [])])
        else:
            tldr.append("- Held-out verdict: no winner yet.")
        tldr.append(f"- Economic success: {econ_success}.")
        tldr.append(
            f"- Recommended deployment set: top {min(ENSEMBLE_TOP_N, len(test_rows))} held-out variants "
            f"(saved in best_signal_ensemble.py)."
        )
    EVOLUTION_TLDR_FILE.write_text("\n".join(tldr).strip() + "\n")


def evaluate_row_on_test(row):
    short_span, long_span = row.get("short_span", 54), row.get("long_span", 90)
    flipped = bool(row.get("flipped"))
    aux = normalize_aux_params(row.get("params", {}))
    mutation_type = row.get("mutation_type", "plain")
    code_override = row.get("code")
    raw_code = code_override or deterministic_code(row)
    fn = None if code_override else deterministic_signal_factory(short_span, long_span, variant=mutation_type, aux=aux)

    def _run_panel_eval(panel_close, panel_volume, panel_vix, panel_tnx, timeout=25):
        if code_override:
            panel_sig, panel_err = run_signal_code(
                code_override,
                panel_close,
                panel_volume,
                vix_s=panel_vix,
                tnx_s=panel_tnx,
                timeout=timeout,
            )
            if panel_err:
                raise RuntimeError(f"heldout eval code error: {panel_err}")
        else:
            panel_sig = fn(panel_close, panel_volume, panel_vix, panel_tnx)
        if flipped:
            panel_sig = -panel_sig
        panel_metrics = backtest(panel_sig, panel_close)
        return panel_sig, panel_metrics, _oriented_metrics(panel_metrics, flipped=False)

    sig, m, oriented = _run_panel_eval(close_test, volume_test, vix_test, tnx_test, timeout=25)

    def _slice_eval(panel_sig, panel_close, cost_bps=0.0):
        sub_m = backtest(panel_sig, panel_close, cost_bps=cost_bps)
        return sub_m, _oriented_metrics(sub_m, flipped=False)

    beta = oriented["beta"]
    consistency = oriented["consistency"]
    test_sharpe = oriented["sharpe"]
    test_ret = oriented["ann_return"]
    test_dd = oriented["max_dd"]
    test_raw = m.get("sharpe_raw", m.get("sharpe", 0.0))
    bench_spread_sh = m.get("benchmark_spread_sharpe", m.get("excess_sharpe", 0.0))
    wf = []
    n = len(close_test)
    sz = max(n // WF_WINDOWS, 1)
    for w in range(WF_WINDOWS):
        lo = w * sz
        hi = (w + 1) * sz if w < WF_WINDOWS - 1 else n
        try:
            sub_m, sub_oriented = _slice_eval(sig.iloc[lo:hi], close_test.iloc[lo:hi])
        except Exception:
            continue
        wf.append({"window": w, "sharpe": sub_oriented.get("sharpe", sub_m.get("sharpe", 0.0))})
    wf_median, wf_min = wf_summary(wf)
    test_score = _selection_score_safe(test_sharpe, wf_median, consistency, beta, m.get("avg_turnover", 0.0), wf_min=wf_min, max_dd=test_dd)
    test_metric_payload = {
        "train_sharpe": test_sharpe,
        "wf_median": wf_median,
        "wf_min": wf_min,
        "beta": beta,
        "turnover": m.get("avg_turnover", 0.0),
        "consistency": consistency,
        "signal_activity": m.get("signal_activity", 1.0),
        "raw_cs_std": m.get("raw_cs_std", 1.0),
        "raw_long_frac": m.get("raw_long_frac", 1.0),
        "raw_short_frac": m.get("raw_short_frac", 1.0),
    }
    test_robustness = _robustness_score_safe(test_metric_payload)
    cost_stress = {}
    for cost_bps in COST_STRESS_BPS:
        m_cost = backtest(sig, close_test, cost_bps=cost_bps)
        cost_oriented = _oriented_metrics(m_cost, flipped=False)
        cost_stress[f"{int(cost_bps)}bps"] = {
            "sharpe": cost_oriented["sharpe"],
            "ann_return": cost_oriented["ann_return"],
            "max_dd": cost_oriented["max_dd"],
        }

    close_full, volume_full, vix_full, tnx_full = _full_test_panels()
    subperiods = []
    for label, start, end in SUBPERIOD_WINDOWS:
        sub_close = close_full.loc[start:end]
        if len(sub_close) < 60:
            continue
        sub_volume = volume_full.reindex(index=sub_close.index, columns=sub_close.columns)
        sub_vix = vix_full.loc[start:end] if vix_full is not None else None
        sub_tnx = tnx_full.loc[start:end] if tnx_full is not None else None
        try:
            _, sub_m, sub_oriented = _run_panel_eval(sub_close, sub_volume, sub_vix, sub_tnx, timeout=25)
        except Exception:
            continue
        subperiods.append(
            {
                "label": label,
                "start": start,
                "end": end,
                "sharpe": sub_oriented["sharpe"],
                "ann_return": sub_oriented["ann_return"],
                "max_dd": sub_oriented["max_dd"],
                "beta": sub_oriented["beta"],
                "turnover": sub_m["avg_turnover"],
            }
        )

    chronological_holdout = []
    for spec in _build_chronological_window_specs(close_test.index, CHRONOLOGICAL_HOLDOUT_SEGMENTS, min_points=60):
        try:
            sub_m, sub_oriented = _slice_eval(
                sig.iloc[spec["lo"]:spec["hi"]],
                close_test.iloc[spec["lo"]:spec["hi"]],
            )
        except Exception:
            continue
        chronological_holdout.append(
            {
                "label": spec["label"],
                "start": spec["start"],
                "end": spec["end"],
                "sharpe": sub_oriented["sharpe"],
                "ann_return": sub_oriented["ann_return"],
                "max_dd": sub_oriented["max_dd"],
                "beta": sub_oriented["beta"],
                "turnover": sub_m["avg_turnover"],
                "benchmark_spread_sharpe": sub_m.get("benchmark_spread_sharpe", sub_m.get("excess_sharpe", 0.0)),
            }
        )

    rolling_holdout = []
    for spec in _build_rolling_window_specs(close_test.index, ROLLING_HOLDOUT_SEGMENTS, min_points=ROLLING_HOLDOUT_MIN_POINTS):
        try:
            sub_m, sub_oriented = _slice_eval(
                sig.iloc[spec["lo"]:spec["hi"]],
                close_test.iloc[spec["lo"]:spec["hi"]],
            )
        except Exception:
            continue
        rolling_holdout.append(
            {
                "label": spec["label"],
                "start": spec["start"],
                "end": spec["end"],
                "sharpe": sub_oriented["sharpe"],
                "ann_return": sub_oriented["ann_return"],
                "max_dd": sub_oriented["max_dd"],
                "beta": sub_oriented["beta"],
                "turnover": sub_m["avg_turnover"],
                "benchmark_spread_sharpe": sub_m.get("benchmark_spread_sharpe", sub_m.get("excess_sharpe", 0.0)),
            }
        )

    row_source = row.get("source", "deterministic")
    row_iter = _heldout_iter_label(row)
    parent_id = str(row.get("parent_id") or _row_identity_safe(row, default_prefix="parent"))
    hypothesis = row.get("hypothesis")
    if not hypothesis:
        if code_override:
            hypothesis = f"Program evolution {row.get('cluster_id', 'unknown')}"
        else:
            hypothesis = f"Deterministic {mutation_type} EWM {short_span}/{long_span}"

    return {
        "iter": row_iter,
        "parent_id": parent_id,
        "source": row.get("source", "deterministic"),
        "mutation_type": mutation_type,
        "short_span": short_span,
        "long_span": long_span,
        "family": row.get("family", "ewm"),
        "base_family": row.get("base_family", "ewm"),
        "signature": row.get("signature"),
        "cluster_id": row.get("cluster_id", "unknown"),
        "params": aux,
        "hypothesis": hypothesis,
        "train_score": row.get("score", 0.0),
        "test_score": test_score,
        "train_sharpe": row.get("train_sharpe", row.get("sharpe", 0.0)),
        "test_sharpe": test_sharpe,
        "test_consistency": consistency,
        "train_robustness_score": row.get("robustness_score"),
        "test_robustness_score": test_robustness,
        "test_raw": test_raw,
        "test_benchmark_spread_sharpe": bench_spread_sh,
        "test_beta": beta,
        "test_turnover": m["avg_turnover"],
        "test_ret": test_ret,
        "test_dd": test_dd,
        "cost_stress": cost_stress,
        "subperiods": subperiods,
        "chronological_holdout": chronological_holdout,
        "rolling_holdout": rolling_holdout,
        "wf_median": wf_median,
        "wf_min": wf_min,
        "wf_windows": wf,
        "equity": oriented["equity"],
        "code": _oriented_signal_code(raw_code, flipped),
        "raw_code": raw_code,
        "code_is_oriented": True,
        "flipped": flipped,
    }


def evaluate_on_test(top_k=TOP_K):
    if not RUN_HELDOUT_EVAL:
        print("held-out evaluation paused")
        return []
    scope_meta = _runtime_family_scope_lists()
    active_sources = set(scope_meta.get("active_sources", []))
    det = [r for r in _safe_load_search_results_local() if r.get("score") is not None]
    deferred_rows = [r for r in det if _row_matches_deferred_stage_family(r)]
    active_scope_rows = [r for r in det if not _row_matches_deferred_stage_family(r)]
    deterministic = [r for r in active_scope_rows if r.get("source") == "deterministic"]
    evolution = [r for r in active_scope_rows if r.get("source") == "parameter_search_evolution"]
    evolution_champions = [r for r in evolution if r.get("origin") == "deterministic_champion"]
    evolution_rest = [r for r in evolution if r.get("origin") != "deterministic_champion"]
    ignored_other = [r for r in active_scope_rows if r.get("source") not in active_sources]
    picked = []
    seen_sig = set()
    per_cluster = {}
    per_mutation = {}
    picked.extend(_pick_diverse(evolution_champions, HELDOUT_MIN_EVOLUTION_CHAMPIONS, seen_sig, per_cluster, per_mutation))
    picked.extend(_pick_diverse(evolution_rest, max(0, HELDOUT_MIN_EVOLUTION - len(picked)), seen_sig, per_cluster, per_mutation))
    picked.extend(_pick_diverse(deterministic, HELDOUT_MIN_DETERMINISTIC, seen_sig, per_cluster, per_mutation))
    remaining = max(top_k, HELDOUT_SHORTLIST_K) - len(picked)
    if remaining > 0:
        already = {id(r) for r in picked}
        rest = [r for r in active_scope_rows if id(r) not in already and r.get("source") in active_sources]
        picked.extend(_pick_diverse(rest, remaining, seen_sig, per_cluster, per_mutation))
    det_ranked = picked[:max(top_k, HELDOUT_SHORTLIST_K)]
    print("held-out execution scope:", scope_meta.get("active_scope"))
    print("held-out configured roadmap families:", ", ".join(scope_meta.get("configured", [])))
    print("held-out executed families:", ", ".join(scope_meta.get("executed", [])))
    print("held-out deferred families:", ", ".join(scope_meta.get("deferred", [])))
    if deferred_rows:
        print(f"held-out deferred rows ignored: {len(deferred_rows)}")
    if ignored_other:
        print(f"held-out non-scope rows ignored: {len(ignored_other)}")
    if det_ranked:
        print(f"held-out shortlist: {len(det_ranked)} rows (max {HELDOUT_MAX_PER_CLUSTER} per cluster)")
        source_counts = {}
        for row in det_ranked:
            src = row.get("source", "unknown")
            source_counts[src] = source_counts.get(src, 0) + 1
        print("shortlist sources:", ", ".join([f"{k}={v}" for k, v in sorted(source_counts.items())]))
        evaluated = []
        for r in det_ranked:
            try:
                evaluated.append(evaluate_row_on_test(r))
            except Exception as e:
                print(f"held-out eval skipped {_row_identity_safe(r)}: {e}")
        return sorted(evaluated, key=lambda r: -_float_or(r.get("test_score"), -999.0))
    return []


test_results = evaluate_on_test()
heldout_report = _build_heldout_report(test_results)
if test_results:
    print(f"\n{'iter':>10} {'train_sc':>9} {'test_sc':>8} {'train_Sh':>9} {'test_Sh':>8} {'rob':>6} {'beta':>7} {'to':>6} {'wf_med':>7} {'test_dd':>8}")
    for r in test_results:
        rob = r.get("test_robustness_score", r.get("train_robustness_score"))
        rob_text = "n/a" if rob is None else f"{_float_or(rob):.1f}"
        print(f"{str(r['iter']):>10} {r['train_score']:>+9.2f} {r['test_score']:>+8.2f} {r['train_sharpe']:>+9.2f} {r['test_sharpe']:>+8.2f} {rob_text:>6} {r['test_beta']:>+7.2f} {r['test_turnover']:>6.2f} {r['wf_median']:>+7.2f} {r['test_dd']:>+8.1%}")
    best = heldout_report.get("leader")
    approved_winner = heldout_report.get("winner")
    comparison = heldout_report.get("comparison", {})
    BEST_CODE.write_text(
        f"# objective=market_neutral_net_sharpe  heldout_status={heldout_report.get('status')}  "
        f"iter {best['iter']}  cluster={best['cluster_id']}  train_score={best['train_score']:+.2f}  "
        f"test_score={best['test_score']:+.2f}  train_Sh={best['train_sharpe']:+.2f}  test_Sh={best['test_sharpe']:+.2f}\n"
        f"# parent={best['parent_id']}  mutation={best['mutation_type']}\n"
        f"# winner_rule_edge_vs_det={_float_or(heldout_report.get('score_edge_vs_deterministic'), 0.0):+.2f}  "
        f"diagnostics={comparison.get('wins', 0)}W/{comparison.get('losses', 0)}L/{comparison.get('ties', 0)}T\n"
        f"# HYPOTHESIS: {best['hypothesis']}\n\nimport numpy as np\n\n{best['code']}\n"
    )
    ensemble_rows = sorted(test_results, key=lambda r: -_float_or(r.get("test_score"), -999.0))[:min(ENSEMBLE_TOP_N, len(test_results))]
    ensemble_file = OUT / "best_signal_ensemble.py"
    ensemble_file.write_text(_ensemble_code(ensemble_rows))
    print(f"\nheld-out leader (by composite score): iter {best['iter']}  saved to {BEST_CODE.name}")
    if approved_winner:
        print(
            "approved winner: "
            f"iter {approved_winner['iter']} edge_vs_det={_float_or(heldout_report.get('score_edge_vs_deterministic'), 0.0):+.2f} "
            f"diagnostics={comparison.get('wins', 0)}W/{comparison.get('losses', 0)}L/{comparison.get('ties', 0)}T"
        )
    else:
        print("approved winner: no winner yet")
        for reason in heldout_report.get("reasons", []):
            print("  rule gap:", reason)
    print(f"ensemble ({len(ensemble_rows)} members) saved to {ensemble_file.name}")
else:
    print("held-out evaluation skipped or no surviving candidates")

_write_evolution_analysis(test_results)


## Cell 12 â€” Plots

In [ ]:
log = load_log()
iters = [e.get("iter", 0) for e in log]
sharpes = [e.get("sharpe") for e in log]

if RUN_REPORTS and log:
    running_best, cur = [], -np.inf
    for s in sharpes:
        if s is not None and s > cur:
            cur = s
        running_best.append(cur if cur > -np.inf else np.nan)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.scatter(iters, [s if s is not None else np.nan for s in sharpes], alpha=0.5, s=40, label="iter Sharpe")
    ax.plot(iters, running_best, "-", lw=2, label="running best")
    ax.axhline(0, ls="--", color="grey", alpha=0.4)
    ax.set_xlabel("Iteration"); ax.set_ylabel("Train Sharpe")
    ax.set_title("AutoResearch v2 - alpha discovery progress")
    ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(SHARPE_PLOT, dpi=140)
    plt.close(fig)

    if test_results:
        fig, ax = plt.subplots(figsize=(10, 5))
        for r in test_results:
            r["equity"].plot(ax=ax, label=f"iter {r['iter']}  trSh={r['train_sharpe']:+.1f} teSh={r['test_sharpe']:+.1f}")
        ax.set_ylabel("Equity")
        ax.set_title(f"Top-{TOP_K} signals on held-out test")
        ax.legend(fontsize=8, loc="best"); ax.grid(alpha=0.3)
        fig.tight_layout(); fig.savefig(EQUITY_PLOT, dpi=140)
        plt.close(fig)
    print("plots saved")
else:
    print("report plots paused")


## Cell 13 â€” Experiment summary

In [ ]:
def _float_or(value, default=0.0):
    try:
        if value is None:
            return default
        return float(value)
    except Exception:
        return default


def _safe_json_value(path, default):
    if path is None:
        return default
    try:
        if (not path.exists()) or path.stat().st_size == 0:
            return default
        text = path.read_text()
        if not text.strip():
            return default
        value = json.loads(text)
        return default if value is None else value
    except Exception:
        return default


def _as_row_list(value):
    if isinstance(value, list):
        return [r for r in value if isinstance(r, dict)]
    if isinstance(value, dict):
        for key in ("rows", "results", "candidates", "lineage", "programs"):
            rows = value.get(key)
            if isinstance(rows, list):
                return [r for r in rows if isinstance(r, dict)]
        return [value] if value.get("score") is not None else []
    return []


def _safe_json_rows(path):
    return _as_row_list(_safe_json_value(path, []))


def _safe_json_dict(path):
    value = _safe_json_value(path, {})
    return value if isinstance(value, dict) else {}


def _path_from_global(name, fallback_name=None):
    path = globals().get(name)
    if path is None and fallback_name and "OUT" in globals():
        path = OUT / fallback_name
    return path


def _safe_rows_from_loader(loader_name, file_name=None):
    loader = globals().get(loader_name)
    if callable(loader):
        try:
            rows = _as_row_list(loader())
            if rows:
                return rows
        except Exception:
            pass
    if file_name:
        return _safe_json_rows(_path_from_global(file_name))
    return []


def _safe_dict_from_global(name, fallback_name=None):
    return _safe_json_dict(_path_from_global(name, fallback_name))


def _safe_text(path):
    try:
        if path is None or (not path.exists()) or path.stat().st_size == 0:
            return ""
        return path.read_text()
    except Exception:
        return ""


def _runtime_scope_metadata():
    return _safe_dict_from_global("RUNTIME_METADATA_FILE")


def _runtime_family_scope_lists():
    runtime_metadata = _runtime_scope_metadata()
    configured = runtime_metadata.get("configured_method_families")
    if not isinstance(configured, list) or not configured:
        configured = list(globals().get("ROADMAP_METHOD_FAMILIES", globals().get("BENCHMARK_METHOD_FAMILIES", [])))
    executed = runtime_metadata.get("executed_method_families")
    if not isinstance(executed, list) or not executed:
        executed = list(globals().get("EXECUTED_METHOD_FAMILIES", []))
    configured = [str(x) for x in configured]
    executed = [str(x) for x in executed]
    deferred = [family for family in configured if family not in executed]
    active_sources = runtime_metadata.get("active_result_sources")
    if not isinstance(active_sources, list) or not active_sources:
        active_sources = list(globals().get("ACTIVE_RESULT_SOURCES", ["deterministic", "parameter_search_evolution"]))
    active_scope = runtime_metadata.get("active_execution_scope") or globals().get(
        "ACTIVE_EXECUTION_SCOPE",
        "deterministic/classical/autoresearch only",
    )
    return {
        "configured": configured,
        "executed": executed,
        "deferred": deferred,
        "active_sources": [str(x) for x in active_sources],
        "active_scope": str(active_scope),
    }


def _row_family_tokens(row):
    if not isinstance(row, dict):
        return set()
    tokens = set()
    for key in (
        "source",
        "family",
        "base_family",
        "benchmark_family",
        "model_family",
        "study_family",
        "study_type",
        "label",
        "name",
    ):
        value = row.get(key)
        if value in (None, ""):
            continue
        text = str(value).strip().lower()
        if text:
            tokens.add(text)
    return tokens


def _row_matches_deferred_stage_family(row):
    deferred_aliases = {
        "lstm",
        "lstm_sharpe",
        "tabular",
        "tabular_ml",
        "gbt",
        "gradient_boosted_trees",
        "combination",
        "combination_studies",
        "ensemble_combination",
    }
    tokens = _row_family_tokens(row)
    return any(alias in token for token in tokens for alias in deferred_aliases)


def _robustness_summary(rows):
    values = [_float_or(r.get("robustness_score"), None) for r in rows if r.get("robustness_score") is not None]
    values = [v for v in values if v is not None]
    if not values:
        return None
    robust_ok = len([r for r in rows if r.get("robust_ok")])
    zero_score = len([v for v in values if v <= 0.0])
    return f"avg={sum(values) / len(values):.1f} best={max(values):.1f} robust_ok={robust_ok}/{len(rows)} zero_score={zero_score}"


def _evolution_health_lines(evolution_summary, partial_report=None):
    partial_report = partial_report if isinstance(partial_report, dict) else {}
    generations = evolution_summary.get("generations", []) if isinstance(evolution_summary, dict) else []
    generations = [g for g in generations if isinstance(g, dict)]
    eval_total = sum(int(_float_or(g.get("eval_count"), 0.0)) for g in generations)
    valid_total = sum(int(_float_or(g.get("valid_count"), 0.0)) for g in generations)
    robust_total = sum(int(_float_or(g.get("robust_count"), 0.0)) for g in generations)

    valid_rate = partial_report.get("valid_rate", partial_report.get("overall_valid_rate"))
    if valid_rate is None and eval_total:
        valid_rate = valid_total / max(eval_total, 1)

    zero_robust = partial_report.get("zero_robust_generations")
    if zero_robust is None and generations:
        zero_robust = [g.get("gen_id") for g in generations if int(_float_or(g.get("robust_count"), 0.0)) == 0]

    zero_robust_streak = partial_report.get("zero_robust_streak")
    if zero_robust_streak is None and generations:
        zero_robust_streak = 0
        for g in reversed(generations):
            if int(_float_or(g.get("robust_count"), 0.0)) == 0:
                zero_robust_streak += 1
            else:
                break

    lines = []
    if valid_rate is not None:
        try:
            rate_text = f"{float(valid_rate):.1%}"
        except Exception:
            rate_text = str(valid_rate)
        suffix = f" ({valid_total}/{eval_total} valid/evaluated)" if eval_total else ""
        lines.append(f"valid_rate={rate_text}{suffix}")
    if zero_robust is not None:
        if isinstance(zero_robust, list):
            lines.append(f"zero_robust_generations={len(zero_robust)} latest={zero_robust[-5:] if zero_robust else []}")
        else:
            lines.append(f"zero_robust_generations={zero_robust}")
    if zero_robust_streak:
        lines.append(f"zero_robust_streak={zero_robust_streak}")
    if eval_total:
        lines.append(f"robust_total={robust_total}/{valid_total} valid rows")
    return lines


def _partial_report_line(partial_report):
    if not isinstance(partial_report, dict) or not partial_report:
        return None
    fields = []
    for key in (
        "status",
        "phase",
        "gen_id",
        "generation",
        "generations_executed",
        "valid_rate",
        "robust_count",
        "zero_robust_streak",
        "stop_reason",
        "updated_at",
        "timestamp",
    ):
        if key in partial_report:
            fields.append(f"{key}={partial_report.get(key)}")
    if not fields:
        fields.append("keys=" + ",".join(list(partial_report.keys())[:8]))
    return " ".join(fields)


test_results = [r for r in globals().get("test_results", []) if isinstance(r, dict)]
log = _safe_rows_from_loader("load_log", "LOG_FILE")
scope_meta = _runtime_family_scope_lists()
active_sources = set(scope_meta.get("active_sources", []))
det_all = [r for r in _safe_rows_from_loader("load_deterministic", "DETERMINISTIC_FILE") if r.get("score") is not None]
param_all = [r for r in _safe_rows_from_loader("load_parameter_search", "PARAM_SEARCH_FILE") if r.get("score") is not None]
if "load_search_results" in globals():
    search_rows_all = _safe_rows_from_loader("load_search_results")
else:
    search_rows_all = []
    if "PARAM_SEARCH_FILE" in globals() and PARAM_SEARCH_FILE.exists():
        search_rows_all.extend(_safe_json_rows(PARAM_SEARCH_FILE))
    if "DETERMINISTIC_FILE" in globals() and DETERMINISTIC_FILE.exists():
        search_rows_all.extend(_safe_json_rows(DETERMINISTIC_FILE))
deferred_train_rows = [r for r in search_rows_all if _row_matches_deferred_stage_family(r)]
search_rows = [
    r for r in search_rows_all
    if (not _row_matches_deferred_stage_family(r))
    and (r.get("source") in active_sources or r.get("source") in (None, ""))
]
det = [r for r in det_all if not _row_matches_deferred_stage_family(r)]
param = [r for r in param_all if not _row_matches_deferred_stage_family(r)]
good = [e for e in search_rows if e.get("train_sharpe") is not None]
fail = [e for e in log if e.get("error")]
best_train_det = max(det, key=lambda e: _float_or(e.get("score"), -999.0)) if det else None
best_train_search = max(param, key=lambda e: _float_or(e.get("score"), -999.0)) if param else None
deferred_stage_test_rows = [r for r in test_results if _row_matches_deferred_stage_family(r)]
test_results = [r for r in test_results if not _row_matches_deferred_stage_family(r)]
heldout_report_builder = globals().get("_build_heldout_report")
heldout_report = heldout_report_builder(test_results) if callable(heldout_report_builder) else {}
best_test = heldout_report.get("leader") if isinstance(heldout_report, dict) else None
if best_test is None:
    best_test = max(test_results, key=lambda r: _float_or(r.get("test_score"), -999.0)) if test_results else None
heldout_winner = heldout_report.get("winner") if isinstance(heldout_report, dict) else None
heldout_best_det = heldout_report.get("best_deterministic") if isinstance(heldout_report, dict) else None
heldout_best_evo = heldout_report.get("best_evolution") if isinstance(heldout_report, dict) else None
heldout_comparison = heldout_report.get("comparison", {}) if isinstance(heldout_report, dict) else {}
heldout_rule_reasons = heldout_report.get("reasons", []) if isinstance(heldout_report, dict) else []
heldout_rule_status = heldout_report.get("status", "no_winner_yet") if isinstance(heldout_report, dict) else "no_winner_yet"
try:
    clusters = cluster_summary(search_rows if search_rows else log)
except Exception:
    clusters = []
heldout_source_counts = {}
for r in test_results:
    src = r.get("source", "unknown")
    heldout_source_counts[src] = heldout_source_counts.get(src, 0) + 1
configured_families = list(dict.fromkeys(scope_meta.get("configured", [])))
metadata_executed_families = list(dict.fromkeys(scope_meta.get("executed", [])))
active_execution_scope = scope_meta.get("active_scope", "deterministic/classical/autoresearch only")
if not configured_families:
    configured_families = list(globals().get("ROADMAP_METHOD_FAMILIES", globals().get("BENCHMARK_METHOD_FAMILIES", [])))
inferred_executed_families = []
if det:
    inferred_executed_families.append("deterministic")
if any(r.get("mutation_type") in {"regime_gate", "regime_momentum", "volume_gate", "volume_confirm", "vol_scale", "vol_adjusted"} for r in det):
    inferred_executed_families.append("classical_risk_managed")
if param or heldout_source_counts.get("parameter_search_evolution", 0) > 0:
    inferred_executed_families.append("autoresearch_evolution")
executed_families = list(dict.fromkeys(metadata_executed_families + inferred_executed_families))
deferred_families = [family for family in configured_families if family not in executed_families]
evolution_summary = _safe_dict_from_global("EVOLUTION_SUMMARY_FILE")
evolution_memory = _safe_dict_from_global("EVOLUTION_MEMORY_FILE")
partial_run_report = _safe_dict_from_global("PARTIAL_REPORT_FILE", "partial_run_report.json")
evolution_health_lines = _evolution_health_lines(evolution_summary, partial_run_report)
partial_run_report_line = _partial_report_line(partial_run_report)
search_robustness_line = _robustness_summary(search_rows)
evolution_policy = evolution_summary.get("policy", {}) if isinstance(evolution_summary.get("policy"), dict) else {}
adaptive_diagnostic_run = bool(evolution_policy.get("adaptive_diagnostic_run"))
latest_family_telemetry = {}
if isinstance(evolution_summary.get("latest"), dict):
    latest_family_telemetry = evolution_summary["latest"].get("family_telemetry", {}) or {}

unique_param_sigs = len({r.get("signature") for r in param if r.get("signature")})
param_unique_ratio = unique_param_sigs / max(len(param), 1)
heldout_has_det = heldout_source_counts.get("deterministic", 0) > 0
heldout_has_evo = heldout_source_counts.get("parameter_search_evolution", 0) > 0
heldout_has_approved_winner = bool(heldout_winner)
heldout_winner_is_evo = bool(heldout_winner and heldout_winner.get("source") == "parameter_search_evolution")
best_train_score = _float_or(best_train_search.get("score"), None) if best_train_search else None
best_test_score = _float_or(best_test.get("test_score"), None) if best_test else None
generalization_gap = None
if best_train_score is not None and best_test_score is not None:
    generalization_gap = float(best_train_score - best_test_score)
best_det_test = heldout_best_det or max([r for r in test_results if r.get("source") == "deterministic"], key=lambda r: _float_or(r.get("test_score"), -999.0), default=None)
best_evo_test = heldout_best_evo or max([r for r in test_results if r.get("source") == "parameter_search_evolution"], key=lambda r: _float_or(r.get("test_score"), -999.0), default=None)
evolution_test_edge = None
if best_det_test and best_evo_test:
    evolution_test_edge = float(_float_or(best_evo_test.get("test_score")) - _float_or(best_det_test.get("test_score")))
leader_vs_det_edge = heldout_report.get("score_edge_vs_deterministic") if isinstance(heldout_report, dict) else None
economic_sharpe_floor = globals().get("ECONOMIC_SHARPE_FLOOR", 0.50)
economic_edge_floor = globals().get("ECONOMIC_EDGE_OVER_DETERMINISTIC", 0.05)
approved_winner_edge_floor = globals().get("APPROVED_WINNER_EDGE_OVER_DETERMINISTIC", 0.10)
economic_success = bool(
    best_evo_test
    and best_evo_test.get("test_sharpe", 0.0) >= economic_sharpe_floor
    and (best_det_test is None or evolution_test_edge >= economic_edge_floor)
)

adherence_score = 0
adherence_notes = []
if evolution_summary and evolution_summary.get("generations_executed", 0) >= 20:
    adherence_score += 20
else:
    adherence_notes.append("not enough autonomous generations")
if evolution_policy.get("train_val_firewall"):
    adherence_score += 20
else:
    adherence_notes.append("missing train/validation firewall")
if evolution_memory and evolution_memory.get("generation_reflections"):
    adherence_score += 15
else:
    adherence_notes.append("reflection memory is absent or unused")
if param_unique_ratio >= 0.25:
    adherence_score += 15
else:
    adherence_notes.append("candidate lineage is too duplicate-heavy")
if heldout_has_det and heldout_has_evo:
    adherence_score += 15
else:
    adherence_notes.append("held-out shortlist lacks both baseline anchors and evolved candidates")
if generalization_gap is not None and generalization_gap <= 0.20:
    adherence_score += 15
else:
    adherence_notes.append("train-to-test gap is still large")
if heldout_has_approved_winner and heldout_winner_is_evo:
    adherence_score += 10
else:
    adherence_notes.append("held-out winner rule not met yet")
if not economic_success:
    adherence_notes.append("economic success gate not met")
adherence_score = min(100, adherence_score)
if not heldout_has_approved_winner:
    adherence_score = min(90, adherence_score)
if not economic_success:
    adherence_score = min(85, adherence_score)
research_result_status = "successful" if heldout_has_approved_winner else "not-yet-successful"

runtime_metadata = _runtime_scope_metadata()
configured_model_id = runtime_metadata.get("configured_model_id", globals().get("CONFIGURED_MODEL_ID", globals().get("MODEL_ID", "unknown")))
actual_model_id = runtime_metadata.get("actual_model_id") or globals().get("ACTIVE_MODEL_ID") or "(not loaded)"
llm_enabled = bool(runtime_metadata.get("llm_stage_enabled", globals().get("RUN_LLM_STAGE", False)))
llm_loaded = bool(runtime_metadata.get("llm_stage_loaded", False) or globals().get("ACTIVE_MODEL_ID") or globals().get("model") is not None)
llm_executed = bool(runtime_metadata.get("llm_stage_executed", False))
llm_calls = int(_float_or(runtime_metadata.get("llm_calls"), 0.0))
if llm_loaded and (not actual_model_id or actual_model_id == "(not loaded)"):
    actual_model_id = globals().get("ACTIVE_MODEL_ID") or globals().get("MODEL_ID") or configured_model_id
llm_call_accounting_note = "none"
if "autoresearch_evolution" in executed_families and llm_calls == 0:
    llm_call_accounting_note = "program evolution executed with zero recorded LLM generation/reflection calls"
hf_meta = runtime_metadata.get("token_status", {}).get("hf", {})
hf_state = "present" if hf_meta.get("present") else "missing"
hf_source = hf_meta.get("source", "none")
run_id = runtime_metadata.get("run_id", globals().get("RUN_ID", "unknown"))
runtime_profile = runtime_metadata.get("run_profile", globals().get("RUN_PROFILE", "unknown"))
artifact_scope = runtime_metadata.get("report_scope", globals().get("REPORT_SCOPE", "unscoped"))
benchmark_mode = bool(runtime_metadata.get("benchmark_mode", globals().get("BENCHMARK_MODE", False)))
if "sync_runtime_metadata" in globals():
    try:
        runtime_metadata = sync_runtime_metadata(
            configured_method_families=list(configured_families),
            executed_method_families=list(executed_families),
            deferred_method_families=list(deferred_families),
            active_execution_scope=active_execution_scope,
            actual_model_id=None if actual_model_id == "(not loaded)" else actual_model_id,
            llm_stage_enabled=llm_enabled,
            llm_stage_loaded=llm_loaded,
            llm_stage_executed=llm_executed,
            llm_calls=llm_calls,
        )
        configured_model_id = runtime_metadata.get("configured_model_id", configured_model_id)
        actual_model_id = runtime_metadata.get("actual_model_id") or actual_model_id
    except Exception:
        pass

md_text = f"""# AutoResearch v2 - Momentum Alpha Discovery

**Run:** {run_id} | profile={runtime_profile} | scope={artifact_scope}
**Benchmark mode:** {benchmark_mode}
**Adaptive diagnostic run:** {adaptive_diagnostic_run}
**Configured model:** {configured_model_id}
**Actual loaded model:** {actual_model_id}
**LLM stage:** enabled={llm_enabled} | loaded={llm_loaded} | executed={llm_executed} | calls={llm_calls}
**LLM call accounting:** {llm_call_accounting_note}
**HF token:** {hf_state} (source={hf_source})
**Universe:** {len(close_all.columns)} US equities | {close_all.index.min().date()} -> {close_all.index.max().date()}
**Train:** through {TRAIN_END} | **Test (held-out):** after
**Selection objective:** market-neutral net Sharpe (`SELECTION_OBJECTIVE={SELECTION_OBJECTIVE}`)

## Method-family scope
- configured roadmap families: {", ".join(configured_families) if configured_families else "none recorded"}
- executed in this stage: {", ".join(executed_families) if executed_families else "none recorded"}
- deferred or not executed in this stage: {", ".join(deferred_families) if deferred_families else "none recorded"}
- active execution scope: {active_execution_scope}
- note: LSTM, tabular ML, GBT, and combination studies were not executed in this stage

## Execution state
- baseline sweep: {RUN_BASELINE_SWEEP}
- deterministic search: {RUN_DETERMINISTIC_SEARCH}
- parameter search: {RUN_PARAM_SEARCH}
- held-out eval: {RUN_HELDOUT_EVAL}
- held-out shortlist evaluated: {len(test_results)}
- held-out sources: {heldout_source_counts if heldout_source_counts else "none"}
- held-out deferred rows ignored: {len(deferred_stage_test_rows)}
- reports: {RUN_REPORTS}
- partial run report: {partial_run_report_line if partial_run_report_line else "not present"}

## Deterministic search
- candidates evaluated: {len(det)}
"""
if best_train_det:
    md_text += f"- best deterministic train score: **{_float_or(best_train_det.get('score')):+.2f}** ({best_train_det.get('cluster_id', 'unknown')})\n"
    md_text += f"- best deterministic train Sharpe: **{_float_or(best_train_det.get('train_sharpe', best_train_det.get('sharpe'))):+.2f}**\n"
else:
    md_text += "- deterministic search not run or no valid rows\n"

md_text += "\n## Parameter search stage\n"
md_text += f"- parameter-search rows in active scope: {len(param)}\n"
md_text += f"- deferred train rows ignored in this stage: {len(deferred_train_rows)}\n"
md_text += f"- combined robust survivors: {len([r for r in search_rows if r.get('robust_ok')])}\n"
if search_robustness_line:
    md_text += f"- robustness score: {search_robustness_line}\n"
if best_train_search:
    md_text += f"- best parameter-search train score: **{_float_or(best_train_search.get('score')):+.2f}** ({best_train_search.get('cluster_id', 'unknown')})\n"
else:
    md_text += "- parameter search skipped or no valid survivors\n"
if (not RUN_PARAM_SEARCH) and len(param) == 0:
    md_text += "- note: held-out shortlist can still be drawn from deterministic survivors when parameter search is paused.\n"
if evolution_summary:
    md_text += (
        f"- evolution generations executed: {evolution_summary.get('generations_executed', 0)} "
        f"(stop: {evolution_summary.get('stop_reason', 'n/a')})\n"
    )
if evolution_health_lines:
    md_text += "- evolution validity/robustness: " + "; ".join(evolution_health_lines) + "\n"
if latest_family_telemetry:
    md_text += "- latest generation family telemetry:\n"
    for key in ("generated", "critic_rejected", "validation_failed", "scored", "scored_beta_high", "robust", "survivor"):
        md_text += f"  - {key}: {latest_family_telemetry.get(key, {})}\n"
if "RUN_MANIFEST_FILE" in globals() and RUN_MANIFEST_FILE.exists():
    md_text += f"- run manifest: `{RUN_MANIFEST_FILE.name}`\n"
if "EVOLUTION_PROGRAM_FILE" in globals() and EVOLUTION_PROGRAM_FILE.exists():
    md_text += f"- evolution program log: `{EVOLUTION_PROGRAM_FILE.name}`\n"

md_text += "\n## Top clusters\n"
if clusters:
    for r in clusters[:5]:
        md_text += f"- {r.get('cluster_id', 'unknown')}: bestScore={_float_or(r.get('best_score')):+.2f} | trainSh={_float_or(r.get('best_sharpe')):+.2f} | count={r.get('count', 0)} | mut={r.get('mutation_type', 'unknown')}\n"
else:
    md_text += "- none\n"

md_text += "\n## AutoResearch Adherence\n"
md_text += f"- score: **{adherence_score}/100**\n"
md_text += f"- result status: **{research_result_status}**\n"
md_text += f"- held-out verdict: **{heldout_rule_status.replace('_', ' ')}**\n"
md_text += f"- unique parameter signatures: {unique_param_sigs}/{len(param)} ({param_unique_ratio:.1%})\n"
if generalization_gap is not None:
    md_text += f"- best train-to-held-out score gap: {generalization_gap:+.2f}\n"
if best_evo_test:
    md_text += f"- best evolved held-out: score={_float_or(best_evo_test.get('test_score')):+.2f} | Sh={_float_or(best_evo_test.get('test_sharpe')):+.2f}\n"
if best_det_test:
    md_text += f"- best deterministic held-out: score={_float_or(best_det_test.get('test_score')):+.2f} | Sh={_float_or(best_det_test.get('test_sharpe')):+.2f}\n"
if evolution_test_edge is not None:
    md_text += f"- evolved edge over deterministic: {evolution_test_edge:+.2f}\n"
if leader_vs_det_edge is not None:
    md_text += f"- current leader edge over deterministic baseline: {leader_vs_det_edge:+.2f}\n"
if heldout_comparison:
    md_text += (
        "- supporting diagnostics vs deterministic baseline: "
        f"{heldout_comparison.get('wins', 0)} wins / {heldout_comparison.get('losses', 0)} losses / "
        f"{heldout_comparison.get('ties', 0)} ties\n"
    )
if search_robustness_line:
    md_text += f"- train robustness score summary: {search_robustness_line}\n"
if evolution_health_lines:
    md_text += "- evolution zero-robust/valid-rate: " + "; ".join(evolution_health_lines) + "\n"
failure_mode_notes = []
if latest_family_telemetry:
    generated_total = sum(int(v) for v in latest_family_telemetry.get("generated", {}).values())
    scored_total = sum(int(v) for v in latest_family_telemetry.get("scored", {}).values())
    if generated_total and scored_total / max(generated_total, 1) < 0.10:
        failure_mode_notes.append("generator validity bottleneck")
if generalization_gap is not None and generalization_gap > 0.50:
    failure_mode_notes.append("scoring overfit / train-to-held-out gap")
if evolution_test_edge is not None and evolution_test_edge < 0:
    failure_mode_notes.append("no evolved held-out edge over deterministic baseline")
if failure_mode_notes:
    md_text += "- evolved failure mode diagnosis: " + "; ".join(failure_mode_notes) + "\n"
md_text += f"- economic success gate: evolved Sh >= {economic_sharpe_floor:.2f} and edge >= {economic_edge_floor:+.2f}\n"
md_text += (
    f"- approved held-out winner gate: highest composite score, edge >= {approved_winner_edge_floor:+.2f} "
    "vs best deterministic baseline, and must not lose a majority of supporting diagnostics\n"
)
if adaptive_diagnostic_run:
    md_text += "- adaptive diagnostic caveat: this run may use lessons from the current held-out period, so it is not final scientific proof.\n"
if adherence_notes:
    md_text += "- open issues: " + "; ".join(adherence_notes) + "\n"
else:
    md_text += "- open issues: none from automated audit\n"
if heldout_rule_reasons and not heldout_has_approved_winner:
    md_text += "- winner rule gaps: " + "; ".join(heldout_rule_reasons) + "\n"

md_text += f"""

## Reflection memo
{load_memo()}

## Held-out verdict
"""
if best_test:
    cs = best_test.get("cost_stress", {})
    cs_line = "n/a"
    if cs:
        keys = [k for k in ("5bps", "10bps", "15bps") if k in cs]
        if keys:
            cs_line = " | ".join([f"{k}:{_float_or(cs.get(k, {}).get('sharpe')):+.2f}" for k in keys])
    subs = best_test.get("subperiods", [])
    sub_line = "n/a"
    if subs:
        sub_line = " | ".join([f"{s.get('label', 'period')}:{_float_or(s.get('sharpe')):+.2f}" for s in subs if isinstance(s, dict)])
    train_rob = best_test.get("train_robustness_score")
    test_rob = best_test.get("test_robustness_score")
    robust_line = "n/a"
    if train_rob is not None or test_rob is not None:
        robust_line = f"train={_float_or(train_rob):.1f}" if train_rob is not None else "train=n/a"
        robust_line += " | " + (f"test={_float_or(test_rob):.1f}" if test_rob is not None else "test=n/a")
    md_text += f"""- leader by composite score: iter {best_test.get('iter')} ({best_test.get('cluster_id', 'unknown')}): train score={_float_or(best_test.get('train_score')):+.2f} -> test score=**{_float_or(best_test.get('test_score')):+.2f}**
- train Sh={_float_or(best_test.get('train_sharpe')):+.2f} | test Sh={_float_or(best_test.get('test_sharpe')):+.2f}
- robustness score: {robust_line}
- test AnnRet: {_float_or(best_test.get('test_ret')):+.1%} | test DD: {_float_or(best_test.get('test_dd')):+.1%} | beta: {_float_or(best_test.get('test_beta')):+.2f} | turnover: {_float_or(best_test.get('test_turnover')):.2f}
- benchmark-spread Sharpe diagnostic: {_float_or(best_test.get('test_benchmark_spread_sharpe')):+.2f}
- walk-forward median/min Sh: {_float_or(best_test.get('wf_median')):+.2f}/{_float_or(best_test.get('wf_min')):+.2f}
- cost-stress test Sh (5/10/15 bps): {cs_line}
- subperiod Sharpe (2015-2018/2019-2021/2022-2024): {sub_line}
"""
    if best_det_test:
        md_text += f"- best deterministic baseline: iter {best_det_test.get('iter')} ({best_det_test.get('cluster_id', 'unknown')}) | test score={_float_or(best_det_test.get('test_score')):+.2f} | test Sh={_float_or(best_det_test.get('test_sharpe')):+.2f}\n"
    if heldout_winner:
        md_text += (
            f"- approved winner: **{heldout_winner.get('cluster_id', 'unknown')}** "
            f"(edge vs deterministic {_float_or(leader_vs_det_edge, 0.0):+.2f}; diagnostics "
            f"{heldout_comparison.get('wins', 0)}W/{heldout_comparison.get('losses', 0)}L/{heldout_comparison.get('ties', 0)}T)\n"
        )
    else:
        md_text += "- approved winner: **no winner yet**\n"
        if heldout_rule_reasons:
            md_text += "- rule gaps: " + "; ".join(heldout_rule_reasons) + "\n"
    if heldout_comparison.get("diagnostics"):
        md_text += "- leader vs deterministic diagnostics:\n"
        for diag in heldout_comparison.get("diagnostics", []):
            md_text += (
                f"  - {diag.get('label')}: leader={_float_or(diag.get('candidate')):+.2f} | "
                f"baseline={_float_or(diag.get('baseline')):+.2f} | delta={_float_or(diag.get('delta')):+.2f} "
                f"| {diag.get('verdict')}\n"
            )

    md_text += f"""
### Hypothesis
{best_test.get('hypothesis', '')}

### Code
```python
{best_test.get('code', '')}
```
"""
else:
    md_text += "- no winner yet: held-out evaluation paused or no surviving candidate passed filters.\n"

if "EVOLUTION_TLDR_FILE" in globals() and EVOLUTION_TLDR_FILE.exists():
    evolution_tldr_text = _safe_text(EVOLUTION_TLDR_FILE)
    if evolution_tldr_text.strip():
        md_text += "\n## Evolution TLDR\n"
        md_text += evolution_tldr_text + "\n"
if evolution_memory and evolution_memory.get("generation_reflections"):
    last_ref = evolution_memory["generation_reflections"][-1]
    md_text += "\n## Latest Generation Reflection\n"
    md_text += "```\n" + str(last_ref.get("reflection", "")) + "\n```\n"

if RUN_REPORTS or RUN_HELDOUT_EVAL:
    SUMMARY_MD.write_text(md_text)
print(md_text)